# Libraries, Directories, Datasets

In [ ]:
import os
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt
import scipy.sparse as sps
from scipy import sparse
from scipy.spatial import cKDTree
from IPython.display import display

plt.rcParams['svg.fonttype'] = 'none'

In [ ]:
output_dir = Path("/outputdirectory")
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
adata1 = sc.read_h5ad("/directoryto/Cevrim_Human_all_cells.h5ad")
adata2 = sc.read_h5ad("/directoryto/Cevrim_Human_only_fibdec_cells.h5ad")
adata3 = sc.read_h5ad("/directoryto/Cevrim_XMens_all_cells.h5ad")
adata4 = sc.read_h5ad("/directoryto/Cevrim_XMens_only_fibdec_cells.h5ad")

# UMAPs

## Mouse

### All cells

In [ ]:
##### All mouse cell lineages ###

key = 'cell_type'
basis = 'X_umap_scVI'   # full obsm key

# Color assignment per cell type
color_map = {
    'dec':       '#00B4C5',  # cyan, slightly deepened (matches IF cyan)
    'epi':       '#D4B900',  # golden yellow (matches IF yellow)
    'vsc':       '#5B7FE6',  # medium blue, clearly bluer than the cyan
    'eFib':      '#CC79A7',  # reddish purple
    'imm':       '#D55E00',  # vermillion
    'smc':       '#4DAF4A',  # true green, away from the cyan
    'pm':        '#8C8C8C',  # mid grey
}

# Warn about any cell types in the data that have no assigned color
labels = adata3.obs[key].astype('category').cat.categories
missing = [l for l in labels if l not in color_map]
if missing:
    print(f"Warning: no color assigned for {missing}; they will be drawn grey (#BBBBBB).")

# Optional: store the palette in AnnData so sc.pl.umap etc. use the same colors
adata3.obs[key] = adata3.obs[key].astype('category')
adata3.uns[f'{key}_colors'] = [color_map.get(l, '#BBBBBB') for l in adata3.obs[key].cat.categories]

# Plot
fig, ax = plt.subplots(figsize=(5, 5))
coords = adata3.obsm[basis]
colors = adata3.obs[key].astype(str).map(lambda x: color_map.get(x, '#BBBBBB')).values

ax.scatter(
    coords[:, 0], coords[:, 1],
    c=colors, s=1, alpha=0.9, edgecolors='none'
)

# Remove everything: labels, ticks, frames, legend
ax.set_aspect('equal')
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_title('')
ax.grid(False)
for spine in ax.spines.values():
    spine.set_visible(False)

fig.patch.set_facecolor('white')
ax.set_facecolor('white')

filename = "20261113_UMAP_Mouse_CellTypes.png"
output_path = output_dir / filename
fig.savefig(output_path, dpi=500, bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f"Saved UMAP to {output_path}")

### Only Immune

In [ ]:
### Immune cell subtypes

key = 'cell_subtype'
basis = 'X_umap_scVI'

immune_color_map = {
    # myeloid (warm)
    'neutrophils':                    '#E69F00',  # orange (or '#A50F15' deep red)
    'monocytes':                      '#C9A227',  # gold
    'Macrophages':                    '#D55E00',  # vermillion (same as immune_cell)
    'Macrophages_cycling':            '#F28E5C',  # light vermillion
    'DC':                             '#8C5A2B',  # brown
    # lymphoid: T and B (blues, green)
    'T_cells':                        '#0072B2',  # blue
    'Rorc_lymphocytes':               '#56B4E9',  # sky blue
    'B_cells':                        '#009E73',  # bluish green
    # lymphoid: uNK (purples)
    'uNK':                            '#CC79A7',  # reddish purple
    'uNK_cytotoxic':                  '#7B5EA7',  # violet
    'uNK_cycling':                    '#B39DDB',  # light violet
}

show_background = False          # False = plot immune cells only

# Check that every immune subtype exists in the data
present = set(adata3.obs[key].astype(str).unique())
absent = [s for s in immune_color_map if s not in present]
if absent:
    print(f"Warning: these subtypes are not in adata3.obs['{key}']: {absent}")


# Split cells
coords = adata3.obsm[basis]
subtype = adata3.obs[key].astype(str).values
is_immune = np.isin(subtype, list(immune_color_map))

# Random draw order for immune cells so no subtype is systematically on top
rng = np.random.default_rng(0)
immune_idx = rng.permutation(np.where(is_immune)[0])
immune_colors = [immune_color_map[s] for s in subtype[immune_idx]]

# Plot
fig, ax = plt.subplots(figsize=(2, 2))

if show_background:
    ax.scatter(
        coords[~is_immune, 0], coords[~is_immune, 1],
        c=background_color, s=25, alpha=0.9, edgecolors='none', rasterized=True
    )

ax.scatter(
    coords[immune_idx, 0], coords[immune_idx, 1],
    c=immune_colors, s=1, alpha=0.9, edgecolors='none', rasterized=True
)

# Remove everything: labels, ticks, frames, legend
ax.set_aspect('equal')
ax.set_axis_off()

fig.patch.set_facecolor('white')
ax.set_facecolor('white')

filename = "20261113_UMAP_Mouse_ImmuneSubtypes.png"
output_path = output_dir / filename
fig.savefig(output_path, dpi=500, bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f"Saved UMAP to {output_path}")

### Only eFib/dec

In [ ]:
### Immune cell subtypes

key = 'cell_subtype'
basis = 'X_umap_scVI'

# Color assignment per cell type
color_map = {
    # endometrial fibroblasts
    'eFib3':    '#4694c5',  # sky blue
    'eFib2':    '#00401d',  # dark green
    'eFib2i':   '#127d35',  # green (related to efib_middle)
    'eFib1':    '#a77207',  # ochre
    'eFibP':    '#a0e1b0',  # mint
    # decidual cells
    'dec2':     '#1e1ee2',  # blue
    'dec1':     '#762a83',  # purple
    'dec3':     '#b11a2d',  # red
    'decAp':    '#fccc3f',  # yellow
}

show_background = False          # False = plot immune cells only

# Check that every immune subtype exists in the data
present = set(adata3.obs[key].astype(str).unique())
absent = [s for s in color_map if s not in present]
if absent:
    print(f"Warning: these subtypes are not in adata3.obs['{key}']: {absent}")


# Split cells
coords = adata3.obsm[basis]
subtype = adata3.obs[key].astype(str).values
is_immune = np.isin(subtype, list(color_map))

# Random draw order for immune cells so no subtype is systematically on top
rng = np.random.default_rng(0)
immune_idx = rng.permutation(np.where(is_immune)[0])
immune_colors = [color_map[s] for s in subtype[immune_idx]]

# Plot
fig, ax = plt.subplots(figsize=(5, 5))

if show_background:
    ax.scatter(
        coords[~is_immune, 0], coords[~is_immune, 1],
        c=background_color, s=1, alpha=0.9, edgecolors='none', rasterized=True
    )

ax.scatter(
    coords[immune_idx, 0], coords[immune_idx, 1],
    c=immune_colors, s=1, alpha=0.9, edgecolors='none', rasterized=True
)

# Remove everything: labels, ticks, frames, legend
ax.set_aspect('equal')
ax.set_axis_off()

fig.patch.set_facecolor('white')
ax.set_facecolor('white')

filename = "20261113_UMAP_Mouse_eFibDec.png"
output_path = output_dir / filename
fig.savefig(output_path, dpi=500, bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f"Saved UMAP to {output_path}")

## Human

### All cell types

In [ ]:
##### All human cell lineages ###

key = 'cell_type'
basis = 'X_umap_scVI'   # full obsm key

# Color assignment per cell type
color_map = {
    'dec':            '#00B4C5',  # cyan, slightly deepened
    'epi':            '#D4B900',  # golden yellow
    'vsc':            '#5B7FE6',  # medium blue, clearly bluer than the cyan
    'eFib':           '#CC79A7',  # reddish purple
    'imm':            '#D55E00',  # vermillion
    'smc':            '#4DAF4A',  # true green
}

# Warn about any cell types in the data that have no assigned color
labels = adata1.obs[key].astype('category').cat.categories
missing = [l for l in labels if l not in color_map]
if missing:
    print(f"Warning: no color assigned for {missing}; they will be drawn grey (#BBBBBB).")

# Optional: store the palette in AnnData so sc.pl.umap etc. use the same colors
adata1.obs[key] = adata1.obs[key].astype('category')
adata1.uns[f'{key}_colors'] = [color_map.get(l, '#BBBBBB') for l in adata1.obs[key].cat.categories]

# Plot
fig, ax = plt.subplots(figsize=(5, 5))
coords = adata1.obsm[basis]
colors = adata1.obs[key].astype(str).map(lambda x: color_map.get(x, '#BBBBBB')).values

ax.scatter(
    coords[:, 0], coords[:, 1],
    c=colors, s=0.35, alpha=0.9, edgecolors='none'
)

# Remove everything: labels, ticks, frames, legend
ax.set_aspect('equal')
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_title('')
ax.grid(False)
for spine in ax.spines.values():
    spine.set_visible(False)

fig.patch.set_facecolor('white')
ax.set_facecolor('white')

filename = "20261113_UMAP_Human_CellTypes.png"
output_path = output_dir / filename
fig.savefig(output_path, dpi=500, bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f"Saved UMAP to {output_path}")

### Only Immune

In [ ]:
### Immune cell subtypes

key = 'cell_subtype'
basis = 'X_umap_scVI'

immune_color_map = {
    # myeloid (warm)
    'Macrophages':      '#D55E00',  # vermillion (same as mouse)
    'Mast_cells':       '#B2182B',  # crimson
    # lymphoid: T cells (blues)
    'T_cells_CD4':      '#0072B2',  # blue (= mouse T_cells)
    'T_cells_CD8':      '#1B2A6B',  # navy
    'Treg':             '#6C8EBF',  # slate blue
    'Rorc_lymphocytes': '#56B4E9',  # sky blue (same as mouse)
    # lymphoid: B cells
    'B_cells':          '#009E73',  # bluish green (same as mouse)
    # lymphoid: uNK (purples)
    'uNK':              '#CC79A7',  # reddish purple (same as mouse)
    'uNK_cytotoxic':    '#7B5EA7',  # violet (same as mouse)
}

show_background = False          # False = plot immune cells only

# Check that every immune subtype exists in the data
present = set(adata1.obs[key].astype(str).unique())
absent = [s for s in immune_color_map if s not in present]
if absent:
    print(f"Warning: these subtypes are not in adata1.obs['{key}']: {absent}")


# Split cells
coords = adata1.obsm[basis]
subtype = adata1.obs[key].astype(str).values
is_immune = np.isin(subtype, list(immune_color_map))

# Random draw order for immune cells so no subtype is systematically on top
rng = np.random.default_rng(0)
immune_idx = rng.permutation(np.where(is_immune)[0])
immune_colors = [immune_color_map[s] for s in subtype[immune_idx]]

# Plot
fig, ax = plt.subplots(figsize=(5, 5))

if show_background:
    ax.scatter(
        coords[~is_immune, 0], coords[~is_immune, 1],
        c=background_color, s=10, alpha=0.9, edgecolors='none', rasterized=True
    )

ax.scatter(
    coords[immune_idx, 0], coords[immune_idx, 1],
    c=immune_colors, s=0.50, alpha=0.9, edgecolors='none', rasterized=True
)

# Remove everything: labels, ticks, frames, legend
ax.set_aspect('equal')
ax.set_axis_off()

fig.patch.set_facecolor('white')
ax.set_facecolor('white')

filename = "20261113_UMAP_Human_ImmuneSubtypes.png"
output_path = output_dir / filename
fig.savefig(output_path, dpi=500, bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f"Saved UMAP to {output_path}")

### Fib/Dec Subtypes

In [ ]:
key = 'cell_subtype'
basis = 'X_umap_scVI'

# Color assignment per cell type
color_map = {
    # endometrial fibroblasts (teals + olive)
    'eFib_a':  '#01665e',  # dark teal
    'eFib_b':  '#35978f',  # teal
    'eFib_c':  '#80cdc1',  # light teal
    'myoFib':  '#8c8c00',  # olive
    # decidual cells (magenta / pink / coral)
    'dec_a':   '#8e0152',  # dark magenta
    'dec_b':   '#de77ae',  # pink
    'dec_c':   '#f46d43',  # coral
    'dec_d':   '#f4a582',  # salmon
}

show_background = False          # False = plot immune cells only

# Check that every immune subtype exists in the data
present = set(adata1.obs[key].astype(str).unique())
absent = [s for s in color_map if s not in present]
if absent:
    print(f"Warning: these subtypes are not in adata3.obs['{key}']: {absent}")


# Split cells
coords = adata1.obsm[basis]
subtype = adata1.obs[key].astype(str).values
is_immune = np.isin(subtype, list(color_map))

# Random draw order for immune cells so no subtype is systematically on top
rng = np.random.default_rng(0)
immune_idx = rng.permutation(np.where(is_immune)[0])
immune_colors = [color_map[s] for s in subtype[immune_idx]]

# Plot
fig, ax = plt.subplots(figsize=(5, 5))

if show_background:
    ax.scatter(
        coords[~is_immune, 0], coords[~is_immune, 1],
        c=background_color, s=1, alpha=0.9, edgecolors='none', rasterized=True
    )

ax.scatter(
    coords[immune_idx, 0], coords[immune_idx, 1],
    c=immune_colors, s=1, alpha=0.9, edgecolors='none', rasterized=True
)

# Remove everything: labels, ticks, frames, legend
ax.set_aspect('equal')
ax.set_axis_off()

fig.patch.set_facecolor('white')
ax.set_facecolor('white')

filename = "20261113_UMAP_Human_eFibDec.png"
output_path = output_dir / filename
fig.savefig(output_path, dpi=500, bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f"Saved UMAP to {output_path}")

### Epi

In [ ]:
key = 'cell_subtype'
basis = 'X_umap_scVI'

# Color assignment per cell type
color_map = {
    'epi1':          '#E3B23C',  # golden yellow
    'epi2':          '#3F7CAC',  # steel blue
    'epi_ciliated':  '#5A9E6F',  # sage green
}

show_background = False          # False = plot immune cells only

# Check that every immune subtype exists in the data
present = set(adata1.obs[key].astype(str).unique())
absent = [s for s in color_map if s not in present]
if absent:
    print(f"Warning: these subtypes are not in adata3.obs['{key}']: {absent}")


# Split cells
coords = adata1.obsm[basis]
subtype = adata1.obs[key].astype(str).values
is_immune = np.isin(subtype, list(color_map))

# Random draw order for immune cells so no subtype is systematically on top
rng = np.random.default_rng(0)
immune_idx = rng.permutation(np.where(is_immune)[0])
immune_colors = [color_map[s] for s in subtype[immune_idx]]

# Plot
fig, ax = plt.subplots(figsize=(5, 5))

if show_background:
    ax.scatter(
        coords[~is_immune, 0], coords[~is_immune, 1],
        c=background_color, s=1, alpha=0.9, edgecolors='none', rasterized=True
    )

ax.scatter(
    coords[immune_idx, 0], coords[immune_idx, 1],
    c=immune_colors, s=1, alpha=0.9, edgecolors='none', rasterized=True
)

# Remove everything: labels, ticks, frames, legend
ax.set_aspect('equal')
ax.set_axis_off()

fig.patch.set_facecolor('white')
ax.set_facecolor('white')

filename = "20261113_UMAP_Human_epi.png"
output_path = output_dir / filename
fig.savefig(output_path, dpi=500, bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f"Saved UMAP to {output_path}")

# Dotplots

## Mouse

### Cell Type Annotation

In [ ]:
### All Mouse Annotations ###

# Marker genes by cell type
marker_genes = [
    'Cdh1', 'Pax8', 
    'Pdgfra',
    'Wnt4', 'Prl8a2', 'Bmp2', 'Cdh3',
    'Flt1', 'Pecam1', 
    'Ptprc', 
    'Acta2', 'Myh11', 
    'Sox6', 'Wdr17',
    'Pgr', 'Esr1'
]

# Desired row order
cell_order = ['epi', 'eFib', 'dec', 'vsc', 'imm', 'smc', 'pm']

# Settings
key = 'cell_type'
layer = 'log1p_counts'


missing = set(cell_order) - set(adata3.obs[key].astype(str).unique())
if missing:
    raise ValueError(f"Missing cell types in adata3.obs['{key}']: {missing}")

# Keep only the listed cell types
sub = adata3[adata3.obs[key].astype(str).isin(cell_order)].copy()
sub.obs[key] = pd.Categorical(sub.obs[key].astype(str), categories=cell_order)
excluded = adata3.obs.loc[~adata3.obs[key].astype(str).isin(cell_order), key].astype(str).value_counts()
print(f"Plotting {sub.n_obs} cells from {len(cell_order)} cell types; excluded {excluded.sum()} cells from {len(excluded)} other types")
print(sub.obs[key].value_counts().reindex(cell_order).to_string(), '\n')

# Drop duplicates and genes not in the data
marker_genes = list(dict.fromkeys(marker_genes))
missing_genes = [g for g in marker_genes if g not in sub.var_names]
if missing_genes:
    print(f"Warning: {len(missing_genes)} marker(s) not in var, dropped: {missing_genes}")
marker_genes = [g for g in marker_genes if g in sub.var_names]

# Dot plot: each gene scaled 0-1 across the listed groups
axes = sc.pl.dotplot(
    sub,
    var_names=marker_genes,
    groupby=key,
    layer=layer,
    use_raw=False,
    categories_order=cell_order,
    figsize=(max(5, 0.15 * len(marker_genes) + 1), max(1.0, len(cell_order) * 0.32)),
    standard_scale='var',
    colorbar_title='Scaled mean\nexpression',
    size_title='Fraction of\nnuclei (%)',
    show=False
)
fig = plt.gcf()

# Dot borders in the main axis
main_ax = axes['mainplot_ax']
for coll in main_ax.collections:
    coll.set_linewidth(0.5)
    coll.set_edgecolor("gray")

# Style all axes (main + legends)
for ax_ in fig.get_axes():
    ax_.set_facecolor('white')
    ax_.tick_params(colors='black')
    ax_.title.set_color('black')
    for label in ax_.get_xticklabels() + ax_.get_yticklabels():
        label.set_color('black')
    for spine in ax_.spines.values():
        spine.set_edgecolor('black')

# Save (SVG)
fig.patch.set_facecolor('white')
base = Path(output_dir) / "20261113-Dotplot_Mouse_CellType_Marker"
fig.savefig(base.with_suffix('.svg'), format='svg', bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f"Saved dotplot to {base.with_suffix('.svg')}")

### Immune Subtypes

In [ ]:
plt.rcParams['svg.fonttype'] = 'none'   # editable text in the SVG

marker_genes = [
    'Ptprc',
    'S100a8', 'S100a9', 'Cxcr2',
    'Fgr', 'Sirpb1b', 'Itgam',
    'Mrc1', 'Adgre1', 'Csf1r',
    'Clec9a', 'Flt3', 'Xcr1', 'Zbtb46',
    'Cd3e', 'Cd3g', 'Trbc2', 'Bcl11b', 'Themis', 'Tcf7',
    'Il23r', 'Cd163l1', 'Rorc',
    'Pax5', 'Ebf1', 'Ighd',
    'Ncr1', 'Klrb1c', 'Styk1',
    'Gzma', 'Gzmb', 'Gzmg', 'Prf1',
    'Mki67', 'Top2a', 'Kif11',
]

# Desired row order (only these subtypes are plotted)
cell_order = [
    'neutrophils',
    'monocytes',
    'Macrophages',
    'Macrophages_cycling',
    'DC',
    'T_cells',
    'Rorc_lymphocytes',
    'B_cells',
    'uNK',
    'uNK_cytotoxic',
    'uNK_cycling',
]

# Settings
key = 'cell_subtype'
layer = 'log1p_counts'

missing = set(cell_order) - set(adata3.obs[key].astype(str).unique())
if missing:
    raise ValueError(f"Missing cell types in adata3.obs['{key}']: {missing}")

# Keep only the listed cell types
sub = adata3[adata3.obs[key].astype(str).isin(cell_order)].copy()
sub.obs[key] = pd.Categorical(sub.obs[key].astype(str), categories=cell_order)
excluded = adata3.obs.loc[~adata3.obs[key].astype(str).isin(cell_order), key].astype(str).value_counts()
print(f"Plotting {sub.n_obs} cells from {len(cell_order)} cell types; excluded {excluded.sum()} cells from {len(excluded)} other types")
print(sub.obs[key].value_counts().reindex(cell_order).to_string(), '\n')

# Drop duplicates and genes not in the data
marker_genes = list(dict.fromkeys(marker_genes))
missing_genes = [g for g in marker_genes if g not in sub.var_names]
if missing_genes:
    print(f"Warning: {len(missing_genes)} marker(s) not in var, dropped: {missing_genes}")
marker_genes = [g for g in marker_genes if g in sub.var_names]

# Dot plot: each gene scaled 0-1 across the listed groups
axes = sc.pl.dotplot(
    sub,
    var_names=marker_genes,
    groupby=key,
    layer=layer,
    use_raw=False,
    categories_order=cell_order,
    figsize=(max(5, 0.25 * len(marker_genes) + 2), max(1.0, len(cell_order) * 0.32)),
    standard_scale='var',
    colorbar_title='Scaled mean\nexpression',
    size_title='Fraction of\nnuclei (%)',
    show=False
)
fig = plt.gcf()

# Dot borders in the main axis
main_ax = axes['mainplot_ax']
for coll in main_ax.collections:
    coll.set_linewidth(0.5)
    coll.set_edgecolor("gray")

# Style all axes (main + legends)
for ax_ in fig.get_axes():
    ax_.set_facecolor('white')
    ax_.tick_params(colors='black')
    ax_.title.set_color('black')
    for label in ax_.get_xticklabels() + ax_.get_yticklabels():
        label.set_color('black')
    for spine in ax_.spines.values():
        spine.set_edgecolor('black')

# Save (SVG)
fig.patch.set_facecolor('white')
base = Path(output_dir) / "20261113-Dotplot_Mouse_ImmuneSubtype_Marker_Dotplot"
fig.savefig(base.with_suffix('.svg'), format='svg', bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f"Saved dotplot to {base.with_suffix('.svg')}")

### eFib/dec

In [ ]:
plt.rcParams['svg.fonttype'] = 'none'   # editable text in the SVG

marker_genes = [
    'Pdgfra', 'Col3a1', 'Col1a2', 'Celf2',
    'Kcnd2', 'Nlgn1', 
    'Gria4', 'Rxfp1', 'Sema5a', 'Cdh11', 
    'Top2a', 'Kif15', 'Knl1', 'Mki67', 'Bub1', 
    'Klf5', 'Hand2', 'Wnt4', 'Cebpb', 'Prl8a2', 'Bmp2', 'Cdh3',  'Gja1', 'Gpx3', 'Ptgs2',
    'Hif1a', 'Epas1', 'Vegfa',
    'Adam15', 'Adam19',
    'Bax','Pgr', 'Esr1',
]

# Desired row order (only these subtypes are plotted)
cell_order = [
    'eFib1',
    'eFib2',
    'eFib2i',
    'eFib3',
    'eFibP',
    'dec1',
    'dec2',
    'dec3',
    'decAp',
]

# Settings
key = 'cell_subtype'
layer = 'log1p_counts'

missing = set(cell_order) - set(adata3.obs[key].astype(str).unique())
if missing:
    raise ValueError(f"Missing cell types in adata3.obs['{key}']: {missing}")

# Keep only the listed cell types
sub = adata3[adata3.obs[key].astype(str).isin(cell_order)].copy()
sub.obs[key] = pd.Categorical(sub.obs[key].astype(str), categories=cell_order)
excluded = adata3.obs.loc[~adata3.obs[key].astype(str).isin(cell_order), key].astype(str).value_counts()
print(f"Plotting {sub.n_obs} cells from {len(cell_order)} cell types; excluded {excluded.sum()} cells from {len(excluded)} other types")
print(sub.obs[key].value_counts().reindex(cell_order).to_string(), '\n')

# Drop duplicates and genes not in the data
marker_genes = list(dict.fromkeys(marker_genes))
missing_genes = [g for g in marker_genes if g not in sub.var_names]
if missing_genes:
    print(f"Warning: {len(missing_genes)} marker(s) not in var, dropped: {missing_genes}")
marker_genes = [g for g in marker_genes if g in sub.var_names]

# Dot plot: each gene scaled 0-1 across the listed groups
axes = sc.pl.dotplot(
    sub,
    var_names=marker_genes,
    groupby=key,
    layer=layer,
    use_raw=False,
    categories_order=cell_order,
    figsize=(max(5, 0.25 * len(marker_genes) + 2), max(1.0, len(cell_order) * 0.32)),
    standard_scale='var',
    colorbar_title='Scaled mean\nexpression',
    size_title='Fraction of\nnuclei (%)',
    show=False
)
fig = plt.gcf()

# Dot borders in the main axis
main_ax = axes['mainplot_ax']
for coll in main_ax.collections:
    coll.set_linewidth(0.5)
    coll.set_edgecolor("gray")

# Style all axes (main + legends)
for ax_ in fig.get_axes():
    ax_.set_facecolor('white')
    ax_.tick_params(colors='black')
    ax_.title.set_color('black')
    for label in ax_.get_xticklabels() + ax_.get_yticklabels():
        label.set_color('black')
    for spine in ax_.spines.values():
        spine.set_edgecolor('black')

# Save (SVG)
fig.patch.set_facecolor('white')
base = Path(output_dir) / "20261113-Dotplot_Mouse_eFibDec_subtype_Marker_Dotplot"
fig.savefig(base.with_suffix('.svg'), format='svg', bbox_inches='tight', facecolor='white')
plt.show()
print(f"Saved dotplot to {base.with_suffix('.svg')}")

### Comparison to Kirkwood 2021

In [ ]:
plt.rcParams['svg.fonttype'] = 'none'   # editable text in the SVG

# Marker genes, grouped (group names appear above the genes)
marker_dict = {
    'F1': ['Spon2', 'Aspg', 'Dpep1', 'Ngfr', 'Angptl7', 'Ifit1', 'Ifit3', 'H2-Q7'],
    'F2': ['Cxcl14', 'Cdh11', 'Wt1', 'Rgs2', 'Smoc2', 'Wnt4', 'Aldh1a2', 'Bmp7'],
    'F3': ['Clec3b', 'Col14a1', 'Fap', 'Cd55', 'Cxcl16', 'Mmp3', 'Efemp1', 'Vit'],
}

# Rows (only these subtypes, only this timepoint)
cell_order = ['eFib1', 'eFib2', 'eFib3', 'eFibP']
timepoint  = 'uninduced'

# Settings
key = 'cell_subtype'
layer = 'log1p_counts'

# Keep only the timepoint and subtypes
sel = (adata3.obs['timepoint'].astype(str) == timepoint) & \
      (adata3.obs[key].astype(str).isin(cell_order))
sub = adata3[sel].copy()

present = [c for c in cell_order if c in set(sub.obs[key].astype(str))]
absent = [c for c in cell_order if c not in present]
if absent:
    print(f"Warning: no {timepoint} nuclei for {absent}; rows dropped")
cell_order = present
sub.obs[key] = pd.Categorical(sub.obs[key].astype(str), categories=cell_order)
print(f"Plotting {sub.n_obs} {timepoint} nuclei")
print(sub.obs[key].value_counts().reindex(cell_order).to_string(), '\n')

# Drop genes not in the data, and any repeated gene
seen, plot_dict = set(), {}
for grp, genes in marker_dict.items():
    keep = [g for g in genes if g in sub.var_names and g not in seen]
    dropped = [g for g in genes if g not in sub.var_names]
    if dropped:
        print(f"Warning: {grp}: not in var, dropped: {dropped}")
    seen.update(keep)
    if keep:
        plot_dict[grp] = keep
n_genes = sum(len(v) for v in plot_dict.values())

# Dot plot: each gene scaled 0-1 across the listed groups
axes = sc.pl.dotplot(
    sub,
    var_names=plot_dict,
    groupby=key,
    layer=layer,
    use_raw=False,
    categories_order=cell_order,
    figsize=(max(5, 0.25 * n_genes + 2), max(1.5, len(cell_order) * 0.32 + 0.6)),
    standard_scale='var',
    colorbar_title='Scaled mean\nexpression',
    size_title='Fraction of\nnuclei (%)',
    show=False
)
fig = plt.gcf()

# Dot borders in the main axis
main_ax = axes['mainplot_ax']
for coll in main_ax.collections:
    coll.set_linewidth(0.5)
    coll.set_edgecolor("gray")

# Style all axes (main + legends + group labels)
for ax_ in fig.get_axes():
    ax_.set_facecolor('white')
    ax_.tick_params(colors='black')
    ax_.title.set_color('black')
    for label in ax_.get_xticklabels() + ax_.get_yticklabels():
        label.set_color('black')
    for spine in ax_.spines.values():
        spine.set_edgecolor('black')

# Save (SVG)
fig.patch.set_facecolor('white')
base = Path(output_dir) / "20261113-Dotplot_Mouse_eFib_uninduced_vs_Kirkwood_Dotplot"
fig.savefig(base.with_suffix('.svg'), format='svg', bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f"Saved dotplot to {base.with_suffix('.svg')}")

### Comparison to Kirkwood 2022

In [ ]:
plt.rcParams['svg.fonttype'] = 'none'   # editable text in the SVG

# Marker genes, grouped (group names appear above the genes)
marker_dict = {
    'F1': ['Spon2', 'Aspg', 'Dpep1', 'Ngfr', 'Angptl7', 'Ifit1', 'Ifit3', 'H2-Q7'],
    'F2': ['Cxcl14', 'Cdh11', 'Wt1', 'Rgs2', 'Smoc2', 'Wnt4', 'Aldh1a2', 'Bmp7'],
    'F3': ['Clec3b', 'Col14a1', 'Fap', 'Cd55', 'Cxcl16', 'Mmp3', 'Efemp1', 'Vit'],
    'F4': ['Krt18', 'Mt1', 'Sdc4', 'Ndrg1', 'Alcam', 'Cd24a', 'Clu', 'Epcam'],
    'F5': ['Col7a1', 'Scube1']
}

# Rows (only these subtypes) and one dotplot per timepoint
cell_order_all = ['eFib1', 'eFib2', 'eFib2i', 'eFib3', 'eFibP', 'dec1', 'dec2', 'dec3', 'decAp']
timepoints     = ['uninduced', '4.5dpi_menstrual']

# Settings
key       = 'cell_subtype'
layer     = 'log1p_counts'
MIN_CELLS = 10        # warn when a row has fewer nuclei than this

for timepoint in timepoints:
    print(f"==== {timepoint} ====")

    # Keep only the timepoint and subtypes
    sel = (adata3.obs['timepoint'].astype(str) == timepoint) & \
          (adata3.obs[key].astype(str).isin(cell_order_all))
    sub = adata3[sel].copy()
    if sub.n_obs == 0:
        print(f"Warning: no {cell_order_all} nuclei for {timepoint}; skipped\n")
        continue

    present = [c for c in cell_order_all if c in set(sub.obs[key].astype(str))]
    absent = [c for c in cell_order_all if c not in present]
    if absent:
        print(f"Warning: no {timepoint} nuclei for {absent}; rows dropped")
    cell_order = present
    sub.obs[key] = pd.Categorical(sub.obs[key].astype(str), categories=cell_order)

    counts = sub.obs[key].value_counts().reindex(cell_order)
    print(f"Plotting {sub.n_obs} {timepoint} nuclei")
    print(counts.to_string(), '\n')
    small = counts[counts < MIN_CELLS].index.tolist()
    if small:
        print(f"Note: fewer than {MIN_CELLS} nuclei in {small}; treat those rows with caution\n")

    # Drop genes not in the data, and any repeated gene
    seen, plot_dict = set(), {}
    for grp, genes in marker_dict.items():
        keep = [g for g in genes if g in sub.var_names and g not in seen]
        dropped = [g for g in genes if g not in sub.var_names]
        if dropped:
            print(f"Warning: {grp}: not in var, dropped: {dropped}")
        seen.update(keep)
        if keep:
            plot_dict[grp] = keep
    n_genes = sum(len(v) for v in plot_dict.values())

    # Dot plot: each gene scaled 0-1 across the listed groups
    axes = sc.pl.dotplot(
        sub,
        var_names=plot_dict,
        groupby=key,
        layer=layer,
        use_raw=False,
        categories_order=cell_order,
        figsize=(max(5, 0.25 * n_genes + 2), max(1.5, len(cell_order) * 0.32 + 0.6)),
        standard_scale='var',
        colorbar_title='Scaled mean\nexpression',
        size_title='Fraction of\nnuclei (%)',
        show=False
    )
    fig = plt.gcf()

    # Dot borders in the main axis
    main_ax = axes['mainplot_ax']
    for coll in main_ax.collections:
        coll.set_linewidth(0.5)
        coll.set_edgecolor("gray")

    # Style all axes (main + legends + group labels)
    for ax_ in fig.get_axes():
        ax_.set_facecolor('white')
        ax_.tick_params(colors='black')
        ax_.title.set_color('black')
        for label in ax_.get_xticklabels() + ax_.get_yticklabels():
            label.set_color('black')
        for spine in ax_.spines.values():
            spine.set_edgecolor('black')

    # Save (SVG)
    fig.patch.set_facecolor('white')
    base = Path(output_dir) / f"20261113-Dotplot_Mouse_eFib_{timepoint}_vs_Kirkwood"
    fig.savefig(base.with_suffix('.svg'), format='svg', bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f"Saved dotplot to {base.with_suffix('.svg')}\n")

### Cxcl12 expression

In [ ]:
# ---------------------------- settings ----------------------------
marker_genes = ['Cxcl12']
cell_order = [
    'epi', 'eFib', 'dec', 'imm', 'vsc'
]
timepoint_order = [
    'uninduced', '1.5dpi_early', '1.5dpi_late', '2.5dpi', '3.5dpi', '4.5dpi_menstrual'
]

ct_key, tp_key, layer = 'cell_type', 'timepoint', 'log1p_counts'
MIN_CELLS = 10         # no dot for cell type × timepoint groups with too few cells
SHARED_SCALE = False   # False: 0–1 per gene across the grid (like standard_scale='var')
                       # True: one absolute color scale across all genes
cmap = 'Reds'          # scanpy's default dotplot colormap

plt.rcParams['svg.fonttype'] = 'none'   # keep text editable in the SVG

# ---------------------------- checks ----------------------------
obs = adata3.obs
ct_all = obs[ct_key].astype(str).values
tp_all = obs[tp_key].astype(str).values

missing_ct = [c for c in cell_order if c not in set(ct_all)]
missing_tp = [t for t in timepoint_order if t not in set(tp_all)]
if missing_ct or missing_tp:
    print(f"Warning: not found in data (these rows/columns will be empty): "
          f"cell types {missing_ct}, timepoints {missing_tp}\n"
          f"Timepoints present: {sorted(set(tp_all))}")
missing_genes = [g for g in marker_genes if g not in adata3.var_names]
if missing_genes:
    raise ValueError(f"Genes not in adata3.var_names: {missing_genes}")

# ---------------------------- helpers ----------------------------
def get_expr(ad, gene, layer):
    X = ad[:, gene].layers[layer] if layer else ad[:, gene].X
    X = X.toarray() if sparse.issparse(X) else np.asarray(X)
    return X.ravel().astype(float)

def grid_stats(gene):
    e = get_expr(adata3, gene, layer)
    df = pd.DataFrame({'ct': ct_all, 'tp': tp_all, 'e': e, 'pos': e > 0})
    g = df.groupby(['ct', 'tp'])
    shape = dict(index=cell_order, columns=timepoint_order)
    n    = g.size().unstack().reindex(**shape).fillna(0)
    mean = g['e'].mean().unstack().reindex(**shape)
    frac = g['pos'].mean().unstack().reindex(**shape)
    sparse_cells = n < MIN_CELLS
    mean[sparse_cells] = np.nan
    frac[sparse_cells] = np.nan
    return n, mean, frac, sparse_cells

if SHARED_SCALE:
    vmax_global = max(float(np.nanmax(grid_stats(g)[1].values)) for g in marker_genes)

# Placeholder AnnData so scanpy's DotPlot can draw a cell type × timepoint grid
# (rows = cell types, columns = timepoints; the dot values are passed in directly)
grid_ad = sc.AnnData(
    X=np.zeros((len(cell_order), len(timepoint_order)), dtype=np.float32),
    obs=pd.DataFrame({'cell_type': pd.Categorical(cell_order, categories=cell_order)},
                     index=[f'row{i}' for i in range(len(cell_order))]),
    var=pd.DataFrame(index=timepoint_order),
)

# ---------------------------- plot + save (one SVG per gene) ----------------------------
for gene in marker_genes:
    n, mean, frac, sparse_cells = grid_stats(gene)

    blanked = [(ct, tp) for ct in cell_order for tp in timepoint_order if sparse_cells.loc[ct, tp]]
    if blanked:
        print(f"[{gene}] no dot for groups with <{MIN_CELLS} cells: {blanked}")

    if SHARED_SCALE:
        color_df, vmin, vmax = mean, 0, vmax_global
    else:
        lo, hi = np.nanmin(mean.values), np.nanmax(mean.values)
        color_df, vmin, vmax = (mean - lo) / (hi - lo if hi > lo else 1), 0, 1

    # blanked groups: size 0 (no dot drawn), color irrelevant
    size_df  = frac.fillna(0)
    color_df = color_df.fillna(0)

    dp = sc.pl.DotPlot(
        grid_ad,
        var_names=timepoint_order,
        groupby='cell_type',
        categories_order=cell_order,
        dot_size_df=size_df,
        dot_color_df=color_df,
        vmin=vmin, vmax=vmax,
    )
    dp.style(cmap=cmap, dot_edge_color='gray', dot_edge_lw=0.5)
    dp.legend(
        colorbar_title='Scaled mean\nexpression' if not SHARED_SCALE else 'Mean expression\n(log1p)',
        size_title='Fraction of\nnuclei (%)',
    )
    dp.make_figure()
    axes = dp.get_axes()                    # <- restored
    fig = axes['mainplot_ax'].figure        # <- restored

    # Gene name (italic) directly above the dot grid; black text everywhere
    axes['mainplot_ax'].set_title(gene, style='italic', color='black')
    for a in fig.get_axes():
        a.set_facecolor('white')
        a.tick_params(colors='black')
        for lab in a.get_xticklabels() + a.get_yticklabels():
            lab.set_color('black')
        for spine in a.spines.values():
            spine.set_edgecolor('black')
    fig.patch.set_facecolor('white')

    svg_path = Path(output_dir) / f"20261113-Dotplot_Mouse_Endometrium_{gene}_Dotplot.svg"
    fig.savefig(svg_path, format='svg', bbox_inches='tight', facecolor='white')
    print(f"Saved dotplot to {svg_path}")
    plt.close(fig)

## Human

### Cell Type Annotation

In [ ]:
### All Human Annotations ###

# Marker genes by cell type
marker_genes = [
    'EPCAM', 'KLF5', 'COL5A1', 'COL6A3', 'DCN', 'LEFTY2', 'CEBPB', 'IGFBP1',
    'PTPRC', 'STK17B', 'PECAM1', 'VWF', 'MCAM', 'CSPG4', 'ACTA2', 'MYH11', 'PGR', 'ESR1'
]

# Desired row order
cell_order = [
    'epi',
    'eFib',
    'dec',
    'imm',
    'vsc',
    'smc',
]

# Settings
key = 'cell_type'
layer = 'log1p_counts'


missing = set(cell_order) - set(adata1.obs[key].astype(str).unique())
if missing:
    raise ValueError(f"Missing cell types in adata1.obs['{key}']: {missing}")

# Keep only the listed cell types
sub = adata1[adata1.obs[key].astype(str).isin(cell_order)].copy()
sub.obs[key] = pd.Categorical(sub.obs[key].astype(str), categories=cell_order)
excluded = adata1.obs.loc[~adata1.obs[key].astype(str).isin(cell_order), key].astype(str).value_counts()
print(f"Plotting {sub.n_obs} cells from {len(cell_order)} cell types; excluded {excluded.sum()} cells from {len(excluded)} other types")
print(sub.obs[key].value_counts().reindex(cell_order).to_string(), '\n')

# Drop duplicates and genes not in the data
marker_genes = list(dict.fromkeys(marker_genes))
missing_genes = [g for g in marker_genes if g not in sub.var_names]
if missing_genes:
    print(f"Warning: {len(missing_genes)} marker(s) not in var, dropped: {missing_genes}")
marker_genes = [g for g in marker_genes if g in sub.var_names]

# Dot plot: each gene scaled 0-1 across the listed groups
axes = sc.pl.dotplot(
    sub,
    var_names=marker_genes,
    groupby=key,
    layer=layer,
    use_raw=False,
    categories_order=cell_order,
    figsize=(max(5, 0.15 * len(marker_genes) + 1), max(1.0, len(cell_order) * 0.32)),
    standard_scale='var',
    colorbar_title='Scaled mean\nexpression',
    size_title='Fraction of\nnuclei (%)',
    show=False
)
fig = plt.gcf()

# Dot borders in the main axis
main_ax = axes['mainplot_ax']
for coll in main_ax.collections:
    coll.set_linewidth(0.5)
    coll.set_edgecolor("gray")

# Style all axes (main + legends)
for ax_ in fig.get_axes():
    ax_.set_facecolor('white')
    ax_.tick_params(colors='black')
    ax_.title.set_color('black')
    for label in ax_.get_xticklabels() + ax_.get_yticklabels():
        label.set_color('black')
    for spine in ax_.spines.values():
        spine.set_edgecolor('black')

# Save (SVG + PNG)
fig.patch.set_facecolor('white')
base = Path(output_dir) / "20261113-Dotplot_Human_CellType_Markers"
fig.savefig(base.with_suffix('.svg'), format='svg', bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f"Saved dotplot to {base.with_suffix('.svg')}")

### Fib/Dec Subtypes

In [ ]:
marker_genes = [
    'C7', 'ACSM1', 'ITGA11', 'CADPS2',
    'CCBE1', 'IGF1', 'TMOD1', 'SFRP4',
    'PTCH1', 'HHIP', 
    'MMP11', 'MMP2', 'MMP14', 'MMP10',
    'LEFTY2', 'CEBPB', 'WNT5A',
    'DIO2', 'PLEKHG1',
    'MMP3', 'IGFBP1',
    'PLAU', 'SERPINE1',
    'TSC22D2', 'RRAS2', 'SLC12A2',
    'FOS', 'HSPH1', 'JUN', 'DNAJB1',
    'HSPA1A', 'HSPB1', 'PLOD2', 'NDRG1', 'MMP1',
]

# Desired row order (only these subtypes are plotted)
cell_order = [
    'myoFib',
    'eFib_a',
    'eFib_b',
    'eFib_c',
    'dec_a',
    'dec_b',
    'dec_c',
    'dec_d',
]

# Settings
key = 'cell_subtype'
layer = 'log1p_counts'

missing = set(cell_order) - set(adata1.obs[key].astype(str).unique())
if missing:
    raise ValueError(f"Missing cell types in adata1.obs['{key}']: {missing}")

# Keep only the listed cell types
sub = adata1[adata1.obs[key].astype(str).isin(cell_order)].copy()
sub.obs[key] = pd.Categorical(sub.obs[key].astype(str), categories=cell_order)
excluded = adata1.obs.loc[~adata1.obs[key].astype(str).isin(cell_order), key].astype(str).value_counts()
print(f"Plotting {sub.n_obs} cells from {len(cell_order)} cell types; excluded {excluded.sum()} cells from {len(excluded)} other types")
print(sub.obs[key].value_counts().reindex(cell_order).to_string(), '\n')

# Drop duplicates and genes not in the data
marker_genes = list(dict.fromkeys(marker_genes))
missing_genes = [g for g in marker_genes if g not in sub.var_names]
if missing_genes:
    print(f"Warning: {len(missing_genes)} marker(s) not in var, dropped: {missing_genes}")
marker_genes = [g for g in marker_genes if g in sub.var_names]

# Dot plot: each gene scaled 0-1 across the listed groups
axes = sc.pl.dotplot(
    sub,
    var_names=marker_genes,
    groupby=key,
    layer=layer,
    use_raw=False,
    categories_order=cell_order,
    figsize=(max(5, 0.25 * len(marker_genes) + 2), max(1.0, len(cell_order) * 0.4)),
    standard_scale='var',
    colorbar_title='Scaled mean\nexpression',
    size_title='Fraction of\nnuclei (%)',
    show=False
)
fig = plt.gcf()

# Dot borders in the main axis
main_ax = axes['mainplot_ax']
for coll in main_ax.collections:
    coll.set_linewidth(0.5)
    coll.set_edgecolor("gray")

# Style all axes (main + legends)
for ax_ in fig.get_axes():
    ax_.set_facecolor('white')
    ax_.tick_params(colors='black')
    ax_.title.set_color('black')
    for label in ax_.get_xticklabels() + ax_.get_yticklabels():
        label.set_color('black')
    for spine in ax_.spines.values():
        spine.set_edgecolor('black')

# Save (SVG)
fig.patch.set_facecolor('white')
base = Path(output_dir) / "20261113-Dotplot_Human_FibDec_subtype_Markers"
fig.savefig(base.with_suffix('.svg'), format='svg', bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f"Saved dotplot to {base.with_suffix('.svg')}")

### Immune Subtypes

In [ ]:
marker_genes = [
    'PTPRC',
    'CD163', 'MRC1', 'MERTK', 'C1QA',
    'CPA3', 'TPSB2', 'MS4A2', 'KIT',
    'CD4', 'CD3D', 'CD3G', 'CD3E',
    'CD8A', 'CD8B', 'DTHD1', 'IFNG-AS1', 'GZMK',
    'FOXP3', 'CTLA4', 'TSHZ2', 'TBC1D4',
    'RORC', 'IL1R1', 'IL23R', 'ABCB5', 'PCDH9',
    'MS4A1', 'CD79A',
    'NCAM1', 'KLRD1',
    'ITGAE', 'LDB2', 'CCL5',
    'ITGAX', 'KLRC1', 'GNLY', 'GZMB', 'PRF1',
]

# Desired row order (only these subtypes are plotted)
cell_order = [
     'Macrophages',
     'Mast_cells',
     'T_cells_CD4', 
     'T_cells_CD8',
     'Treg',
     'Rorc_lymphocytes',
     'B_cells',
     'uNK', 
     'uNK_cytotoxic',
]

# Settings
key = 'cell_subtype'
layer = 'log1p_counts'

missing = set(cell_order) - set(adata1.obs[key].astype(str).unique())
if missing:
    raise ValueError(f"Missing cell types in adata1.obs['{key}']: {missing}")

# Keep only the listed cell types
sub = adata1[adata1.obs[key].astype(str).isin(cell_order)].copy()
sub.obs[key] = pd.Categorical(sub.obs[key].astype(str), categories=cell_order)
excluded = adata1.obs.loc[~adata1.obs[key].astype(str).isin(cell_order), key].astype(str).value_counts()
print(f"Plotting {sub.n_obs} cells from {len(cell_order)} cell types; excluded {excluded.sum()} cells from {len(excluded)} other types")
print(sub.obs[key].value_counts().reindex(cell_order).to_string(), '\n')

# Drop duplicates and genes not in the data
marker_genes = list(dict.fromkeys(marker_genes))
missing_genes = [g for g in marker_genes if g not in sub.var_names]
if missing_genes:
    print(f"Warning: {len(missing_genes)} marker(s) not in var, dropped: {missing_genes}")
marker_genes = [g for g in marker_genes if g in sub.var_names]

# Dot plot: each gene scaled 0-1 across the listed groups
axes = sc.pl.dotplot(
    sub,
    var_names=marker_genes,
    groupby=key,
    layer=layer,
    use_raw=False,
    categories_order=cell_order,
    figsize=(max(5, 0.25 * len(marker_genes) + 2), max(1.0, len(cell_order) * 0.32)),
    standard_scale='var',
    colorbar_title='Scaled mean\nexpression',
    size_title='Fraction of\nnuclei (%)',
    show=False
)
fig = plt.gcf()

# Dot borders in the main axis
main_ax = axes['mainplot_ax']
for coll in main_ax.collections:
    coll.set_linewidth(0.5)
    coll.set_edgecolor("gray")

# Style all axes (main + legends)
for ax_ in fig.get_axes():
    ax_.set_facecolor('white')
    ax_.tick_params(colors='black')
    ax_.title.set_color('black')
    for label in ax_.get_xticklabels() + ax_.get_yticklabels():
        label.set_color('black')
    for spine in ax_.spines.values():
        spine.set_edgecolor('black')

# Save (SVG)
fig.patch.set_facecolor('white')
base = Path(output_dir) / "20261113-Dotplot_Human_Immune_subtype_Markers"
fig.savefig(base.with_suffix('.svg'), format='svg', bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f"Saved dotplot to {base.with_suffix('.svg')}")

### Epi subtypes

In [ ]:
marker_genes = [
    'FOXJ1', 'CAPS', 'RSPH1', 'CDHR3', 'DNAH12', 'DNAH9', 'DNAH5',
    'PAEP', 'IDO1', 'CLDN10', 'ITGB6', 'MMP7', 'LGR5', 'SPP1', 'GPX3',
    'PGR', 'ESR1', 'BCL2', 'GREM2', 'IHH', 'AXIN2', 'CDH2', 'ALDH1A1', 'BOC',
]

# Desired row order (only these subtypes are plotted)
cell_order = [
        'epi_ciliated',
        'epi1',
        'epi2',
]

# Settings
key = 'cell_subtype'
layer = 'log1p_counts'

missing = set(cell_order) - set(adata1.obs[key].astype(str).unique())
if missing:
    raise ValueError(f"Missing cell types in adata1.obs['{key}']: {missing}")

# Keep only the listed cell types
sub = adata1[adata1.obs[key].astype(str).isin(cell_order)].copy()
sub.obs[key] = pd.Categorical(sub.obs[key].astype(str), categories=cell_order)
excluded = adata1.obs.loc[~adata1.obs[key].astype(str).isin(cell_order), key].astype(str).value_counts()
print(f"Plotting {sub.n_obs} cells from {len(cell_order)} cell types; excluded {excluded.sum()} cells from {len(excluded)} other types")
print(sub.obs[key].value_counts().reindex(cell_order).to_string(), '\n')

# Drop duplicates and genes not in the data
marker_genes = list(dict.fromkeys(marker_genes))
missing_genes = [g for g in marker_genes if g not in sub.var_names]
if missing_genes:
    print(f"Warning: {len(missing_genes)} marker(s) not in var, dropped: {missing_genes}")
marker_genes = [g for g in marker_genes if g in sub.var_names]

# Dot plot: each gene scaled 0-1 across the listed groups
axes = sc.pl.dotplot(
    sub,
    var_names=marker_genes,
    groupby=key,
    layer=layer,
    use_raw=False,
    categories_order=cell_order,
    figsize=(max(5, 0.25 * len(marker_genes) + 2), max(1.0, len(cell_order) * 0.32)),
    standard_scale='var',
    colorbar_title='Scaled mean\nexpression',
    size_title='Fraction of\nnuclei (%)',
    show=False
)
fig = plt.gcf()

# Dot borders in the main axis
main_ax = axes['mainplot_ax']
for coll in main_ax.collections:
    coll.set_linewidth(0.5)
    coll.set_edgecolor("gray")

# Style all axes (main + legends)
for ax_ in fig.get_axes():
    ax_.set_facecolor('white')
    ax_.tick_params(colors='black')
    ax_.title.set_color('black')
    for label in ax_.get_xticklabels() + ax_.get_yticklabels():
        label.set_color('black')
    for spine in ax_.spines.values():
        spine.set_edgecolor('black')

# Save (SVG)
fig.patch.set_facecolor('white')
base = Path(output_dir) / "20261113-Dotplot_Human_epi_subtype_Markers"
fig.savefig(base.with_suffix('.svg'), format='svg', bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f"Saved dotplot to {base.with_suffix('.svg')}")

# Stacked Bar Plots

## Mouse

### Cell Types

In [ ]:
plt.rcParams['svg.fonttype'] = 'none'   # editable text in the SVG

# Settings
key        = 'cell_type'
group_key  = 'timepoint'
sample_key = 'sample'
mode       = 'per_sample'   # 'per_sample' = mean of per-sample proportions; 'pooled' = all cells pooled

group_order = ['uninduced', '1.5dpi_early', '1.5dpi_late', '2.5dpi', '3.5dpi', '4.5dpi_menstrual']
celltype_order = [
    'eFib',
    'dec',
    'epi',
    'imm',
    'vsc',
    'smc',
    'pm',
]

colors = {
    'dec':  '#00B4C5',  # cyan
    'epi':  '#D4B900',  # golden yellow
    'vsc':  '#5B7FE6',  # medium blue
    'eFib': '#CC79A7',  # reddish purple
    'imm':  '#D55E00',  # vermillion
    'smc':  '#4DAF4A',  # green
    'pm':   '#8C8C8C',  # grey
}

fontsize = 6
mm = 1 / 25.4
fig_w, fig_h = 45 * mm, 33 * mm   # size of the plotting area; labels and legend extend outside

missing_colors = [ct for ct in celltype_order if ct not in colors]
if missing_colors:
    raise ValueError(f"Missing colors for: {missing_colors}")

obs = adata3.obs
for col in [key, group_key, sample_key]:
    if col not in obs:
        raise ValueError(f"{col} not found in adata3.obs")

df = pd.DataFrame({
    'group':     obs[group_key].astype(str).values,
    'sample':    obs[sample_key].astype(str).values,
    'cell_type': obs[key].astype(str).values,
})

missing_groups = [g for g in group_order if g not in set(df['group'])]
if missing_groups:
    raise ValueError(f"Missing in adata3.obs['{group_key}']: {missing_groups}")

# Keep only the listed cell types and timepoints
unexpected = sorted(set(df['cell_type']) - set(celltype_order))
if unexpected:
    print(f"Excluded {df['cell_type'].isin(unexpected).sum():,} cells of other types: {unexpected}")
extra_groups = sorted(set(df['group']) - set(group_order))
if extra_groups:
    print(f"Excluded timepoints: {extra_groups}")
df = df[df['cell_type'].isin(celltype_order) & df['group'].isin(group_order)]

# Proportions
counts = (df.groupby(['group', 'cell_type']).size()
            .unstack(fill_value=0)
            .reindex(index=group_order, columns=celltype_order, fill_value=0))
n_cells = counts.sum(axis=1)
n_samples = df.groupby('group')['sample'].nunique().reindex(group_order)

if mode == 'per_sample':
    sample_counts = (df.groupby(['group', 'sample', 'cell_type']).size()
                       .unstack(fill_value=0)
                       .reindex(columns=celltype_order, fill_value=0))
    sample_perc = sample_counts.div(sample_counts.sum(axis=1), axis=0) * 100
    perc = sample_perc.groupby(level='group').mean().reindex(group_order)
elif mode == 'pooled':
    perc = counts.div(n_cells, axis=0) * 100
else:
    raise ValueError("mode must be 'per_sample' or 'pooled'")

print(perc.round(1).assign(n_samples=n_samples, n_cells=n_cells).to_string(), '\n')

# Stacked bar plot (top bar = first timepoint)
fig, ax = plt.subplots(figsize=(fig_w, fig_h))
y = np.arange(len(group_order))
left = np.zeros(len(y))
for ct in celltype_order:
    vals = perc[ct].values
    ax.barh(y, vals, left=left, height=0.8, color=colors[ct],
            edgecolor='none', linewidth=0, label=ct)
    left += vals

ax.set_xlim(0, 100)
ax.set_ylim(len(y) - 0.5, -0.5)

# Axes, ticks, labels
ax.set_yticks(y)
ax.set_yticklabels(group_order, fontsize=fontsize)
ax.set_xticks([0, 25, 50, 75, 100])
ax.tick_params(axis='x', labelsize=fontsize)
ax.tick_params(length=2, width=0.5, colors='black')
ax.set_xlabel('Proportion of nuclei (%)', fontsize=fontsize)
for side in ['top', 'right']:
    ax.spines[side].set_visible(False)
for side in ['left', 'bottom']:
    ax.spines[side].set_linewidth(0.5)
    ax.spines[side].set_edgecolor('black')

# Legend (same order as the stacking, left to right)
ax.legend(title='Cell type', fontsize=fontsize, title_fontsize=fontsize,
          loc='upper left', bbox_to_anchor=(1.02, 1), frameon=False,
          handlelength=1, handleheight=1, borderaxespad=0)

# Save SVG only
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
svg_path = Path(output_dir) / f"20261113-StackedBar_Mouse_All_CellTypeComposition_{mode}.svg"
fig.savefig(svg_path, format='svg', bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f"Saved {svg_path}")

### Immune Subtypes

In [ ]:
plt.rcParams['svg.fonttype'] = 'none'   # editable text in the SVG

# Settings
key        = 'cell_subtype'
group_key  = 'timepoint'
sample_key = 'sample'
mode       = 'per_sample'   # 'per_sample' = mean of per-sample proportions; 'pooled' = all cells pooled

group_order = ['uninduced', '1.5dpi_early', '1.5dpi_late', '2.5dpi', '3.5dpi', '4.5dpi_menstrual']
celltype_order = [
    'neutrophils',
    'monocytes',
    'Macrophages',
    'Macrophages_cycling',
    'DC',
    'T_cells',
    'Rorc_lymphocytes',
    'B_cells',
    'uNK',
    'uNK_cytotoxic',
    'uNK_cycling',
]

colors = {
    # myeloid (warm)
    'neutrophils':                    '#E69F00',  # orange (or '#A50F15' deep red)
    'monocytes':                      '#C9A227',  # gold
    'Macrophages':                    '#D55E00',  # vermillion (same as immune_cell)
    'Macrophages_cycling':            '#F28E5C',  # light vermillion
    'DC':                             '#8C5A2B',  # brown
    # lymphoid: T and B (blues, green)
    'T_cells':                        '#0072B2',  # blue
    'Rorc_lymphocytes':               '#56B4E9',  # sky blue
    'B_cells':                        '#009E73',  # bluish green
    # lymphoid: uNK (purples)
    'uNK':                            '#CC79A7',  # reddish purple
    'uNK_cytotoxic':                  '#7B5EA7',  # violet
    'uNK_cycling':                    '#B39DDB',  # light violet
}


fontsize = 6
mm = 1 / 25.4
fig_w, fig_h = 45 * mm, 33 * mm   # size of the plotting area; labels and legend extend outside

missing_colors = [ct for ct in celltype_order if ct not in colors]
if missing_colors:
    raise ValueError(f"Missing colors for: {missing_colors}")

obs = adata3.obs
for col in [key, group_key, sample_key]:
    if col not in obs:
        raise ValueError(f"{col} not found in adata3.obs")

df = pd.DataFrame({
    'group':     obs[group_key].astype(str).values,
    'sample':    obs[sample_key].astype(str).values,
    'cell_type': obs[key].astype(str).values,
})

missing_groups = [g for g in group_order if g not in set(df['group'])]
if missing_groups:
    raise ValueError(f"Missing in adata3.obs['{group_key}']: {missing_groups}")

# Keep only the listed subtypes and timepoints
unexpected = sorted(set(df['cell_type']) - set(celltype_order))
if unexpected:
    print(f"Excluded {df['cell_type'].isin(unexpected).sum():,} cells of other types: {unexpected}")
extra_groups = sorted(set(df['group']) - set(group_order))
if extra_groups:
    print(f"Excluded timepoints: {extra_groups}")
df = df[df['cell_type'].isin(celltype_order) & df['group'].isin(group_order)]

# Proportions
counts = (df.groupby(['group', 'cell_type']).size()
            .unstack(fill_value=0)
            .reindex(index=group_order, columns=celltype_order, fill_value=0))
n_cells = counts.sum(axis=1)
n_samples = df.groupby('group')['sample'].nunique().reindex(group_order)

if mode == 'per_sample':
    sample_counts = (df.groupby(['group', 'sample', 'cell_type']).size()
                       .unstack(fill_value=0)
                       .reindex(columns=celltype_order, fill_value=0))
    sample_perc = sample_counts.div(sample_counts.sum(axis=1), axis=0) * 100
    perc = sample_perc.groupby(level='group').mean().reindex(group_order)
elif mode == 'pooled':
    perc = counts.div(n_cells, axis=0) * 100
else:
    raise ValueError("mode must be 'per_sample' or 'pooled'")

print(perc.round(1).assign(n_samples=n_samples, n_cells=n_cells).to_string(), '\n')

# Stacked bar plot (top bar = first timepoint)
fig, ax = plt.subplots(figsize=(fig_w, fig_h))
y = np.arange(len(group_order))
left = np.zeros(len(y))
for ct in celltype_order:
    vals = perc[ct].values
    ax.barh(y, vals, left=left, height=0.8, color=colors[ct],
            edgecolor='none', linewidth=0, label=ct)
    left += vals

ax.set_xlim(0, 100)
ax.set_ylim(len(y) - 0.5, -0.5)

# Axes, ticks, labels
ax.set_yticks(y)
ax.set_yticklabels(group_order, fontsize=fontsize)
ax.set_xticks([0, 25, 50, 75, 100])
ax.tick_params(axis='x', labelsize=fontsize)
ax.tick_params(length=2, width=0.5, colors='black')
ax.set_xlabel('Proportion of immune nuclei (%)', fontsize=fontsize)
for side in ['top', 'right']:
    ax.spines[side].set_visible(False)
for side in ['left', 'bottom']:
    ax.spines[side].set_linewidth(0.5)
    ax.spines[side].set_edgecolor('black')

# Legend (same order as the stacking, left to right)
ax.legend(title='Immune subtype', fontsize=fontsize, title_fontsize=fontsize,
          loc='upper left', bbox_to_anchor=(1.02, 1), frameon=False,
          handlelength=1, handleheight=1, borderaxespad=0)

# Save SVG only
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
svg_path = Path(output_dir) / f"20261113-StackedBar_Mouse_Immune_SubTypeComposition_{mode}.svg"
fig.savefig(svg_path, format='svg', bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f"Saved {svg_path}")

### eFib/dec

In [ ]:
plt.rcParams['svg.fonttype'] = 'none'   # editable text in the SVG

# Settings
key        = 'cell_subtype'
group_key  = 'timepoint'
sample_key = 'sample'
mode       = 'per_sample'   # 'per_sample' = mean of per-sample proportions; 'pooled' = all cells pooled

group_order = ['uninduced', '1.5dpi_early', '1.5dpi_late', '2.5dpi', '3.5dpi', '4.5dpi_menstrual']
celltype_order = [
    'eFib1',
    'eFib2',
    'eFib2i',
    'eFib3',
    'eFibP',
    'dec1',
    'dec2',
    'dec3',
    'decAp',
]

colors = {
    # endometrial fibroblasts
    'eFib1':  '#a77207',  # ochre
    'eFib2':  '#00401d',  # dark green
    'eFib2i': '#127d35',  # green
    'eFib3':  '#4694c5',  # sky blue
    'eFibP':  '#a0e1b0',  # mint
    # decidual cells
    'dec1':   '#762a83',  # purple
    'dec2':   '#1e1ee2',  # blue
    'dec3':   '#b11a2d',  # red
    'decAp':  '#fccc3f',  # yellow
}

fontsize = 6
mm = 1 / 25.4
fig_w, fig_h = 45 * mm, 33 * mm   # size of the plotting area; labels and legend extend outside

missing_colors = [ct for ct in celltype_order if ct not in colors]
if missing_colors:
    raise ValueError(f"Missing colors for: {missing_colors}")

obs = adata3.obs
for col in [key, group_key, sample_key]:
    if col not in obs:
        raise ValueError(f"{col} not found in adata3.obs")

df = pd.DataFrame({
    'group':     obs[group_key].astype(str).values,
    'sample':    obs[sample_key].astype(str).values,
    'cell_type': obs[key].astype(str).values,
})

missing_groups = [g for g in group_order if g not in set(df['group'])]
if missing_groups:
    raise ValueError(f"Missing in adata3.obs['{group_key}']: {missing_groups}")

# Keep only the listed subtypes and timepoints
unexpected = sorted(set(df['cell_type']) - set(celltype_order))
if unexpected:
    print(f"Excluded {df['cell_type'].isin(unexpected).sum():,} cells of other types: {unexpected}")
extra_groups = sorted(set(df['group']) - set(group_order))
if extra_groups:
    print(f"Excluded timepoints: {extra_groups}")
df = df[df['cell_type'].isin(celltype_order) & df['group'].isin(group_order)]

# Proportions
counts = (df.groupby(['group', 'cell_type']).size()
            .unstack(fill_value=0)
            .reindex(index=group_order, columns=celltype_order, fill_value=0))
n_cells = counts.sum(axis=1)
n_samples = df.groupby('group')['sample'].nunique().reindex(group_order)

if mode == 'per_sample':
    sample_counts = (df.groupby(['group', 'sample', 'cell_type']).size()
                       .unstack(fill_value=0)
                       .reindex(columns=celltype_order, fill_value=0))
    sample_perc = sample_counts.div(sample_counts.sum(axis=1), axis=0) * 100
    perc = sample_perc.groupby(level='group').mean().reindex(group_order)
elif mode == 'pooled':
    perc = counts.div(n_cells, axis=0) * 100
else:
    raise ValueError("mode must be 'per_sample' or 'pooled'")

print(perc.round(1).assign(n_samples=n_samples, n_cells=n_cells).to_string(), '\n')

# Stacked bar plot (top bar = first timepoint)
fig, ax = plt.subplots(figsize=(fig_w, fig_h))
y = np.arange(len(group_order))
left = np.zeros(len(y))
for ct in celltype_order:
    vals = perc[ct].values
    ax.barh(y, vals, left=left, height=0.8, color=colors[ct],
            edgecolor='none', linewidth=0, label=ct)
    left += vals

ax.set_xlim(0, 100)
ax.set_ylim(len(y) - 0.5, -0.5)

# Axes, ticks, labels
ax.set_yticks(y)
ax.set_yticklabels(group_order, fontsize=fontsize)
ax.set_xticks([0, 25, 50, 75, 100])
ax.tick_params(axis='x', labelsize=fontsize)
ax.tick_params(length=2, width=0.5, colors='black')
ax.set_xlabel('Proportion of stromal nuclei (%)', fontsize=fontsize)
for side in ['top', 'right']:
    ax.spines[side].set_visible(False)
for side in ['left', 'bottom']:
    ax.spines[side].set_linewidth(0.5)
    ax.spines[side].set_edgecolor('black')

# Legend (same order as the stacking, left to right)
ax.legend(title='Stromal subtype', fontsize=fontsize, title_fontsize=fontsize,
          loc='upper left', bbox_to_anchor=(1.02, 1), frameon=False,
          handlelength=1, handleheight=1, borderaxespad=0)

# Save SVG only
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
svg_path = Path(output_dir) / f"20261113-StackedBar_Mouse_eFibDec_SubTypeComposition_{mode}.svg"
fig.savefig(svg_path, format='svg', bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f"Saved {svg_path}")

## Human

### All cell types

In [ ]:
plt.rcParams['svg.fonttype'] = 'none'   # editable text in the SVG

# Settings
key        = 'cell_type'
group_key  = 'timepoint'
sample_key = 'sample'
mode       = 'per_sample'   # 'per_sample' = mean of per-sample proportions; 'pooled' = all cells pooled

group_order = ['secretory', 'menstrual']
celltype_order = [
    'eFib',
    'dec',
    'epi',
    'imm',
    'vsc',
    'smc',
]

colors = {
    'eFib': '#CC79A7',  # reddish purple
    'dec':  '#00B4C5',  # cyan
    'epi':  '#D4B900',  # golden yellow
    'imm':  '#D55E00',  # vermillion
    'vsc':  '#5B7FE6',  # medium blue
    'smc':  '#4DAF4A',  # green
}

fontsize = 6
mm = 1 / 25.4
fig_w, fig_h = 45 * mm, 11 * mm   # 11 mm for 2 bars ≈ same bar thickness as the 6-timepoint mouse plots

missing_colors = [ct for ct in celltype_order if ct not in colors]
if missing_colors:
    raise ValueError(f"Missing colors for: {missing_colors}")

obs = adata1.obs
for col in [key, group_key, sample_key]:
    if col not in obs:
        raise ValueError(f"{col} not found in adata1.obs")

df = pd.DataFrame({
    'group':     obs[group_key].astype(str).values,
    'sample':    obs[sample_key].astype(str).values,
    'cell_type': obs[key].astype(str).values,
})

missing_groups = [g for g in group_order if g not in set(df['group'])]
if missing_groups:
    raise ValueError(f"Missing in adata1.obs['{group_key}']: {missing_groups}")

# Keep only the listed cell types and timepoints
unexpected = sorted(set(df['cell_type']) - set(celltype_order))
if unexpected:
    print(f"Excluded {df['cell_type'].isin(unexpected).sum():,} cells of other types: {unexpected}")
extra_groups = sorted(set(df['group']) - set(group_order))
if extra_groups:
    print(f"Excluded timepoints: {extra_groups}")
df = df[df['cell_type'].isin(celltype_order) & df['group'].isin(group_order)]

# Proportions
counts = (df.groupby(['group', 'cell_type']).size()
            .unstack(fill_value=0)
            .reindex(index=group_order, columns=celltype_order, fill_value=0))
n_cells = counts.sum(axis=1)
n_samples = df.groupby('group')['sample'].nunique().reindex(group_order)

if mode == 'per_sample':
    sample_counts = (df.groupby(['group', 'sample', 'cell_type']).size()
                       .unstack(fill_value=0)
                       .reindex(columns=celltype_order, fill_value=0))
    sample_perc = sample_counts.div(sample_counts.sum(axis=1), axis=0) * 100
    perc = sample_perc.groupby(level='group').mean().reindex(group_order)
elif mode == 'pooled':
    perc = counts.div(n_cells, axis=0) * 100
else:
    raise ValueError("mode must be 'per_sample' or 'pooled'")

print(perc.round(1).assign(n_samples=n_samples, n_cells=n_cells).to_string(), '\n')

# Stacked bar plot (top bar = first timepoint)
fig, ax = plt.subplots(figsize=(fig_w, fig_h))
y = np.arange(len(group_order))
left = np.zeros(len(y))
for ct in celltype_order:
    vals = perc[ct].values
    ax.barh(y, vals, left=left, height=0.8, color=colors[ct],
            edgecolor='none', linewidth=0, label=ct)
    left += vals

ax.set_xlim(0, 100)
ax.set_ylim(len(y) - 0.5, -0.5)

# Axes, ticks, labels
ax.set_yticks(y)
ax.set_yticklabels(group_order, fontsize=fontsize)
ax.set_xticks([0, 25, 50, 75, 100])
ax.tick_params(axis='x', labelsize=fontsize)
ax.tick_params(length=2, width=0.5, colors='black')
ax.set_xlabel('Proportion of nuclei (%)', fontsize=fontsize)
for side in ['top', 'right']:
    ax.spines[side].set_visible(False)
for side in ['left', 'bottom']:
    ax.spines[side].set_linewidth(0.5)
    ax.spines[side].set_edgecolor('black')

# Legend (same order as the stacking, left to right)
ax.legend(title='Cell type', fontsize=fontsize, title_fontsize=fontsize,
          loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False, ncol=2,
          handlelength=1, handleheight=1, borderaxespad=0, columnspacing=1)

# Save SVG only
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
svg_path = Path(output_dir) / f"20261113-StackedBar_Human_All_CellTypeComposition_{mode}.svg"
fig.savefig(svg_path, format='svg', bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f"Saved {svg_path}")

### Only Immune

In [ ]:
plt.rcParams['svg.fonttype'] = 'none'   # editable text in the SVG

# Settings
key        = 'cell_subtype'
group_key  = 'timepoint'
sample_key = 'sample'
mode       = 'per_sample'   # 'per_sample' = mean of per-sample proportions; 'pooled' = all cells pooled

group_order = ['secretory', 'menstrual']

colors = {
    # myeloid (warm)
    'Macrophages':      '#D55E00',  # vermillion (same as mouse)
    'Mast_cells':       '#B2182B',  # crimson
    # lymphoid: T cells (blues)
    'T_cells_CD4':      '#0072B2',  # blue (= mouse T_cells)
    'T_cells_CD8':      '#1B2A6B',  # navy
    'Treg':             '#6C8EBF',  # slate blue
    'Rorc_lymphocytes': '#56B4E9',  # sky blue (same as mouse)
    # lymphoid: B cells
    'B_cells':          '#009E73',  # bluish green (same as mouse)
    # lymphoid: uNK (purples)
    'uNK':              '#CC79A7',  # reddish purple (same as mouse)
    'uNK_cytotoxic':    '#7B5EA7',  # violet (same as mouse)
}
celltype_order = list(colors)   # stacking and legend follow the color map order

fontsize = 6
mm = 1 / 25.4
fig_w, fig_h = 45 * mm, 11 * mm   # 11 mm for 2 bars ≈ same bar thickness as the mouse plots

obs = adata1.obs
for col in [key, group_key, sample_key]:
    if col not in obs:
        raise ValueError(f"{col} not found in adata1.obs")

df = pd.DataFrame({
    'group':     obs[group_key].astype(str).values,
    'sample':    obs[sample_key].astype(str).values,
    'cell_type': obs[key].astype(str).values,
})

missing_groups = [g for g in group_order if g not in set(df['group'])]
if missing_groups:
    raise ValueError(f"Missing in adata1.obs['{group_key}']: {missing_groups}")

missing_types = [ct for ct in celltype_order if ct not in set(df['cell_type'])]
if missing_types:
    print(f"Warning: subtypes not found in adata1.obs['{key}']: {missing_types}")

# Keep only the listed subtypes and timepoints
unexpected = sorted(set(df['cell_type']) - set(celltype_order))
if unexpected:
    print(f"Excluded {df['cell_type'].isin(unexpected).sum():,} cells of other types: {unexpected}")
extra_groups = sorted(set(df['group']) - set(group_order))
if extra_groups:
    print(f"Excluded timepoints: {extra_groups}")
df = df[df['cell_type'].isin(celltype_order) & df['group'].isin(group_order)]

# Proportions
counts = (df.groupby(['group', 'cell_type']).size()
            .unstack(fill_value=0)
            .reindex(index=group_order, columns=celltype_order, fill_value=0))
n_cells = counts.sum(axis=1)
n_samples = df.groupby('group')['sample'].nunique().reindex(group_order)

if mode == 'per_sample':
    sample_counts = (df.groupby(['group', 'sample', 'cell_type']).size()
                       .unstack(fill_value=0)
                       .reindex(columns=celltype_order, fill_value=0))
    sample_perc = sample_counts.div(sample_counts.sum(axis=1), axis=0) * 100
    perc = sample_perc.groupby(level='group').mean().reindex(group_order)
elif mode == 'pooled':
    perc = counts.div(n_cells, axis=0) * 100
else:
    raise ValueError("mode must be 'per_sample' or 'pooled'")

print(perc.round(1).assign(n_samples=n_samples, n_cells=n_cells).to_string(), '\n')

# Stacked bar plot (top bar = first timepoint)
fig, ax = plt.subplots(figsize=(fig_w, fig_h))
y = np.arange(len(group_order))
left = np.zeros(len(y))
for ct in celltype_order:
    vals = perc[ct].values
    ax.barh(y, vals, left=left, height=0.8, color=colors[ct],
            edgecolor='none', linewidth=0, label=ct)
    left += vals

ax.set_xlim(0, 100)
ax.set_ylim(len(y) - 0.5, -0.5)

# Axes, ticks, labels
ax.set_yticks(y)
ax.set_yticklabels(group_order, fontsize=fontsize)
ax.set_xticks([0, 25, 50, 75, 100])
ax.tick_params(axis='x', labelsize=fontsize)
ax.tick_params(length=2, width=0.5, colors='black')
ax.set_xlabel('Proportion of immune nuclei (%)', fontsize=fontsize)
for side in ['top', 'right']:
    ax.spines[side].set_visible(False)
for side in ['left', 'bottom']:
    ax.spines[side].set_linewidth(0.5)
    ax.spines[side].set_edgecolor('black')

# Legend (same order as the stacking, left to right)
ax.legend(title='Immune subtype', fontsize=fontsize, title_fontsize=fontsize,
          loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False, ncol=2,
          handlelength=1, handleheight=1, borderaxespad=0, columnspacing=1)

# Save SVG only
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
svg_path = Path(output_dir) / f"20261113-StackedBar_Human_Immune_SubTypeComposition_{mode}.svg"
fig.savefig(svg_path, format='svg', bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f"Saved {svg_path}")

### Fib/dec

In [ ]:
plt.rcParams['svg.fonttype'] = 'none'   # editable text in the SVG

# Settings
key        = 'cell_subtype'
group_key  = 'timepoint'
sample_key = 'sample'
mode       = 'per_sample'   # 'per_sample' = mean of per-sample proportions; 'pooled' = all cells pooled

group_order = ['secretory', 'menstrual']

colors = {
    # fibroblasts (olive + teals)
    'myoFib': '#8c8c00',  # olive
    'eFib_a': '#01665e',  # dark teal
    'eFib_b': '#35978f',  # teal
    'eFib_c': '#80cdc1',  # light teal
    # decidual cells (magenta / pink / coral)
    'dec_a':  '#8e0152',  # dark magenta
    'dec_b':  '#de77ae',  # pink
    'dec_c':  '#f46d43',  # coral
    'dec_d':  '#f4a582',  # salmon
}
celltype_order = list(colors)   # stacking and legend follow the color map order

fontsize = 6
mm = 1 / 25.4
fig_w, fig_h = 45 * mm, 11 * mm   # 11 mm for 2 bars ≈ same bar thickness as the mouse plots

obs = adata1.obs
for col in [key, group_key, sample_key]:
    if col not in obs:
        raise ValueError(f"{col} not found in adata1.obs")

df = pd.DataFrame({
    'group':     obs[group_key].astype(str).values,
    'sample':    obs[sample_key].astype(str).values,
    'cell_type': obs[key].astype(str).values,
})

missing_groups = [g for g in group_order if g not in set(df['group'])]
if missing_groups:
    raise ValueError(f"Missing in adata1.obs['{group_key}']: {missing_groups}")

missing_types = [ct for ct in celltype_order if ct not in set(df['cell_type'])]
if missing_types:
    print(f"Warning: subtypes not found in adata1.obs['{key}']: {missing_types}")

# Keep only the listed subtypes and timepoints
unexpected = sorted(set(df['cell_type']) - set(celltype_order))
if unexpected:
    print(f"Excluded {df['cell_type'].isin(unexpected).sum():,} cells of other types: {unexpected}")
extra_groups = sorted(set(df['group']) - set(group_order))
if extra_groups:
    print(f"Excluded timepoints: {extra_groups}")
df = df[df['cell_type'].isin(celltype_order) & df['group'].isin(group_order)]

# Proportions
counts = (df.groupby(['group', 'cell_type']).size()
            .unstack(fill_value=0)
            .reindex(index=group_order, columns=celltype_order, fill_value=0))
n_cells = counts.sum(axis=1)
n_samples = df.groupby('group')['sample'].nunique().reindex(group_order)

if mode == 'per_sample':
    sample_counts = (df.groupby(['group', 'sample', 'cell_type']).size()
                       .unstack(fill_value=0)
                       .reindex(columns=celltype_order, fill_value=0))
    sample_perc = sample_counts.div(sample_counts.sum(axis=1), axis=0) * 100
    perc = sample_perc.groupby(level='group').mean().reindex(group_order)
elif mode == 'pooled':
    perc = counts.div(n_cells, axis=0) * 100
else:
    raise ValueError("mode must be 'per_sample' or 'pooled'")

print(perc.round(1).assign(n_samples=n_samples, n_cells=n_cells).to_string(), '\n')

# Stacked bar plot (top bar = first timepoint)
fig, ax = plt.subplots(figsize=(fig_w, fig_h))
y = np.arange(len(group_order))
left = np.zeros(len(y))
for ct in celltype_order:
    vals = perc[ct].values
    ax.barh(y, vals, left=left, height=0.8, color=colors[ct],
            edgecolor='none', linewidth=0, label=ct)
    left += vals

ax.set_xlim(0, 100)
ax.set_ylim(len(y) - 0.5, -0.5)

# Axes, ticks, labels
ax.set_yticks(y)
ax.set_yticklabels(group_order, fontsize=fontsize)
ax.set_xticks([0, 25, 50, 75, 100])
ax.tick_params(axis='x', labelsize=fontsize)
ax.tick_params(length=2, width=0.5, colors='black')
ax.set_xlabel('Proportion of stromal nuclei (%)', fontsize=fontsize)
for side in ['top', 'right']:
    ax.spines[side].set_visible(False)
for side in ['left', 'bottom']:
    ax.spines[side].set_linewidth(0.5)
    ax.spines[side].set_edgecolor('black')

# Legend (same order as the stacking, left to right)
ax.legend(title='Stromal subtype', fontsize=fontsize, title_fontsize=fontsize,
          loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False, ncol=2,
          handlelength=1, handleheight=1, borderaxespad=0, columnspacing=1)

# Save SVG only
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
svg_path = Path(output_dir) / f"20261113-StackedBar_Human_FibDec_SubTypeComposition_{mode}.svg"
fig.savefig(svg_path, format='svg', bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f"Saved {svg_path}")

# Spatial Plots

## Mouse

### Cell types

In [ ]:
# Settings
key_cluster = "cell_type"
key_sample  = "timepoint"
x_key, y_key = "x_um_dbscan", "y_um_dbscan"

# Per timepoint: rotation (counter-clockwise) and window in the rotated image (order = grid order)
panels = {
    "uninduced":        {"rot": 322.75, "view": [(2050, 4650), (-300, 2300)]},
    "1.5dpi_early":     {"rot": 28,     "view": [(750, 2800),  (925, 2975)]},
    "1.5dpi_late":      {"rot": 303,    "view": [(150, 6250),  (-300, 5800)]},
    "2.5dpi":           {"rot": 175,    "view": [(2900, 9600), (4200, 10900)]},
    "3.5dpi":           {"rot": 135,    "view": [(2600, 9100), (2200, 8700)]},
    "4.5dpi_menstrual": {"rot": 81,     "view": [(900, 9700),  (1350, 10150)]},
}

color_map = {
    'dec':  '#00B4C5',
    'eFib': '#CC79A7',
    'vsc':  '#5B7FE6',
    'epi':  '#D4B900',
    'imm':  '#D55E00',
    'pm':   '#8C8C8C',
    'smc':  '#4DAF4A',
}
draw_order  = ['smc', 'vsc', 'imm', 'dec', 'epi', 'eFib', 'pm']   # later = on top
other_color = "#B0B0B0"
dot_size, dot_alpha = 25, 1.0
fig_size_in, dpi_save = 6, 300        # reference size for dot scaling
grid_panel_in = 3                     # size of each panel in the combined image (inches)
grid_layout   = "row"                 # "row" = side by side; "column" = stacked
background    = "black"
SHOW_AXES     = False                 # True to check windows with coordinates (nothing is saved)

# ---------------- checks ----------------
for col in [key_cluster, key_sample, x_key, y_key]:
    if col not in adata3.obs:
        raise ValueError(f"{col} not found in adata3.obs")
no_color = sorted(set(adata3.obs[key_cluster].astype(str)) - set(color_map))
if no_color:
    print(f"Warning: no color for {no_color}; drawn as {other_color}")

# ---------------- rotate each section once ----------------
rotated = {}
for timepoint, cfg in panels.items():
    obs = adata3.obs
    mask = obs[key_sample].astype(str) == timepoint
    if not mask.any():
        print(f"Warning: no nuclei for {timepoint}. Skipping.")
        continue
    x   = obs.loc[mask, x_key].astype(float).to_numpy()
    y   = obs.loc[mask, y_key].astype(float).to_numpy()
    lab = obs.loc[mask, key_cluster].astype(str).to_numpy()
    ok = np.isfinite(x) & np.isfinite(y)
    x, y, lab = x[ok], y[ok], lab[ok]

    # rotate around the center of the whole section
    cx, cy = 0.5 * (x.min() + x.max()), 0.5 * (y.min() + y.max())
    t = np.deg2rad(cfg["rot"])
    xr = (x - cx) * np.cos(t) - (y - cy) * np.sin(t) + cx
    yr = (x - cx) * np.sin(t) + (y - cy) * np.cos(t) + cy
    rotated[timepoint] = (xr, yr, lab)

    (vx0, vx1), (vy0, vy1) = cfg["view"]
    n_in = ((xr >= vx0) & (xr <= vx1) & (yr >= vy0) & (yr <= vy1)).sum()
    print(f"{timepoint}: rotation={cfg['rot']}°, {n_in} of {xr.size} nuclei in view")

def draw_panel(ax, timepoint, dot_s):
    xr, yr, lab = rotated[timepoint]
    (vx0, vx1), (vy0, vy1) = panels[timepoint]["view"]
    uncolored = ~np.isin(lab, list(color_map))
    if uncolored.any():
        ax.scatter(xr[uncolored], yr[uncolored], s=dot_s, color=other_color,
                   alpha=dot_alpha, linewidth=0, zorder=1)
    for ct in draw_order:
        m = lab == ct
        if m.any():
            ax.scatter(xr[m], yr[m], s=dot_s, color=color_map[ct],
                       alpha=dot_alpha, linewidth=0, zorder=2)
    ax.set_xlim(vx0, vx1)
    ax.set_ylim(vy0, vy1)
    ax.set_aspect("equal", adjustable="box")
    ax.set_facecolor(background)

order = [tp for tp in panels if tp in rotated]
fg = 'white' if background == 'black' else 'black'

if SHOW_AXES:
    # Check mode: one panel at a time with coordinates, nothing saved
    for timepoint in order:
        fig, ax = plt.subplots(figsize=(fig_size_in, fig_size_in))
        draw_panel(ax, timepoint, dot_size)
        fig.patch.set_facecolor(background)
        ax.set_title(timepoint, color=fg)
        ax.tick_params(colors=fg, labelsize=7)
        ax.grid(True, color='gray', alpha=0.3, lw=0.5)
        plt.show()
else:
    # Save a combined image, panels touching, no labels
    n = len(order)
    dot_size_grid = dot_size * (grid_panel_in / fig_size_in) ** 2
    if grid_layout == "row":
        fig = plt.figure(figsize=(grid_panel_in * n, grid_panel_in))
        positions = [[i / n, 0, 1 / n, 1] for i in range(n)]
    else:
        fig = plt.figure(figsize=(grid_panel_in, grid_panel_in * n))
        positions = [[0, 1 - (i + 1) / n, 1, 1 / n] for i in range(n)]

    for timepoint, pos in zip(order, positions):
        ax = fig.add_axes(pos)
        draw_panel(ax, timepoint, dot_size_grid)
        ax.axis("off")

    fig.patch.set_facecolor(background)
    outpath = os.path.join(output_dir, f"20261113-Spatial_Mouse-celltype-spatial-grid-{grid_layout}.png")
    fig.savefig(outpath, dpi=dpi_save, facecolor=background, pad_inches=0)
    plt.show()
    print(f"saved {outpath}")

### All cell types, grid

In [ ]:
### Mouse spatial grid: rows = timepoints, columns = cell types (rotate, then crop) ###

plt.rcParams['svg.fonttype'] = 'none'   # labels stay editable text in the SVG

# Settings
key_cluster = "cell_type"
key_sample  = "timepoint"
x_key, y_key = "x_um_dbscan", "y_um_dbscan"

# Per timepoint: rotation (counter-clockwise) and window in the rotated image
# (order = rows, top to bottom)
panels = {
    "uninduced":        {"rot": 322.75, "view": [(2050, 4650), (-300, 2300)]},
    "1.5dpi_early":     {"rot": 28,     "view": [(750, 2800),  (925, 2975)]},
    "1.5dpi_late":      {"rot": 303,    "view": [(150, 6250),  (-300, 5800)]},
    "2.5dpi":           {"rot": 175,    "view": [(2900, 9600), (4200, 10900)]},
    "3.5dpi":           {"rot": 135,    "view": [(2600, 9100), (2200, 8700)]},
    "4.5dpi_menstrual": {"rot": 81,     "view": [(900, 9700),  (1350, 10150)]},
}

# Columns, left to right: cell_type label -> column title
label_names = {
    'epi':  'epithelial cell',
    'eFib': 'endometrial fibroblast',
    'dec':  'decidual cell',
    'vsc':  'endothelial cell',
    'imm':  'immune cell',
    'smc':  'smooth muscle cell',
    'pm':   'perimetrial cell',
}

highlight_color = "#fe01fe"
other_color     = "#B0B0B0"
dot_size, dot_alpha = 25, 1.0
fig_size_in    = 6             # reference: size of a single image
panel_in       = 2.0           # size of each grid panel (inches)
gap_in         = 0.1           # space between panels (inches); 0 = touching
show_labels    = True          # row/column labels outside the grid
label_fontsize = 9
label_pad_in   = 0.35          # room for labels (inches)
background     = "black"
dpi_save       = 300           # PNG resolution and resolution of the rasterized panels in the SVG

basename = "20261113-Spatial_Mouse-CellTypeHighlight-Grid-spatial"

# ---------------- checks ----------------
for col in [key_cluster, key_sample, x_key, y_key]:
    if col not in adata3.obs:
        raise ValueError(f"{col} not found in adata3.obs")
present = set(adata3.obs[key_cluster].astype(str))
col_order = [c for c in label_names if c in present]
skipped = [c for c in label_names if c not in present]
if skipped:
    print(f"Warning: cell types not in adata3.obs['{key_cluster}'], skipped: {skipped}")

# ---------------- rotate each section once ----------------
rotated = {}
for timepoint, cfg in panels.items():
    obs = adata3.obs
    mask = (obs[key_sample].astype(str) == timepoint).to_numpy()
    if not mask.any():
        print(f"Warning: no nuclei for {timepoint}. Skipping.")
        continue
    x   = obs.loc[mask, x_key].astype(float).to_numpy()
    y   = obs.loc[mask, y_key].astype(float).to_numpy()
    lab = obs.loc[mask, key_cluster].astype(str).to_numpy()
    ok = np.isfinite(x) & np.isfinite(y)
    x, y, lab = x[ok], y[ok], lab[ok]

    # rotate around the center of the whole section
    cx, cy = 0.5 * (x.min() + x.max()), 0.5 * (y.min() + y.max())
    t = np.deg2rad(cfg["rot"])
    xr = (x - cx) * np.cos(t) - (y - cy) * np.sin(t) + cx
    yr = (x - cx) * np.sin(t) + (y - cy) * np.cos(t) + cy
    rotated[timepoint] = (xr, yr, lab)
    print(f"{timepoint}: rotation={cfg['rot']}°, n_points={xr.size}")

def draw_panel(ax, timepoint, cell_type, dot_s):
    xr, yr, lab = rotated[timepoint]
    (vx0, vx1), (vy0, vy1) = panels[timepoint]["view"]
    hit = lab == cell_type
    if (~hit).any():
        ax.scatter(xr[~hit], yr[~hit], s=dot_s, color=other_color,
                   alpha=dot_alpha, linewidth=0, zorder=1, rasterized=True)
    if hit.any():
        ax.scatter(xr[hit], yr[hit], s=dot_s, color=highlight_color,
                   alpha=dot_alpha, linewidth=0, zorder=2, rasterized=True)
    ax.set_xlim(vx0, vx1)
    ax.set_ylim(vy0, vy1)
    ax.set_aspect("equal", adjustable="box")
    ax.set_facecolor(background)
    ax.axis("off")

row_order = [tp for tp in panels if tp in rotated]

# ---------------- grid figure ----------------
n_rows, n_cols = len(row_order), len(col_order)
left_in = label_pad_in if show_labels else 0.0     # room for row labels
top_in  = label_pad_in if show_labels else 0.0     # room for column titles
W = left_in + n_cols * panel_in + (n_cols - 1) * gap_in
H = top_in  + n_rows * panel_in + (n_rows - 1) * gap_in
dot_size_grid = dot_size * (panel_in / fig_size_in) ** 2   # same visual density as the single images
text_color = 'white' if background == 'black' else 'black'

fig = plt.figure(figsize=(W, H))
fig.patch.set_facecolor(background)

for r, tp in enumerate(row_order):
    bottom = (H - top_in - (r + 1) * panel_in - r * gap_in) / H
    for c, ct in enumerate(col_order):
        left = (left_in + c * (panel_in + gap_in)) / W
        ax = fig.add_axes([left, bottom, panel_in / W, panel_in / H])
        draw_panel(ax, tp, ct, dot_size_grid)

        if show_labels and r == 0:
            fig.text(left + panel_in / W / 2, (H - top_in + 0.05) / H, label_names[ct],
                     ha='center', va='bottom', fontsize=label_fontsize, color=text_color)
        if show_labels and c == 0:
            fig.text((left_in - 0.05) / W, bottom + panel_in / H / 2, tp,
                     ha='right', va='center', rotation=90, fontsize=label_fontsize, color=text_color)

# ---------------- save ----------------
png_path = os.path.join(output_dir, f"{basename}.png")
svg_path = os.path.join(output_dir, f"{basename}.svg")
fig.savefig(png_path, dpi=dpi_save, facecolor=background, pad_inches=0)
fig.savefig(svg_path, format='svg', dpi=dpi_save, facecolor=background, pad_inches=0)
plt.close(fig)
print(f"Saved {png_path} (and .svg)")

### eFib/dec Subtypes

In [ ]:
### Mouse spatial grid: one panel per timepoint, colored by eFib/dec subtype (rotate, then crop) ###

# Settings
key_cluster = "cell_subtype"
key_sample  = "timepoint"
x_key, y_key = "x_um_dbscan", "y_um_dbscan"

# Per timepoint: rotation (counter-clockwise) and window in the rotated image (order = grid order)
panels = {
    "uninduced":        {"rot": 322.75, "view": [(2050, 4650), (-300, 2300)]},
    "1.5dpi_early":     {"rot": 28,     "view": [(750, 2800),  (925, 2975)]},
    "1.5dpi_late":      {"rot": 303,    "view": [(150, 6250),  (-300, 5800)]},
    "2.5dpi":           {"rot": 175,    "view": [(2900, 9600), (4200, 10900)]},
    "3.5dpi":           {"rot": 135,    "view": [(2600, 9100), (2200, 8700)]},
    "4.5dpi_menstrual": {"rot": 81,     "view": [(900, 9700),  (1350, 10150)]},
}

color_map = {
    # endometrial fibroblasts
    'eFib1':  '#a77207',  # ochre
    'eFib2':  '#00401d',  # dark green
    'eFib2i': '#127d35',  # green
    'eFib3':  '#4694c5',  # sky blue
    'eFibP':  '#a0e1b0',  # mint
    # decidual cells
    'dec1':   '#762a83',  # purple
    'dec2':   '#1e1ee2',  # blue
    'dec3':   '#b11a2d',  # red
    'decAp':  '#fccc3f',  # yellow
}
draw_order  = ['eFib1', 'eFib2i', 'eFib2', 'eFib3', 'eFibP',
               'dec2', 'dec1', 'dec3', 'decAp']        # later = on top
other_color = "#231f20"       # every nucleus that isn't an eFib/dec subtype
dot_size, dot_alpha = 25, 1.0
fig_size_in, dpi_save = 6, 300        # reference size for dot scaling
grid_panel_in = 3                     # size of each panel in the combined image (inches)
grid_layout   = "row"                 # "row" = side by side; "column" = stacked
background    = "black"
SHOW_AXES     = False                 # True to check windows with coordinates (nothing is saved)

# ---------------- checks ----------------
for col in [key_cluster, key_sample, x_key, y_key]:
    if col not in adata3.obs:
        raise ValueError(f"{col} not found in adata3.obs")
present = set(adata3.obs[key_cluster].astype(str))
not_found = [s for s in color_map if s not in present]
if not_found:
    print(f"Warning: subtypes in color_map but not in adata3: {not_found}")

# ---------------- rotate each section once ----------------
rotated = {}
for timepoint, cfg in panels.items():
    obs = adata3.obs
    mask = (obs[key_sample].astype(str) == timepoint).to_numpy()
    if not mask.any():
        print(f"Warning: no nuclei for {timepoint}. Skipping.")
        continue
    x   = obs.loc[mask, x_key].astype(float).to_numpy()
    y   = obs.loc[mask, y_key].astype(float).to_numpy()
    lab = obs.loc[mask, key_cluster].astype(str).to_numpy()
    ok = np.isfinite(x) & np.isfinite(y)
    x, y, lab = x[ok], y[ok], lab[ok]

    # rotate around the center of the whole section (same as the cell-type images)
    cx, cy = 0.5 * (x.min() + x.max()), 0.5 * (y.min() + y.max())
    t = np.deg2rad(cfg["rot"])
    xr = (x - cx) * np.cos(t) - (y - cy) * np.sin(t) + cx
    yr = (x - cx) * np.sin(t) + (y - cy) * np.cos(t) + cy
    rotated[timepoint] = (xr, yr, lab)

    n_sub = np.isin(lab, list(color_map)).sum()
    print(f"{timepoint}: rotation={cfg['rot']}°, {n_sub} eFib/dec nuclei of {xr.size}")

def draw_panel(ax, timepoint, dot_s):
    xr, yr, lab = rotated[timepoint]
    (vx0, vx1), (vy0, vy1) = panels[timepoint]["view"]
    other = ~np.isin(lab, list(color_map))
    if other.any():
        ax.scatter(xr[other], yr[other], s=dot_s, color=other_color,
                   alpha=dot_alpha, linewidth=0, zorder=1)
    for st in draw_order:
        m = lab == st
        if m.any():
            ax.scatter(xr[m], yr[m], s=dot_s, color=color_map[st],
                       alpha=dot_alpha, linewidth=0, zorder=2)
    ax.set_xlim(vx0, vx1)
    ax.set_ylim(vy0, vy1)
    ax.set_aspect("equal", adjustable="box")
    ax.set_facecolor(background)

order = [tp for tp in panels if tp in rotated]
fg = 'white' if background == 'black' else 'black'

if SHOW_AXES:
    # Check mode: one panel at a time with coordinates, nothing saved
    for timepoint in order:
        fig, ax = plt.subplots(figsize=(fig_size_in, fig_size_in))
        draw_panel(ax, timepoint, dot_size)
        fig.patch.set_facecolor(background)
        ax.set_title(timepoint, color=fg)
        ax.tick_params(colors=fg, labelsize=7)
        ax.grid(True, color='gray', alpha=0.3, lw=0.5)
        plt.show()
else:
    # Save a combined image, panels touching, no labels
    n = len(order)
    dot_size_grid = dot_size * (grid_panel_in / fig_size_in) ** 2
    if grid_layout == "row":
        fig = plt.figure(figsize=(grid_panel_in * n, grid_panel_in))
        positions = [[i / n, 0, 1 / n, 1] for i in range(n)]
    else:
        fig = plt.figure(figsize=(grid_panel_in, grid_panel_in * n))
        positions = [[0, 1 - (i + 1) / n, 1, 1 / n] for i in range(n)]

    for timepoint, pos in zip(order, positions):
        ax = fig.add_axes(pos)
        draw_panel(ax, timepoint, dot_size_grid)
        ax.axis("off")

    fig.patch.set_facecolor(background)
    outpath = os.path.join(output_dir, f"20261113-Spatial_Mouse-eFibDec-spatial-grid-{grid_layout}.png")
    fig.savefig(outpath, dpi=dpi_save, facecolor=background, pad_inches=0)
    plt.close()
    print(f"saved {outpath}")

### eFib/dec Subtypes Grid

In [ ]:
### Mouse spatial grid: rows = timepoints, columns = eFib/dec subtypes (rotate, then crop) ###

plt.rcParams['svg.fonttype'] = 'none'   # labels stay editable text in the SVG

# ---------------- settings ----------------
key_cluster = "cell_subtype"
key_sample  = "timepoint"
x_key, y_key = "x_um_dbscan", "y_um_dbscan"

# rows: per timepoint rotation (counter-clockwise) and window in the ROTATED image
panels = {
    "uninduced":        {"rot": 322.75, "view": [(2050, 4650), (-300, 2300)]},
    "1.5dpi_early":     {"rot": 28,     "view": [(750, 2800),  (925, 2975)]},
    "1.5dpi_late":      {"rot": 303,    "view": [(150, 6250),  (-300, 5800)]},
    "2.5dpi":           {"rot": 175,    "view": [(2900, 9600), (4200, 10900)]},
    "3.5dpi":           {"rot": 135,    "view": [(2600, 9100), (2200, 8700)]},
    "4.5dpi_menstrual": {"rot": 81,     "view": [(900, 9700),  (1350, 10150)]},
}

# columns, left to right
cluster_order = ['eFib1', 'eFib2', 'eFib2i', 'eFib3', 'eFibP',
                 'dec1', 'dec2', 'dec3', 'decAp']

highlight_color = "#fe01fe"
other_color     = "#B0B0B0"   # every other nucleus, including the other subtypes
dot_size, dot_alpha = 25, 1.0
fig_size_in     = 6           # reference size of the single images
panel_in        = 2.0         # size of each grid panel (inches)
gap_in          = 0.1       # space between panels (inches); 0 = touching
show_labels     = True        # row/column labels outside the grid
label_fontsize  = 9
label_pad_in    = 0.35        # room for labels (inches)
background      = "black"
dpi_save        = 300         # PNG resolution and resolution of the rasterized panels in the SVG
dot_size_grid   = dot_size * (panel_in / fig_size_in) ** 2   # same visual density as the single images

basename = "20261113-Spatial_Mouse-eFibDecHighlight-Grid-spatial"

# ---------------- checks ----------------
for col in [key_cluster, key_sample, x_key, y_key]:
    if col not in adata3.obs:
        raise ValueError(f"{col} not found in adata3.obs")
present = set(adata3.obs[key_cluster].astype(str))
col_order = [c for c in cluster_order if c in present]
skipped = [c for c in cluster_order if c not in present]
if skipped:
    print(f"Warning: subtypes not in adata3.obs['{key_cluster}'], skipped: {skipped}")

# ---------------- rotate each section once ----------------
rotated = {}
for timepoint, cfg in panels.items():
    obs = adata3.obs
    mask = obs[key_sample].astype(str) == timepoint
    if not mask.any():
        print(f"Warning: no nuclei for {timepoint}. Skipping.")
        continue
    x   = obs.loc[mask, x_key].astype(float).to_numpy()
    y   = obs.loc[mask, y_key].astype(float).to_numpy()
    lab = obs.loc[mask, key_cluster].astype(str).to_numpy()
    ok = np.isfinite(x) & np.isfinite(y)
    x, y, lab = x[ok], y[ok], lab[ok]

    cx, cy = 0.5 * (x.min() + x.max()), 0.5 * (y.min() + y.max())
    t = np.deg2rad(cfg["rot"])
    xr = (x - cx) * np.cos(t) - (y - cy) * np.sin(t) + cx
    yr = (x - cx) * np.sin(t) + (y - cy) * np.cos(t) + cy
    rotated[timepoint] = (xr, yr, lab)
    print(f"{timepoint}: rotation={cfg['rot']}°, n_points={xr.size}")

row_order = [tp for tp in panels if tp in rotated]

# ---------------- grid figure ----------------
n_rows, n_cols = len(row_order), len(col_order)
left_in = label_pad_in if show_labels else 0.0     # room for row labels
top_in  = label_pad_in if show_labels else 0.0     # room for column titles
W = left_in + n_cols * panel_in + (n_cols - 1) * gap_in
H = top_in  + n_rows * panel_in + (n_rows - 1) * gap_in
fig = plt.figure(figsize=(W, H))
fig.patch.set_facecolor(background)
text_color = 'white' if background == 'black' else 'black'

for r, timepoint in enumerate(row_order):
    xr, yr, lab = rotated[timepoint]
    (vx0, vx1), (vy0, vy1) = panels[timepoint]["view"]
    for c, st in enumerate(col_order):
        left   = (left_in + c * (panel_in + gap_in)) / W
        bottom = (H - top_in - (r + 1) * panel_in - r * gap_in) / H
        ax = fig.add_axes([left, bottom, panel_in / W, panel_in / H])

        hit = lab == st
        if (~hit).any():
            ax.scatter(xr[~hit], yr[~hit], s=dot_size_grid, color=other_color,
                       alpha=dot_alpha, linewidth=0, zorder=1, rasterized=True)
        if hit.any():
            ax.scatter(xr[hit], yr[hit], s=dot_size_grid, color=highlight_color,
                       alpha=dot_alpha, linewidth=0, zorder=2, rasterized=True)
        ax.set_xlim(vx0, vx1)
        ax.set_ylim(vy0, vy1)
        ax.set_aspect("equal", adjustable="box")
        ax.axis("off")

        if show_labels and r == 0:
            fig.text(left + panel_in / W / 2, (H - top_in + 0.05) / H, st,
                     ha='center', va='bottom', fontsize=label_fontsize, color=text_color)
        if show_labels and c == 0:
            fig.text((left_in - 0.05) / W, bottom + panel_in / H / 2, timepoint,
                     ha='right', va='center', rotation=90, fontsize=label_fontsize, color=text_color)

# ---------------- save ----------------
svg_path = os.path.join(output_dir, f"{basename}.svg")
fig.savefig(png_path, dpi=dpi_save, facecolor=background, pad_inches=0)
fig.savefig(svg_path, format='svg', dpi=dpi_save, facecolor=background, pad_inches=0)
plt.show()
print(f"Saved {png_path} (and .svg)")

### eFib/dec for SAMap

In [ ]:
# Settings
key_cluster = "cell_type"
key_sample  = "timepoint"
x_key, y_key = "x_um_dbscan", "y_um_dbscan"

# Per timepoint: rotation (counter-clockwise) and window in the rotated image (order = left to right)
panels = {
    "1.5dpi_late": {"rot": 303, "view": [(150, 6250),  (-300, 5800)]},
    "3.5dpi":      {"rot": 135, "view": [(2600, 9100), (2200, 8700)]},
}

color_map = {
    'eFib': '#CC79A7',   # magenta (same as the cell-type maps)
    'dec':  '#00B4C5',   # cyan
}
draw_order  = ['eFib', 'dec']      # later = on top
shuffle     = False                # True = random draw order, so neither type systematically covers the other
dot_size, dot_alpha = 25, 1.0
fig_size_in   = 6                  # reference size for dot scaling
grid_panel_in = 3                  # size of each panel (inches)
background    = "black"
dpi_save      = 300

outpath = os.path.join(output_dir, "20261113-Spatial_Mouse-eFibDec-overlay-adata4-1.5late_3.5dpi.png")

# ---------------- checks ----------------
for col in [key_cluster, key_sample, x_key, y_key]:
    if col not in adata3.obs:
        raise ValueError(f"{col} not found in adata3.obs")
n_missing = (~adata4.obs_names.isin(adata3.obs_names)).sum()
if n_missing:
    raise ValueError(f"{n_missing} cells in adata4 are not in adata3")

# Labels and coordinates from adata3, restricted to the adata4 nuclei
obs4 = adata3.obs.loc[adata4.obs_names, [key_cluster, key_sample, x_key, y_key]]

# ---------------- rotate each section once ----------------
rotated = {}
for tp, cfg in panels.items():
    # rotation centre from the whole section in adata3, so the windows match the other images
    m3 = (adata3.obs[key_sample].astype(str) == tp).to_numpy()
    if not m3.any():
        print(f"Warning: no nuclei for {tp} in adata3. Skipping.")
        continue
    fx = adata3.obs.loc[m3, x_key].astype(float).to_numpy()
    fy = adata3.obs.loc[m3, y_key].astype(float).to_numpy()
    ok3 = np.isfinite(fx) & np.isfinite(fy)
    cx, cy = 0.5 * (fx[ok3].min() + fx[ok3].max()), 0.5 * (fy[ok3].min() + fy[ok3].max())

    # adata4 nuclei of this timepoint, eFib and dec only
    o = obs4[(obs4[key_sample].astype(str) == tp) & obs4[key_cluster].astype(str).isin(color_map)]
    x   = o[x_key].astype(float).to_numpy()
    y   = o[y_key].astype(float).to_numpy()
    lab = o[key_cluster].astype(str).to_numpy()
    ok = np.isfinite(x) & np.isfinite(y)
    x, y, lab = x[ok], y[ok], lab[ok]

    t = np.deg2rad(cfg["rot"])
    xr = (x - cx) * np.cos(t) - (y - cy) * np.sin(t) + cx
    yr = (x - cx) * np.sin(t) + (y - cy) * np.cos(t) + cy
    rotated[tp] = (xr, yr, lab)

    counts = ', '.join(f"{ct}={int((lab == ct).sum())}" for ct in color_map)
    print(f"{tp}: rotation={cfg['rot']}°, {counts}")

def draw_panel(ax, tp, dot_s):
    xr, yr, lab = rotated[tp]
    if shuffle:
        idx = np.random.default_rng(0).permutation(len(lab))
        ax.scatter(xr[idx], yr[idx], s=dot_s, c=[color_map[l] for l in lab[idx]],
                   alpha=dot_alpha, linewidth=0)
    else:
        for ct in draw_order:
            m = lab == ct
            if m.any():
                ax.scatter(xr[m], yr[m], s=dot_s, color=color_map[ct],
                           alpha=dot_alpha, linewidth=0)
    (vx0, vx1), (vy0, vy1) = panels[tp]["view"]
    ax.set_xlim(vx0, vx1)
    ax.set_ylim(vy0, vy1)
    ax.set_aspect("equal", adjustable="box")
    ax.set_facecolor(background)
    ax.axis("off")

order = [tp for tp in panels if tp in rotated]

# ---------------- two panels side by side, touching, no labels ----------------
n = len(order)
dot_size_grid = dot_size * (grid_panel_in / fig_size_in) ** 2
fig = plt.figure(figsize=(grid_panel_in * n, grid_panel_in))
for i, tp in enumerate(order):
    ax = fig.add_axes([i / n, 0, 1 / n, 1])
    draw_panel(ax, tp, dot_size_grid)

fig.patch.set_facecolor(background)
fig.savefig(outpath, dpi=dpi_save, facecolor=background, pad_inches=0)
plt.close()
print(f"Saved {outpath}")

### Spatial pseudotime

In [ ]:
from scipy.spatial import cKDTree

# Settings
key_value  = "pseudotime"
key_sample = "timepoint"
x_key, y_key = "x_um_dbscan", "y_um_dbscan"   # same coordinates as the other spatial images

# Same rotation (counter-clockwise) and window in the rotated image as the cell-type grids
panels = {
    "1.5dpi_late": {"rot": 303, "view": [(150, 6250),  (-300, 5800)]},
    "3.5dpi":      {"rot": 135, "view": [(2600, 9100), (2200, 8700)]},
}

cmap        = plt.get_cmap("viridis")
other_color = "#231f20"   # cells without a pseudotime value, drawn at the bottom
dot_size, dot_alpha = 25, 1.0
fig_size_in, dpi_save = 6, 300
background  = "black"

# Pseudotime discontinuity filter
filter_groups = {}           # panels to filter; set() = none
filter_k      = 5            # spatial neighbours
filter_q      = 0.90         # keep cells whose residual is <= this quantile

# ---------------- checks ----------------
for col in [key_sample, x_key, y_key]:
    if col not in adata3.obs:
        raise ValueError(f"{col} not found in adata3.obs")
if key_value not in adata4.obs:
    raise ValueError(f"{key_value} not found in adata4.obs")
n_missing = (~adata4.obs_names.isin(adata3.obs_names)).sum()
if n_missing:
    raise ValueError(f"{n_missing} cells in adata4 are not in adata3")

# Timepoint and coordinates from adata3 (as in the other images), pseudotime from adata4
obs4 = adata3.obs.loc[adata4.obs_names, [key_sample, x_key, y_key]].copy()
obs4[key_value] = adata4.obs[key_value].astype(float).values

# Shared color scale across all adata4 cells
vals_all = obs4[key_value].to_numpy(float)
vmin, vmax = float(np.nanmin(vals_all)), float(np.nanmax(vals_all))
print(f"{key_value}: range {vmin:.3f}–{vmax:.3f}, "
      f"{np.isfinite(vals_all).sum():,}/{vals_all.size:,} cells have a value")

def pt_discontinuity_filter(xy, pt, k=5, q=0.90):
    """True = keep: |pt - median pt of k nearest spatial neighbours| <= its q-quantile."""
    xy, pt = np.asarray(xy, float), np.asarray(pt, float)
    ok = np.isfinite(pt) & np.isfinite(xy).all(axis=1)
    xy_ok, pt_ok = xy[ok], pt[ok]
    _, j = cKDTree(xy_ok).query(xy_ok, k=k + 1)    # column 0 is the cell itself
    resid = np.abs(pt_ok - np.median(pt_ok[j[:, 1:]], axis=1))
    keep = np.zeros(len(pt), bool)
    keep[ok] = resid <= np.quantile(resid, q)
    return keep

# ---------------- one image per panel ----------------
for timepoint, cfg in panels.items():
    # Rotation centre from the whole section in adata3, so the windows match the other images
    m3 = adata3.obs[key_sample].astype(str) == timepoint
    x3 = adata3.obs.loc[m3, x_key].astype(float).to_numpy()
    y3 = adata3.obs.loc[m3, y_key].astype(float).to_numpy()
    ok3 = np.isfinite(x3) & np.isfinite(y3)
    if not ok3.any():
        print(f"Warning: no nuclei for {timepoint} in adata3. Skipping.")
        continue
    cx = 0.5 * (x3[ok3].min() + x3[ok3].max())
    cy = 0.5 * (y3[ok3].min() + y3[ok3].max())

    # adata4 cells of this timepoint
    m4 = obs4[key_sample].astype(str) == timepoint
    x   = obs4.loc[m4, x_key].to_numpy(float)
    y   = obs4.loc[m4, y_key].to_numpy(float)
    val = obs4.loc[m4, key_value].to_numpy(float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y, val = x[ok], y[ok], val[ok]
    if x.size == 0:
        print(f"Warning: no adata4 cells for {timepoint}. Skipping.")
        continue

    # Discontinuity filter on the whole section (before cropping); cells without pseudotime are kept
    if timepoint in filter_groups:
        has_pt = np.isfinite(val)
        if has_pt.sum() > filter_k:
            keep_pt = pt_discontinuity_filter(np.c_[x, y], val, k=filter_k, q=filter_q)
            drop = has_pt & ~keep_pt
            print(f"{timepoint}: discontinuity filter removed {drop.sum():,} / {has_pt.sum():,} pseudotime cells")
            x, y, val = x[~drop], y[~drop], val[~drop]
        else:
            print(f"{timepoint}: too few pseudotime cells for the filter; skipped")

    # Rotate, then use the same window as the other images
    t = np.deg2rad(cfg["rot"])
    xr = (x - cx) * np.cos(t) - (y - cy) * np.sin(t) + cx
    yr = (x - cx) * np.sin(t) + (y - cy) * np.cos(t) + cy
    (vx0, vx1), (vy0, vy1) = cfg["view"]

    has_val = np.isfinite(val)
    n_in = ((xr >= vx0) & (xr <= vx1) & (yr >= vy0) & (yr <= vy1)).sum()
    print(f"{timepoint}: rotation={cfg['rot']}°, {n_in} of {xr.size} cells in view, "
          f"{has_val.sum()} with pseudotime")

    fig = plt.figure(figsize=(fig_size_in, fig_size_in))
    ax = fig.add_axes([0, 0, 1, 1])

    # Cells without pseudotime first (bottom layer)
    if (~has_val).any():
        ax.scatter(xr[~has_val], yr[~has_val], s=dot_size, color=other_color,
                   alpha=dot_alpha, linewidth=0, zorder=1)

    # Cells with pseudotime, sorted so later pseudotime is drawn on top
    if has_val.any():
        o = np.argsort(val[has_val])
        ax.scatter(xr[has_val][o], yr[has_val][o], c=val[has_val][o],
                   cmap=cmap, vmin=vmin, vmax=vmax,
                   s=dot_size, alpha=dot_alpha, linewidth=0, zorder=2)

    ax.set_xlim(vx0, vx1)
    ax.set_ylim(vy0, vy1)
    ax.set_aspect("equal", adjustable="box")
    ax.set_facecolor(background)
    ax.axis("off")
    fig.patch.set_facecolor(background)

    outpath = os.path.join(output_dir, f"20261113-Spatial_Mouse-{timepoint}-pseudotime.png")
    fig.savefig(outpath, dpi=dpi_save, facecolor=background, pad_inches=0)
    plt.close(fig)
    print(f"Saved {outpath}")

# ---------------- colorbar as a separate image ----------------
fig, ax = plt.subplots(figsize=(0.3, 2.0))
cb = fig.colorbar(plt.cm.ScalarMappable(norm=plt.Normalize(vmin, vmax), cmap=cmap), cax=ax)
cb.outline.set_visible(False)
cb.set_ticks([])
fig.patch.set_alpha(0)
ax.patch.set_alpha(0)

outpath = os.path.join(output_dir, "20261113-Spatial_Mouse-pseudotime-colorbar.png")
fig.savefig(outpath, dpi=dpi_save, bbox_inches="tight", pad_inches=0, transparent=True)
plt.close(fig)
print(f"Saved {outpath}")

### Cxcl12 Expression Spatial

In [ ]:
plt.rcParams['svg.fonttype'] = 'none'   # labels stay editable text in the SVG

# Settings
gene        = "Cxcl12"
key_sample  = "timepoint"
x_key, y_key = "x_um_dbscan", "y_um_dbscan"   # same coordinates as the other spatial images
layer       = "log1p_counts"

# Per timepoint: rotation (counter-clockwise) and window in the rotated image (order = grid order)
panels = {
    "uninduced":        {"rot": 322.75, "view": [(2050, 4650), (-300, 2300)]},
    "1.5dpi_early":     {"rot": 28,     "view": [(750, 2800),  (925, 2975)]},
    "1.5dpi_late":      {"rot": 303,    "view": [(150, 6250),  (-300, 5800)]},
    "2.5dpi":           {"rot": 175,    "view": [(2900, 9600), (4200, 10900)]},
    "3.5dpi":           {"rot": 135,    "view": [(2600, 9100), (2200, 8700)]},
    "4.5dpi_menstrual": {"rot": 81,     "view": [(900, 9700),  (1350, 10150)]},
}

cmap        = plt.get_cmap("viridis")
vmax_pct    = 99.5           # per panel: top of the scale = this percentile of expressing cells in view
dot_size, dot_alpha = 25, 1.0
fig_size_in   = 6            # reference size for dot scaling
panel_in      = 3.0          # size of each spatial panel (inches)
gap_in        = 0.10         # horizontal space between panels (inches)
cbar_w_frac   = 0.6          # colorbar width as a fraction of the panel width
cbar_h_in     = 0.08         # colorbar thickness (inches)
cbar_gap_in   = 0.08         # space between panel and colorbar (inches)
show_labels   = True         # timepoint names above the panels
fontsize      = 8
background    = "black"
dpi_save      = 300          # resolution of the rasterized spatial maps inside the SVG
text_color    = 'white' if background == 'black' else 'black'

# ---------------- checks ----------------
for col in [key_sample, x_key, y_key]:
    if col not in adata3.obs:
        raise ValueError(f"{col} not found in adata3.obs")
if gene not in adata3.var_names:
    raise ValueError(f"{gene} not found in adata3.var_names")

# Expression (log-normalized) for all cells
X = adata3[:, gene].layers[layer] if layer else adata3[:, gene].X
expr_all = (X.toarray() if sparse.issparse(X) else np.asarray(X)).ravel().astype(float)

# ---------------- rotate each section once, per-panel color scale ----------------
rotated, scales = {}, {}
for timepoint, cfg in panels.items():
    obs = adata3.obs
    mask = (obs[key_sample].astype(str) == timepoint).to_numpy()
    if not mask.any():
        print(f"Warning: no nuclei for {timepoint}. Skipping.")
        continue
    x   = obs.loc[mask, x_key].astype(float).to_numpy()
    y   = obs.loc[mask, y_key].astype(float).to_numpy()
    val = expr_all[mask]
    ok = np.isfinite(x) & np.isfinite(y)
    x, y, val = x[ok], y[ok], val[ok]

    # rotate around the center of the whole section (same as the other images)
    cx, cy = 0.5 * (x.min() + x.max()), 0.5 * (y.min() + y.max())
    t = np.deg2rad(cfg["rot"])
    xr = (x - cx) * np.cos(t) - (y - cy) * np.sin(t) + cx
    yr = (x - cx) * np.sin(t) + (y - cy) * np.cos(t) + cy
    rotated[timepoint] = (xr, yr, val)

    (vx0, vx1), (vy0, vy1) = cfg["view"]
    in_view = (xr >= vx0) & (xr <= vx1) & (yr >= vy0) & (yr <= vy1)
    pos = val[in_view & (val > 0)]
    vmax = float(np.percentile(pos, vmax_pct)) if pos.size else 1.0
    scales[timepoint] = (0.0, vmax)
    print(f"{timepoint}: {in_view.sum()} nuclei in view, {gene}+ = {pos.size:,} "
          f"({pos.size / max(in_view.sum(), 1):.1%}), scale 0–{vmax:.2f} log1p")

order = [tp for tp in panels if tp in rotated]

# ---------------- figure layout (inches) ----------------
n = len(order)
title_in = 0.30 if show_labels else 0.0
cbar_lab_in = 0.20                           # room for the min/max numbers under the colorbar
W = n * panel_in + (n - 1) * gap_in
H = title_in + panel_in + cbar_gap_in + cbar_h_in + cbar_lab_in
fig = plt.figure(figsize=(W, H))
fig.patch.set_facecolor(background)

dot_size_grid = dot_size * (panel_in / fig_size_in) ** 2

for i, timepoint in enumerate(order):
    xr, yr, val = rotated[timepoint]
    (vx0, vx1), (vy0, vy1) = panels[timepoint]["view"]
    vmin, vmax = scales[timepoint]
    x0_in = i * (panel_in + gap_in)

    # Spatial map (rasterized -> embedded as an image in the SVG)
    panel_bottom_in = cbar_lab_in + cbar_h_in + cbar_gap_in
    ax = fig.add_axes([x0_in / W, panel_bottom_in / H, panel_in / W, panel_in / H])
    o = np.argsort(val)                      # higher expression drawn on top
    ax.scatter(xr[o], yr[o], c=val[o], cmap=cmap, vmin=vmin, vmax=vmax,
               s=dot_size_grid, alpha=dot_alpha, linewidth=0, rasterized=True)
    ax.set_xlim(vx0, vx1)
    ax.set_ylim(vy0, vy1)
    ax.set_aspect("equal", adjustable="box")
    ax.set_facecolor(background)
    ax.axis("off")

    if show_labels:
        fig.text((x0_in + panel_in / 2) / W, (panel_bottom_in + panel_in + 0.06) / H, timepoint,
                 ha='center', va='bottom', fontsize=fontsize, color=text_color)

    # Colorbar under the panel, with min and max values
    cb_w_in = panel_in * cbar_w_frac
    cax = fig.add_axes([(x0_in + (panel_in - cb_w_in) / 2) / W, cbar_lab_in / H,
                        cb_w_in / W, cbar_h_in / H])
    cb = fig.colorbar(plt.cm.ScalarMappable(norm=plt.Normalize(vmin, vmax), cmap=cmap),
                      cax=cax, orientation='horizontal')
    cb.outline.set_visible(False)
    cb.set_ticks([vmin, vmax])
    cb.set_ticklabels([f"{vmin:.0f}", f"{vmax:.1f}"])
    cax.tick_params(labelsize=fontsize - 1, colors=text_color, length=0, pad=2)

# ---------------- save ----------------
outpath = os.path.join(output_dir, f"20261113-Spatial_Mouse-{gene}-spatial-grid-perpanel.svg")
fig.savefig(outpath, format='svg', dpi=dpi_save, facecolor=background, pad_inches=0)
plt.close()
print(f"Saved {outpath}")

### Notum expression

In [ ]:
plt.rcParams['svg.fonttype'] = 'none'   # labels stay editable text in the SVG

# Settings
gene        = "Notum"
key_sample  = "timepoint"
x_key, y_key = "x_um_dbscan", "y_um_dbscan"   # same coordinates as the other spatial images
layer       = "log1p_counts"

# Per timepoint: rotation (counter-clockwise) and window in the rotated image (order = grid order)
panels = {
    "uninduced":        {"rot": 322.75, "view": [(2050, 4650), (-300, 2300)]},
    "1.5dpi_early":     {"rot": 28,     "view": [(750, 2800),  (925, 2975)]},
    "1.5dpi_late":      {"rot": 303,    "view": [(150, 6250),  (-300, 5800)]},
    "2.5dpi":           {"rot": 175,    "view": [(2900, 9600), (4200, 10900)]},
    "3.5dpi":           {"rot": 135,    "view": [(2600, 9100), (2200, 8700)]},
    "4.5dpi_menstrual": {"rot": 81,     "view": [(900, 9700),  (1350, 10150)]},
}

cmap        = plt.get_cmap("viridis")
scale       = 'shared'       # 'shared' = one scale for all panels; 'per_panel' = each panel its own
vmax_pct    = 99.5           # top of the scale = this percentile of expressing nuclei in view
dot_size, dot_alpha = 25, 1.0
fig_size_in   = 6            # reference size for dot scaling
panel_in      = 3.0          # size of each spatial panel (inches)
gap_in        = 0.10         # horizontal space between panels (inches)
cbar_w_frac   = 0.6          # colorbar width as a fraction of one panel's width
cbar_h_in     = 0.08         # colorbar thickness (inches)
cbar_gap_in   = 0.08         # space between panels and colorbar (inches)
show_labels   = True         # timepoint names above the panels
fontsize      = 8
background    = "black"
dpi_save      = 300          # resolution of the rasterized spatial maps inside the SVG
text_color    = 'white' if background == 'black' else 'black'

# ---------------- checks ----------------
for col in [key_sample, x_key, y_key]:
    if col not in adata3.obs:
        raise ValueError(f"{col} not found in adata3.obs")
if gene not in adata3.var_names:
    raise ValueError(f"{gene} not found in adata3.var_names")
if scale not in ('shared', 'per_panel'):
    raise ValueError("scale must be 'shared' or 'per_panel'")

# Expression (log-normalized) for all cells
X = adata3[:, gene].layers[layer] if layer else adata3[:, gene].X
expr_all = (X.toarray() if sps.issparse(X) else np.asarray(X)).ravel().astype(float)

# ---------------- rotate each section once ----------------
rotated, pos_vals = {}, {}
for timepoint, cfg in panels.items():
    obs = adata3.obs
    mask = (obs[key_sample].astype(str) == timepoint).to_numpy()
    if not mask.any():
        print(f"Warning: no nuclei for {timepoint}. Skipping.")
        continue
    x   = obs.loc[mask, x_key].astype(float).to_numpy()
    y   = obs.loc[mask, y_key].astype(float).to_numpy()
    val = expr_all[mask]
    ok = np.isfinite(x) & np.isfinite(y)
    x, y, val = x[ok], y[ok], val[ok]

    # rotate around the center of the whole section (same as the other images)
    cx, cy = 0.5 * (x.min() + x.max()), 0.5 * (y.min() + y.max())
    t = np.deg2rad(cfg["rot"])
    xr = (x - cx) * np.cos(t) - (y - cy) * np.sin(t) + cx
    yr = (x - cx) * np.sin(t) + (y - cy) * np.cos(t) + cy
    rotated[timepoint] = (xr, yr, val)

    (vx0, vx1), (vy0, vy1) = cfg["view"]
    in_view = (xr >= vx0) & (xr <= vx1) & (yr >= vy0) & (yr <= vy1)
    pos_vals[timepoint] = val[in_view & (val > 0)]
    n_view = int(in_view.sum())
    print(f"{timepoint}: {n_view} nuclei in view, {gene}+ = {pos_vals[timepoint].size:,} "
          f"({pos_vals[timepoint].size / max(n_view, 1):.1%})")

order = [tp for tp in panels if tp in rotated]

# ---------------- color scale ----------------
def top_of_scale(v):
    return float(np.percentile(v, vmax_pct)) if v.size else 1.0

if scale == 'shared':
    vmax_shared = top_of_scale(np.concatenate([pos_vals[tp] for tp in order]))
    scales = {tp: (0.0, vmax_shared) for tp in order}
    print(f"\nShared scale: 0–{vmax_shared:.2f} log1p")
else:
    scales = {tp: (0.0, top_of_scale(pos_vals[tp])) for tp in order}
    print("\nPer-panel scales: " + ', '.join(f"{tp} 0–{scales[tp][1]:.2f}" for tp in order))

# ---------------- figure layout (inches) ----------------
n = len(order)
title_in = 0.30 if show_labels else 0.0
cbar_lab_in = 0.20                           # room for the min/max numbers under the colorbar
W = n * panel_in + (n - 1) * gap_in
H = title_in + panel_in + cbar_gap_in + cbar_h_in + cbar_lab_in
panel_bottom_in = cbar_lab_in + cbar_h_in + cbar_gap_in
fig = plt.figure(figsize=(W, H))
fig.patch.set_facecolor(background)

dot_size_grid = dot_size * (panel_in / fig_size_in) ** 2

def add_colorbar(x_center_in, vmin, vmax):
    cb_w_in = panel_in * cbar_w_frac
    cax = fig.add_axes([(x_center_in - cb_w_in / 2) / W, cbar_lab_in / H, cb_w_in / W, cbar_h_in / H])
    cb = fig.colorbar(plt.cm.ScalarMappable(norm=plt.Normalize(vmin, vmax), cmap=cmap),
                      cax=cax, orientation='horizontal')
    cb.outline.set_visible(False)
    cb.set_ticks([vmin, vmax])
    cb.set_ticklabels([f"{vmin:.0f}", f"{vmax:.1f}"])
    cax.tick_params(labelsize=fontsize - 1, colors=text_color, length=0, pad=2)

for i, timepoint in enumerate(order):
    xr, yr, val = rotated[timepoint]
    (vx0, vx1), (vy0, vy1) = panels[timepoint]["view"]
    vmin, vmax = scales[timepoint]
    x0_in = i * (panel_in + gap_in)

    # Spatial map (rasterized -> embedded as an image in the SVG)
    ax = fig.add_axes([x0_in / W, panel_bottom_in / H, panel_in / W, panel_in / H])
    o = np.argsort(val)                      # higher expression drawn on top
    ax.scatter(xr[o], yr[o], c=val[o], cmap=cmap, vmin=vmin, vmax=vmax,
               s=dot_size_grid, alpha=dot_alpha, linewidth=0, rasterized=True)
    ax.set_xlim(vx0, vx1)
    ax.set_ylim(vy0, vy1)
    ax.set_aspect("equal", adjustable="box")
    ax.set_facecolor(background)
    ax.axis("off")

    if show_labels:
        fig.text((x0_in + panel_in / 2) / W, (panel_bottom_in + panel_in + 0.06) / H, timepoint,
                 ha='center', va='bottom', fontsize=fontsize, color=text_color)

    if scale == 'per_panel':
        add_colorbar(x0_in + panel_in / 2, vmin, vmax)

# One colorbar under the middle of the row for the shared scale
if scale == 'shared':
    add_colorbar(W / 2, *scales[order[0]])

# ---------------- save ----------------
outpath = os.path.join(output_dir, f"20261113-Spatial_Mouse-{gene}-spatial-grid-{scale}.svg")
fig.savefig(outpath, format='svg', dpi=dpi_save, facecolor=background, pad_inches=0)
plt.close()
print(f"Saved {outpath}")

### Transferred Stro-7 Module Scores

In [ ]:
# Settings
score_col    = "Ulrich2024_sg_Stro7_top50_transferred"   # column in adata4.obs
key_sample   = "timepoint"
x_key, y_key = "x_um_dbscan", "y_um_dbscan"   # same coordinates as the other spatial images

# Per timepoint: rotation (counter-clockwise) and window in the rotated image (order = grid order)
panels = {
    "uninduced":        {"rot": 322.75, "view": [(2050, 4650), (-300, 2300)]},
    "1.5dpi_early":     {"rot": 28,     "view": [(750, 2800),  (925, 2975)]},
    "1.5dpi_late":      {"rot": 303,    "view": [(150, 6250),  (-300, 5800)]},
    "2.5dpi":           {"rot": 175,    "view": [(2900, 9600), (4200, 10900)]},
    "3.5dpi":           {"rot": 135,    "view": [(2600, 9100), (2200, 8700)]},
    "4.5dpi_menstrual": {"rot": 81,     "view": [(900, 9700),  (1350, 10150)]},
}

cmap          = plt.get_cmap("viridis")
scale         = 'shared'             # 'shared' = one scale for all timepoints; 'per_panel' = each its own
pct           = (1, 99)              # color range clipped to these percentiles (nuclei in view)
show_background = True               # draw nuclei without a score (not in adata4, or NaN) underneath
other_color   = "#231f20"
dot_size, dot_alpha = 25, 1.0
fig_size_in   = 6                    # reference size for dot scaling
grid_panel_in = 3                    # size of each panel in the combined image (inches)
grid_layout   = "row"                # "row" = side by side; "column" = stacked
background    = "black"
dpi_save      = 300

tag     = score_col.replace('_transferred', '')
outpath = os.path.join(output_dir, f"20261113-Spatial_Mouse-{tag}-spatial-grid-{grid_layout}.png")
cb_path = os.path.join(output_dir, f"20261113-Spatial_Mouse-{tag}-colorbar.png")

# ---------------- checks ----------------
for col in [key_sample, x_key, y_key]:
    if col not in adata3.obs:
        raise ValueError(f"{col} not found in adata3.obs")
if score_col not in adata4.obs:
    raise ValueError(f"{score_col} not found in adata4.obs")
n_missing = (~adata4.obs_names.isin(adata3.obs_names)).sum()
if n_missing:
    raise ValueError(f"{n_missing} cells in adata4 are not in adata3")

# Score for every adata3 nucleus (NaN if not in adata4 or not scored)
score_all = pd.Series(np.nan, index=adata3.obs_names)
score_all.loc[adata4.obs_names] = adata4.obs[score_col].astype(float).values

# ---------------- rotate each section once ----------------
rotated = {}
for tp, cfg in panels.items():
    obs = adata3.obs
    mask = (obs[key_sample].astype(str) == tp).to_numpy()
    if not mask.any():
        print(f"Warning: no nuclei for {tp}. Skipping.")
        continue
    x = obs.loc[mask, x_key].astype(float).to_numpy()
    y = obs.loc[mask, y_key].astype(float).to_numpy()
    s = score_all.to_numpy()[mask]
    ok = np.isfinite(x) & np.isfinite(y)
    x, y, s = x[ok], y[ok], s[ok]

    # rotate around the center of the whole section (same as the other images)
    cx, cy = 0.5 * (x.min() + x.max()), 0.5 * (y.min() + y.max())
    t = np.deg2rad(cfg["rot"])
    xr = (x - cx) * np.cos(t) - (y - cy) * np.sin(t) + cx
    yr = (x - cx) * np.sin(t) + (y - cy) * np.cos(t) + cy

    (vx0, vx1), (vy0, vy1) = cfg["view"]
    in_view = (xr >= vx0) & (xr <= vx1) & (yr >= vy0) & (yr <= vy1)
    rotated[tp] = dict(x=xr, y=yr, s=s, in_view=in_view)

order = [tp for tp in panels if tp in rotated]

# ---------------- color ranges ----------------
if scale == 'shared':
    vals = np.concatenate([rotated[tp]["s"][rotated[tp]["in_view"]] for tp in order])
    lo, hi = np.nanpercentile(vals, pct)
    ranges = {tp: (lo, hi) for tp in order}
else:
    ranges = {tp: tuple(np.nanpercentile(rotated[tp]["s"][rotated[tp]["in_view"]], pct)) for tp in order}

summary = pd.DataFrame({
    'nuclei in view':   [int(rotated[tp]["in_view"].sum()) for tp in order],
    'scored in view':   [int(np.isfinite(rotated[tp]["s"][rotated[tp]["in_view"]]).sum()) for tp in order],
    'median score':     [np.nanmedian(rotated[tp]["s"][rotated[tp]["in_view"]]) for tp in order],
    'color min':        [ranges[tp][0] for tp in order],
    'color max':        [ranges[tp][1] for tp in order],
}, index=pd.Index(order, name=tag))
print(summary.round(3).to_string(), '\n')

def draw_panel(ax, tp, dot_s):
    p = rotated[tp]
    has = np.isfinite(p["s"])
    if show_background and (~has).any():
        ax.scatter(p["x"][~has], p["y"][~has], s=dot_s, color=other_color,
                   alpha=dot_alpha, linewidth=0, zorder=1)
    o = np.argsort(p["s"][has])              # highest scores drawn on top
    vmin, vmax = ranges[tp]
    ax.scatter(p["x"][has][o], p["y"][has][o], c=p["s"][has][o], cmap=cmap, vmin=vmin, vmax=vmax,
               s=dot_s, alpha=dot_alpha, linewidth=0, zorder=2)
    (vx0, vx1), (vy0, vy1) = panels[tp]["view"]
    ax.set_xlim(vx0, vx1)
    ax.set_ylim(vy0, vy1)
    ax.set_aspect("equal", adjustable="box")
    ax.set_facecolor(background)
    ax.axis("off")

# ---------------- combined image, panels touching, no labels ----------------
n = len(order)
dot_size_grid = dot_size * (grid_panel_in / fig_size_in) ** 2
if grid_layout == "row":
    fig = plt.figure(figsize=(grid_panel_in * n, grid_panel_in))
    positions = [[i / n, 0, 1 / n, 1] for i in range(n)]
else:
    fig = plt.figure(figsize=(grid_panel_in, grid_panel_in * n))
    positions = [[0, 1 - (i + 1) / n, 1, 1 / n] for i in range(n)]

for tp, pos in zip(order, positions):
    ax = fig.add_axes(pos)
    draw_panel(ax, tp, dot_size_grid)

fig.patch.set_facecolor(background)
fig.savefig(outpath, dpi=dpi_save, facecolor=background, pad_inches=0)
plt.show()
print(f"Saved {outpath}")

# ---------------- colorbar as a separate image (shared scale only) ----------------
if scale == 'shared':
    fig, ax = plt.subplots(figsize=(0.3, 2.0))
    cb = fig.colorbar(plt.cm.ScalarMappable(norm=plt.Normalize(lo, hi), cmap=cmap), cax=ax)
    cb.outline.set_visible(False)
    cb.set_ticks([])
    fig.patch.set_alpha(0)
    ax.patch.set_alpha(0)
    fig.savefig(cb_path, dpi=dpi_save, bbox_inches="tight", pad_inches=0, transparent=True)
    plt.close()
    print(f"Saved {cb_path}  (scale {lo:.3f} to {hi:.3f})")

## Human

### All cell types

In [ ]:
# Settings
key_cluster = "cell_type"
key_sample  = "timepoint"
x_key, y_key = "x_um_dbscan", "y_um_dbscan"
flip_h = True              # mirror the images left-right (applied after rotation)

# Per timepoint: crop in raw coordinates (None = whole section), then rotate (counter-clockwise)
panels = {
    "secretory": {"crop": [(400, 5500), (750, 5250)], "rot": -90},
    "menstrual": {"crop": None,                         "rot": 90},
}

color_map = {
    'dec':  '#00B4C5',  # cyan
    'eFib': '#CC79A7',  # reddish purple
    'vsc':  '#5B7FE6',  # medium blue
    'epi':  '#D4B900',  # golden yellow
    'imm':  '#D55E00',  # vermillion
    'smc':  '#4DAF4A',  # green
}
draw_order  = ['smc', 'vsc', 'imm', 'dec', 'epi', 'eFib']   # later = on top
other_color = "#B0B0B0"
dot_size, dot_alpha = 8, 1.0
fig_size_in = 6            # reference: longer side of a single image
grid_h_in   = 3            # height of each panel in the combined image (width follows the tissue)
pad_frac    = 0.01         # margin around the tissue, as a fraction of its extent
background  = "black"
dpi_save    = 300

# ---------------- checks ----------------
for col in [key_cluster, key_sample, x_key, y_key]:
    if col not in adata1.obs:
        raise ValueError(f"{col} not found in adata1.obs")
no_color = sorted(set(adata1.obs[key_cluster].astype(str)) - set(color_map))
if no_color:
    print(f"Warning: no color for {no_color}; drawn as {other_color}")

# ---------------- crop and rotate each section once ----------------
rotated = {}
for timepoint, cfg in panels.items():
    obs = adata1.obs
    mask = (obs[key_sample].astype(str) == timepoint).to_numpy()
    if not mask.any():
        print(f"Warning: no nuclei for {timepoint}. Skipping.")
        continue
    x   = obs.loc[mask, x_key].astype(float).to_numpy()
    y   = obs.loc[mask, y_key].astype(float).to_numpy()
    lab = obs.loc[mask, key_cluster].astype(str).to_numpy()
    ok = np.isfinite(x) & np.isfinite(y)
    x, y, lab = x[ok], y[ok], lab[ok]

    if cfg["crop"] is not None:
        (xmin, xmax), (ymin, ymax) = cfg["crop"]
        keep = (x >= xmin) & (x <= xmax) & (y >= ymin) & (y <= ymax)
        x, y, lab = x[keep], y[keep], lab[keep]
        cx, cy = 0.5 * (xmin + xmax), 0.5 * (ymin + ymax)
    else:
        cx, cy = 0.5 * (x.min() + x.max()), 0.5 * (y.min() + y.max())

    t = np.deg2rad(cfg["rot"])
    xr = (x - cx) * np.cos(t) - (y - cy) * np.sin(t) + cx
    yr = (x - cx) * np.sin(t) + (y - cy) * np.cos(t) + cy
    if flip_h:
        xr = -xr

    # tight limits around the tissue plus a small margin
    px = pad_frac * (xr.max() - xr.min())
    py = pad_frac * (yr.max() - yr.min())
    xlim = (xr.min() - px, xr.max() + px)
    ylim = (yr.min() - py, yr.max() + py)
    rotated[timepoint] = (xr, yr, lab, xlim, ylim)
    print(f"{timepoint}: rotation={cfg['rot']}°, n_points={xr.size}")

def draw_panel(ax, timepoint, dot_s):
    xr, yr, lab, xlim, ylim = rotated[timepoint]
    uncolored = ~np.isin(lab, list(color_map))
    if uncolored.any():
        ax.scatter(xr[uncolored], yr[uncolored], s=dot_s, color=other_color,
                   alpha=dot_alpha, linewidth=0, zorder=1)
    for ct in draw_order:
        m = lab == ct
        if m.any():
            ax.scatter(xr[m], yr[m], s=dot_s, color=color_map[ct],
                       alpha=dot_alpha, linewidth=0, zorder=2)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal", adjustable="box")
    ax.set_facecolor(background)
    ax.axis("off")

order = [tp for tp in panels if tp in rotated]

# ---------------- combined image: equal heights, widths follow each tissue ----------------
widths = []
for tp in order:
    _, _, _, xlim, ylim = rotated[tp]
    widths.append(grid_h_in * (xlim[1] - xlim[0]) / (ylim[1] - ylim[0]))
W, H = sum(widths), grid_h_in
dot_size_grid = dot_size * (grid_h_in / fig_size_in) ** 2

fig = plt.figure(figsize=(W, H))
left = 0.0
for tp, w in zip(order, widths):
    ax = fig.add_axes([left / W, 0, w / W, 1])
    draw_panel(ax, tp, dot_size_grid)
    left += w

fig.patch.set_facecolor(background)
outpath = os.path.join(output_dir, "20261113-Spatial_Human-celltype-spatial-grid-row.png")
fig.savefig(outpath, dpi=dpi_save, facecolor=background, pad_inches=0)
plt.show()
print(f"Saved {outpath}")

### All cell types, grid

In [ ]:
### Human spatial grid: rows = cell types, columns = timepoints (crop, rotate, flip) ###

plt.rcParams['svg.fonttype'] = 'none'   # labels stay editable text in the SVG

# Settings
key_cluster = "cell_type"
key_sample  = "timepoint"
x_key, y_key = "x_um_dbscan", "y_um_dbscan"
flip_h      = True             # mirror each section left-right (applied after rotation)

# Per timepoint: crop in raw coordinates (None = whole section), then rotate (counter-clockwise)
# (order = columns, left to right)
panels = {
    "secretory": {"crop": [(400, 5500), (750, 5250)], "rot": -90},
    "menstrual": {"crop": None,                         "rot": 90},
}

# Rows, top to bottom: cell_type label -> row label
label_names = {
    'epi':  'epithelial cell',
    'eFib': 'endometrial fibroblast',
    'dec':  'decidual cell',
    'imm':  'immune cell',
    'vsc':  'endothelial cell',
    'smc':  'smooth muscle cell',
}

highlight_color = "#fe01fe"
other_color     = "#B0B0B0"
dot_size, dot_alpha = 12, 1.0
fig_size_in    = 6             # reference: size of a single image
panel_h_in     = 2.0           # height of each panel (inches); width follows the tissue
gap_in         = 0.025         # space between panels (inches); 0 = touching
pad_frac       = 0.01          # margin around the tissue, as a fraction of its extent
show_labels    = True          # row/column labels outside the grid
label_fontsize = 9
label_pad_in   = 0.35          # room for labels (inches)
background     = "black"
dpi_save       = 300           # resolution of the rasterized panels inside the SVG

# ---------------- checks ----------------
for col in [key_cluster, key_sample, x_key, y_key]:
    if col not in adata1.obs:
        raise ValueError(f"{col} not found in adata1.obs")
present = set(adata1.obs[key_cluster].astype(str))
row_order = [c for c in label_names if c in present]
skipped = [c for c in label_names if c not in present]
if skipped:
    print(f"Warning: cell types not in adata1.obs['{key_cluster}'], skipped: {skipped}")

# ---------------- crop, rotate and flip each section once ----------------
rotated = {}
for timepoint, cfg in panels.items():
    obs = adata1.obs
    mask = (obs[key_sample].astype(str) == timepoint).to_numpy()
    if not mask.any():
        print(f"Warning: no nuclei for {timepoint}. Skipping.")
        continue
    x   = obs.loc[mask, x_key].astype(float).to_numpy()
    y   = obs.loc[mask, y_key].astype(float).to_numpy()
    lab = obs.loc[mask, key_cluster].astype(str).to_numpy()
    ok = np.isfinite(x) & np.isfinite(y)
    x, y, lab = x[ok], y[ok], lab[ok]

    if cfg["crop"] is not None:
        (xmin, xmax), (ymin, ymax) = cfg["crop"]
        keep = (x >= xmin) & (x <= xmax) & (y >= ymin) & (y <= ymax)
        x, y, lab = x[keep], y[keep], lab[keep]
        cx, cy = 0.5 * (xmin + xmax), 0.5 * (ymin + ymax)
    else:
        cx, cy = 0.5 * (x.min() + x.max()), 0.5 * (y.min() + y.max())

    t = np.deg2rad(cfg["rot"])
    xr = (x - cx) * np.cos(t) - (y - cy) * np.sin(t) + cx
    yr = (x - cx) * np.sin(t) + (y - cy) * np.cos(t) + cy
    if flip_h:
        xr = -xr

    # tight limits around the tissue plus a small margin
    px = pad_frac * (xr.max() - xr.min())
    py = pad_frac * (yr.max() - yr.min())
    xlim = (xr.min() - px, xr.max() + px)
    ylim = (yr.min() - py, yr.max() + py)
    rotated[timepoint] = (xr, yr, lab, xlim, ylim)
    print(f"{timepoint}: rotation={cfg['rot']}°, flipped={flip_h}, n_points={xr.size}")

def draw_panel(ax, timepoint, cell_type, dot_s):
    xr, yr, lab, xlim, ylim = rotated[timepoint]
    hit = lab == cell_type
    if (~hit).any():
        ax.scatter(xr[~hit], yr[~hit], s=dot_s, color=other_color,
                   alpha=dot_alpha, linewidth=0, zorder=1, rasterized=True)
    if hit.any():
        ax.scatter(xr[hit], yr[hit], s=dot_s, color=highlight_color,
                   alpha=dot_alpha, linewidth=0, zorder=2, rasterized=True)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal", adjustable="box")
    ax.set_facecolor(background)
    ax.axis("off")

col_order = [tp for tp in panels if tp in rotated]

# ---------------- grid: equal heights, column widths follow each tissue ----------------
widths = []
for tp in col_order:
    _, _, _, xlim, ylim = rotated[tp]
    widths.append(panel_h_in * (xlim[1] - xlim[0]) / (ylim[1] - ylim[0]))

n_rows, n_cols = len(row_order), len(col_order)
left_in = label_pad_in if show_labels else 0.0     # room for row labels
top_in  = label_pad_in if show_labels else 0.0     # room for column titles
W = left_in + sum(widths) + (n_cols - 1) * gap_in
H = top_in  + n_rows * panel_h_in + (n_rows - 1) * gap_in
dot_size_grid = dot_size * (panel_h_in / fig_size_in) ** 2
text_color = 'white' if background == 'black' else 'black'

fig = plt.figure(figsize=(W, H))
fig.patch.set_facecolor(background)

for r, ct in enumerate(row_order):
    bottom = (H - top_in - (r + 1) * panel_h_in - r * gap_in) / H
    x0_in = left_in
    for c, (tp, w) in enumerate(zip(col_order, widths)):
        ax = fig.add_axes([x0_in / W, bottom, w / W, panel_h_in / H])
        draw_panel(ax, tp, ct, dot_size_grid)

        if show_labels and r == 0:
            fig.text((x0_in + w / 2) / W, (H - top_in + 0.05) / H, tp,
                     ha='center', va='bottom', fontsize=label_fontsize, color=text_color)
        if show_labels and c == 0:
            fig.text((left_in - 0.05) / W, bottom + panel_h_in / H / 2, label_names[ct],
                     ha='right', va='center', rotation=90, fontsize=label_fontsize, color=text_color)
        x0_in += w + gap_in

# ---------------- save ----------------
outpath = os.path.join(output_dir, "20261113-Spatial_Human-CellTypeHighlight-Grid-spatial.svg")
fig.savefig(outpath, format='svg', dpi=dpi_save, facecolor=background, pad_inches=0)
plt.close()
print(f"Saved {outpath}")

### Fib/dec Subtypes

In [ ]:
# Settings
key_cluster = "cell_subtype"
key_sample  = "timepoint"
x_key, y_key = "x_um_dbscan", "y_um_dbscan"
flip_h = True              # mirror the images left-right (applied after rotation)

# Per timepoint: crop in raw coordinates (None = whole section), then rotate (counter-clockwise)
panels = {
    "secretory": {"crop": [(400, 5500), (750, 5250)], "rot": -90},
    "menstrual": {"crop": None,                         "rot": 90},
}

color_map = {
    # fibroblasts (olive + teals)
    'myoFib':  '#8c8c00',  # olive
    'eFib_a':  '#01665e',  # dark teal
    'eFib_b':  '#35978f',  # teal
    'eFib_c':  '#80cdc1',  # light teal
    # decidual cells (magenta / pink / coral)
    'dec_a':   '#8e0152',  # dark magenta
    'dec_b':   '#de77ae',  # pink
    'dec_c':   '#f46d43',  # coral
    'dec_d':   '#f4a582',  # salmon
}
draw_order  = ['myoFib', 'eFib_a', 'eFib_b', 'dec_a', 'eFib_c',
               'dec_b', 'dec_c', 'dec_d']     # later = on top (rare menstrual states last)
other_color = "#231f20"    # every nucleus that isn't a fib/dec subtype
dot_size, dot_alpha = 8, 1.0
fig_size_in = 6            # reference: longer side of a single image
grid_h_in   = 3            # height of each panel in the combined image (width follows the tissue)
pad_frac    = 0.01         # margin around the tissue, as a fraction of its extent
background  = "black"
dpi_save    = 300

# ---------------- checks ----------------
for col in [key_cluster, key_sample, x_key, y_key]:
    if col not in adata1.obs:
        raise ValueError(f"{col} not found in adata1.obs")
present = set(adata1.obs[key_cluster].astype(str))
not_found = [s for s in color_map if s not in present]
if not_found:
    print(f"Warning: subtypes in color_map but not in adata1: {not_found}")

# ---------------- crop, rotate and flip each section once ----------------
rotated = {}
for timepoint, cfg in panels.items():
    obs = adata1.obs
    mask = (obs[key_sample].astype(str) == timepoint).to_numpy()
    if not mask.any():
        print(f"Warning: no nuclei for {timepoint}. Skipping.")
        continue
    x   = obs.loc[mask, x_key].astype(float).to_numpy()
    y   = obs.loc[mask, y_key].astype(float).to_numpy()
    lab = obs.loc[mask, key_cluster].astype(str).to_numpy()
    ok = np.isfinite(x) & np.isfinite(y)
    x, y, lab = x[ok], y[ok], lab[ok]

    if cfg["crop"] is not None:
        (xmin, xmax), (ymin, ymax) = cfg["crop"]
        keep = (x >= xmin) & (x <= xmax) & (y >= ymin) & (y <= ymax)
        x, y, lab = x[keep], y[keep], lab[keep]
        cx, cy = 0.5 * (xmin + xmax), 0.5 * (ymin + ymax)
    else:
        cx, cy = 0.5 * (x.min() + x.max()), 0.5 * (y.min() + y.max())

    t = np.deg2rad(cfg["rot"])
    xr = (x - cx) * np.cos(t) - (y - cy) * np.sin(t) + cx
    yr = (x - cx) * np.sin(t) + (y - cy) * np.cos(t) + cy
    if flip_h:
        xr = -xr

    # tight limits around the tissue plus a small margin
    px = pad_frac * (xr.max() - xr.min())
    py = pad_frac * (yr.max() - yr.min())
    xlim = (xr.min() - px, xr.max() + px)
    ylim = (yr.min() - py, yr.max() + py)
    rotated[timepoint] = (xr, yr, lab, xlim, ylim)

    n_sub = np.isin(lab, list(color_map)).sum()
    print(f"{timepoint}: rotation={cfg['rot']}°, flipped={flip_h}, {n_sub} fib/dec nuclei of {xr.size}")

def draw_panel(ax, timepoint, dot_s):
    xr, yr, lab, xlim, ylim = rotated[timepoint]
    other = ~np.isin(lab, list(color_map))
    if other.any():
        ax.scatter(xr[other], yr[other], s=dot_s, color=other_color,
                   alpha=dot_alpha, linewidth=0, zorder=1)
    for st in draw_order:
        m = lab == st
        if m.any():
            ax.scatter(xr[m], yr[m], s=dot_s, color=color_map[st],
                       alpha=dot_alpha, linewidth=0, zorder=2)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal", adjustable="box")
    ax.set_facecolor(background)
    ax.axis("off")

order = [tp for tp in panels if tp in rotated]

# ---------------- combined image: equal heights, widths follow each tissue ----------------
widths = []
for tp in order:
    _, _, _, xlim, ylim = rotated[tp]
    widths.append(grid_h_in * (xlim[1] - xlim[0]) / (ylim[1] - ylim[0]))
W, H = sum(widths), grid_h_in
dot_size_grid = dot_size * (grid_h_in / fig_size_in) ** 2

fig = plt.figure(figsize=(W, H))
left = 0.0
for tp, w in zip(order, widths):
    ax = fig.add_axes([left / W, 0, w / W, 1])
    draw_panel(ax, tp, dot_size_grid)
    left += w

fig.patch.set_facecolor(background)
outpath = os.path.join(output_dir, "20261113-Spatial_Human-FibDec-spatial-grid-row.png")
fig.savefig(outpath, dpi=dpi_save, facecolor=background, pad_inches=0)
plt.close()
print(f"Saved {outpath}")

### Fib/Dec Grid

In [ ]:
### Human spatial grid: rows = stromal subtypes, columns = timepoints (crop, rotate, flip) ###

plt.rcParams['svg.fonttype'] = 'none'   # labels stay editable text in the SVG

# Settings
key_cluster = "cell_subtype"
key_sample  = "timepoint"
x_key, y_key = "x_um_dbscan", "y_um_dbscan"
flip_h      = True             # mirror each section left-right (applied after rotation)

# Per timepoint: crop in raw coordinates (None = whole section), then rotate (counter-clockwise)
# (order = columns, left to right)
panels = {
    "secretory": {"crop": [(400, 5500), (750, 5250)], "rot": -90},
    "menstrual": {"crop": None,                         "rot": 90},
}

# Rows, top to bottom: cell_subtype label -> row label
label_names = {
    'myoFib': 'myoFib',
    'eFib_a': 'eFib_a',
    'eFib_b': 'eFib_b',
    'eFib_c': 'eFib_c',
    'dec_a':  'dec_a',
    'dec_b':  'dec_b',
    'dec_c':  'dec_c',
    'dec_d':  'dec_d',
}

highlight_color = "#fe01fe"
other_color     = "#B0B0B0"
dot_size, dot_alpha = 12, 1.0
fig_size_in    = 6             # reference: size of a single image
panel_h_in     = 2.0           # height of each panel (inches); width follows the tissue
gap_in         = 0.025         # space between panels (inches); 0 = touching
pad_frac       = 0.01          # margin around the tissue, as a fraction of its extent
show_labels    = True          # row/column labels outside the grid
label_fontsize = 9
label_pad_in   = 0.35          # room for labels (inches)
background     = "black"
dpi_save       = 300           # resolution of the rasterized panels inside the SVG

# ---------------- checks ----------------
for col in [key_cluster, key_sample, x_key, y_key]:
    if col not in adata2.obs:
        raise ValueError(f"{col} not found in adata2.obs")
present = set(adata2.obs[key_cluster].astype(str))
row_order = [c for c in label_names if c in present]
skipped = [c for c in label_names if c not in present]
if skipped:
    print(f"Warning: subtypes not in adata2.obs['{key_cluster}'], skipped: {skipped}")
    print(f"         labels present: {sorted(present)}")

# ---------------- crop, rotate and flip each section once ----------------
rotated = {}
for timepoint, cfg in panels.items():
    obs = adata2.obs
    mask = (obs[key_sample].astype(str) == timepoint).to_numpy()
    if not mask.any():
        print(f"Warning: no nuclei for {timepoint}. Skipping.")
        continue
    x   = obs.loc[mask, x_key].astype(float).to_numpy()
    y   = obs.loc[mask, y_key].astype(float).to_numpy()
    lab = obs.loc[mask, key_cluster].astype(str).to_numpy()
    ok = np.isfinite(x) & np.isfinite(y)
    x, y, lab = x[ok], y[ok], lab[ok]

    if cfg["crop"] is not None:
        (xmin, xmax), (ymin, ymax) = cfg["crop"]
        keep = (x >= xmin) & (x <= xmax) & (y >= ymin) & (y <= ymax)
        x, y, lab = x[keep], y[keep], lab[keep]
        cx, cy = 0.5 * (xmin + xmax), 0.5 * (ymin + ymax)
    else:
        cx, cy = 0.5 * (x.min() + x.max()), 0.5 * (y.min() + y.max())

    t = np.deg2rad(cfg["rot"])
    xr = (x - cx) * np.cos(t) - (y - cy) * np.sin(t) + cx
    yr = (x - cx) * np.sin(t) + (y - cy) * np.cos(t) + cy
    if flip_h:
        xr = -xr

    # tight limits around the tissue plus a small margin
    px = pad_frac * (xr.max() - xr.min())
    py = pad_frac * (yr.max() - yr.min())
    xlim = (xr.min() - px, xr.max() + px)
    ylim = (yr.min() - py, yr.max() + py)
    rotated[timepoint] = (xr, yr, lab, xlim, ylim)
    print(f"{timepoint}: rotation={cfg['rot']}°, flipped={flip_h}, n_points={xr.size}")

def draw_panel(ax, timepoint, subtype, dot_s):
    xr, yr, lab, xlim, ylim = rotated[timepoint]
    hit = lab == subtype
    if (~hit).any():
        ax.scatter(xr[~hit], yr[~hit], s=dot_s, color=other_color,
                   alpha=dot_alpha, linewidth=0, zorder=1, rasterized=True)
    if hit.any():
        ax.scatter(xr[hit], yr[hit], s=dot_s, color=highlight_color,
                   alpha=dot_alpha, linewidth=0, zorder=2, rasterized=True)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal", adjustable="box")
    ax.set_facecolor(background)
    ax.axis("off")

col_order = [tp for tp in panels if tp in rotated]

# ---------------- grid: equal heights, column widths follow each tissue ----------------
widths = []
for tp in col_order:
    _, _, _, xlim, ylim = rotated[tp]
    widths.append(panel_h_in * (xlim[1] - xlim[0]) / (ylim[1] - ylim[0]))

n_rows, n_cols = len(row_order), len(col_order)
left_in = label_pad_in if show_labels else 0.0     # room for row labels
top_in  = label_pad_in if show_labels else 0.0     # room for column titles
W = left_in + sum(widths) + (n_cols - 1) * gap_in
H = top_in  + n_rows * panel_h_in + (n_rows - 1) * gap_in
dot_size_grid = dot_size * (panel_h_in / fig_size_in) ** 2
text_color = 'white' if background == 'black' else 'black'

fig = plt.figure(figsize=(W, H))
fig.patch.set_facecolor(background)

for r, st in enumerate(row_order):
    bottom = (H - top_in - (r + 1) * panel_h_in - r * gap_in) / H
    x0_in = left_in
    for c, (tp, w) in enumerate(zip(col_order, widths)):
        ax = fig.add_axes([x0_in / W, bottom, w / W, panel_h_in / H])
        draw_panel(ax, tp, st, dot_size_grid)

        if show_labels and r == 0:
            fig.text((x0_in + w / 2) / W, (H - top_in + 0.05) / H, tp,
                     ha='center', va='bottom', fontsize=label_fontsize, color=text_color)
        if show_labels and c == 0:
            fig.text((left_in - 0.05) / W, bottom + panel_h_in / H / 2, label_names[st],
                     ha='right', va='center', rotation=90, fontsize=label_fontsize, color=text_color)
        x0_in += w + gap_in

# ---------------- save ----------------
outpath = os.path.join(output_dir, "20261113-Spatial_Human-FibDecHighlight-Grid-spatial.svg")
fig.savefig(outpath, format='svg', dpi=dpi_save, facecolor=background, pad_inches=0)
plt.close()
print(f"Saved {outpath}")

### Epi subtypes

In [ ]:
# Settings
key_cluster = "cell_subtype"
key_sample  = "timepoint"
x_key, y_key = "x_um_dbscan", "y_um_dbscan"
flip_h = True              # mirror the images left-right (applied after rotation)

# Per timepoint: crop in raw coordinates (None = whole section), then rotate (counter-clockwise)
panels = {
    "secretory": {"crop": [(400, 5500), (750, 5250)], "rot": -90},
    "menstrual": {"crop": None,                         "rot": 90},
}

color_map = {
    'epi1':          '#E3B23C',  # golden yellow
    'epi2':          '#3F7CAC',  # steel blue
    'epi_ciliated':  '#5A9E6F',  # sage green
}
draw_order  = ['epi2', 'epi1', 'epi_ciliated']   # later = on top (rare ciliated cells last)
other_color = "#565656"    # every nucleus that isn't an epithelial subtype
dot_size, dot_alpha = 8, 1.0
fig_size_in = 6            # reference: longer side of a single image
grid_h_in   = 3            # height of each panel in the combined image (width follows the tissue)
pad_frac    = 0.01         # margin around the tissue, as a fraction of its extent
background  = "black"
dpi_save    = 300

# ---------------- checks ----------------
for col in [key_cluster, key_sample, x_key, y_key]:
    if col not in adata1.obs:
        raise ValueError(f"{col} not found in adata1.obs")
present = set(adata1.obs[key_cluster].astype(str))
not_found = [s for s in color_map if s not in present]
if not_found:
    print(f"Warning: subtypes in color_map but not in adata1: {not_found}")

# ---------------- crop, rotate and flip each section once ----------------
rotated = {}
for timepoint, cfg in panels.items():
    obs = adata1.obs
    mask = (obs[key_sample].astype(str) == timepoint).to_numpy()
    if not mask.any():
        print(f"Warning: no nuclei for {timepoint}. Skipping.")
        continue
    x   = obs.loc[mask, x_key].astype(float).to_numpy()
    y   = obs.loc[mask, y_key].astype(float).to_numpy()
    lab = obs.loc[mask, key_cluster].astype(str).to_numpy()
    ok = np.isfinite(x) & np.isfinite(y)
    x, y, lab = x[ok], y[ok], lab[ok]

    if cfg["crop"] is not None:
        (xmin, xmax), (ymin, ymax) = cfg["crop"]
        keep = (x >= xmin) & (x <= xmax) & (y >= ymin) & (y <= ymax)
        x, y, lab = x[keep], y[keep], lab[keep]
        cx, cy = 0.5 * (xmin + xmax), 0.5 * (ymin + ymax)
    else:
        cx, cy = 0.5 * (x.min() + x.max()), 0.5 * (y.min() + y.max())

    t = np.deg2rad(cfg["rot"])
    xr = (x - cx) * np.cos(t) - (y - cy) * np.sin(t) + cx
    yr = (x - cx) * np.sin(t) + (y - cy) * np.cos(t) + cy
    if flip_h:
        xr = -xr

    # tight limits around the tissue plus a small margin
    px = pad_frac * (xr.max() - xr.min())
    py = pad_frac * (yr.max() - yr.min())
    xlim = (xr.min() - px, xr.max() + px)
    ylim = (yr.min() - py, yr.max() + py)
    rotated[timepoint] = (xr, yr, lab, xlim, ylim)

    counts = ', '.join(f"{st}={int((lab == st).sum())}" for st in color_map)
    print(f"{timepoint}: rotation={cfg['rot']}°, flipped={flip_h}, {xr.size} nuclei; {counts}")

def draw_panel(ax, timepoint, dot_s):
    xr, yr, lab, xlim, ylim = rotated[timepoint]
    other = ~np.isin(lab, list(color_map))
    if other.any():
        ax.scatter(xr[other], yr[other], s=dot_s, color=other_color,
                   alpha=dot_alpha, linewidth=0, zorder=1)
    for st in draw_order:
        m = lab == st
        if m.any():
            ax.scatter(xr[m], yr[m], s=dot_s, color=color_map[st],
                       alpha=dot_alpha, linewidth=0, zorder=2)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal", adjustable="box")
    ax.set_facecolor(background)
    ax.axis("off")

order = [tp for tp in panels if tp in rotated]

# ---------------- combined image: equal heights, widths follow each tissue ----------------
widths = []
for tp in order:
    _, _, _, xlim, ylim = rotated[tp]
    widths.append(grid_h_in * (xlim[1] - xlim[0]) / (ylim[1] - ylim[0]))
W, H = sum(widths), grid_h_in
dot_size_grid = dot_size * (grid_h_in / fig_size_in) ** 2

fig = plt.figure(figsize=(W, H))
left = 0.0
for tp, w in zip(order, widths):
    ax = fig.add_axes([left / W, 0, w / W, 1])
    draw_panel(ax, tp, dot_size_grid)
    left += w

fig.patch.set_facecolor(background)
outpath = os.path.join(output_dir, "20261113-Spatial_Human-Epi-spatial-grid-row.png")
fig.savefig(outpath, dpi=dpi_save, facecolor=background, pad_inches=0)
plt.show()
print(f"Saved {outpath}")

### Transferred eFib/dec labels

In [ ]:
# Settings
key_label   = "transferred_mouse_fibdec"   # in adata2.obs
key_sample  = "timepoint"
x_key, y_key = "x_um_dbscan", "y_um_dbscan"
flip_h = True              # mirror the images left-right (applied after rotation)

# Per timepoint: crop in raw coordinates (None = whole section), then rotate (counter-clockwise)
panels = {
    "secretory": {"crop": [(400, 5500), (750, 5250)], "rot": -90},
    "menstrual": {"crop": None,                       "rot": 90},
}

# Mouse eFib/dec colors (same as the mouse figures)
color_map = {
    'decidual_cell':  '#00B4C5',  # cyan
    'endometrial_fibroblast': '#CC79A7',  # reddish purple
    'nan':   '#231f20'
}

draw_order  = ['nan', 'endometrial_fibroblast', 'decidual_cell']        # later = on top (same as mouse)
other_color = "#231f20" 
dot_size, dot_alpha = 8, 1.0
fig_size_in = 6            # reference: longer side of a single image
grid_h_in   = 3            # height of each panel in the combined image (width follows the tissue)
pad_frac    = 0.01         # margin around the tissue, as a fraction of its extent
background  = "black"
dpi_save    = 300

# ---------------- checks ----------------
for col in [key_sample, x_key, y_key]:
    if col not in adata1.obs:
        raise ValueError(f"{col} not found in adata1.obs")
if key_label not in adata2.obs:
    raise ValueError(f"{key_label} not found in adata2.obs")
n_missing = (~adata2.obs_names.isin(adata1.obs_names)).sum()
if n_missing:
    raise ValueError(f"{n_missing} cells in adata2 are not in adata1")

present = set(adata2.obs[key_label].dropna().astype(str))
no_color = sorted(present - set(color_map))
if no_color:
    print(f"Warning: labels without a color (drawn as {other_color}): {no_color}")
not_found = [s for s in color_map if s not in present]
if not_found:
    print(f"Note: colored labels not present in adata2: {not_found}")

# Transferred label for every adata1 nucleus ('' = not in adata2)
lab_all = pd.Series('', index=adata1.obs_names, dtype=object)
lab_all.loc[adata2.obs_names] = adata2.obs[key_label].astype(str).values

# ---------------- crop, rotate and flip each section once ----------------
rotated = {}
for timepoint, cfg in panels.items():
    obs = adata1.obs
    mask = (obs[key_sample].astype(str) == timepoint).to_numpy()
    if not mask.any():
        print(f"Warning: no nuclei for {timepoint}. Skipping.")
        continue
    x   = obs.loc[mask, x_key].astype(float).to_numpy()
    y   = obs.loc[mask, y_key].astype(float).to_numpy()
    lab = lab_all.to_numpy()[mask]
    ok = np.isfinite(x) & np.isfinite(y)
    x, y, lab = x[ok], y[ok], lab[ok]

    if cfg["crop"] is not None:
        (xmin, xmax), (ymin, ymax) = cfg["crop"]
        keep = (x >= xmin) & (x <= xmax) & (y >= ymin) & (y <= ymax)
        x, y, lab = x[keep], y[keep], lab[keep]
        cx, cy = 0.5 * (xmin + xmax), 0.5 * (ymin + ymax)
    else:
        cx, cy = 0.5 * (x.min() + x.max()), 0.5 * (y.min() + y.max())

    t = np.deg2rad(cfg["rot"])
    xr = (x - cx) * np.cos(t) - (y - cy) * np.sin(t) + cx
    yr = (x - cx) * np.sin(t) + (y - cy) * np.cos(t) + cy
    if flip_h:
        xr = -xr

    # tight limits around the tissue plus a small margin
    px = pad_frac * (xr.max() - xr.min())
    py = pad_frac * (yr.max() - yr.min())
    xlim = (xr.min() - px, xr.max() + px)
    ylim = (yr.min() - py, yr.max() + py)
    rotated[timepoint] = (xr, yr, lab, xlim, ylim)

    counts = pd.Series(lab[np.isin(lab, list(color_map))]).value_counts().reindex(draw_order, fill_value=0)
    print(f"{timepoint}: rotation={cfg['rot']}°, flipped={flip_h}, "
          f"{int(counts.sum())} labelled nuclei of {xr.size}: "
          + ', '.join(f"{k}={v}" for k, v in counts.items()))

def draw_panel(ax, timepoint, dot_s):
    xr, yr, lab, xlim, ylim = rotated[timepoint]
    other = ~np.isin(lab, list(color_map))
    if other.any():
        ax.scatter(xr[other], yr[other], s=dot_s, color=other_color,
                   alpha=dot_alpha, linewidth=0, zorder=1)
    for st in draw_order:
        m = lab == st
        if m.any():
            ax.scatter(xr[m], yr[m], s=dot_s, color=color_map[st],
                       alpha=dot_alpha, linewidth=0, zorder=2)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal", adjustable="box")
    ax.set_facecolor(background)
    ax.axis("off")

order = [tp for tp in panels if tp in rotated]

# ---------------- combined image: equal heights, widths follow each tissue ----------------
widths = []
for tp in order:
    _, _, _, xlim, ylim = rotated[tp]
    widths.append(grid_h_in * (xlim[1] - xlim[0]) / (ylim[1] - ylim[0]))
W, H = sum(widths), grid_h_in
dot_size_grid = dot_size * (grid_h_in / fig_size_in) ** 2

fig = plt.figure(figsize=(W, H))
left = 0.0
for tp, w in zip(order, widths):
    ax = fig.add_axes([left / W, 0, w / W, 1])
    draw_panel(ax, tp, dot_size_grid)
    left += w

fig.patch.set_facecolor(background)
outpath = os.path.join(output_dir, "20261113-Spatial_Human-transferredMouseFibDec-spatial-grid-row.png")
fig.savefig(outpath, dpi=dpi_save, facecolor=background, pad_inches=0)
plt.close()
print(f"Saved {outpath}")

### Transferred Pseudotime

In [ ]:
# Settings
key_value  = "transferred_mouse_pseudotime"   # in adata2.obs
key_sample = "timepoint"
x_key, y_key = "x_um_dbscan", "y_um_dbscan"
flip_h = True                     # mirror the images left-right (applied after rotation)

# Per timepoint: crop in raw coordinates (None = whole section), then rotate (counter-clockwise)
panels = {
    "secretory": {"crop": [(400, 5500), (750, 5250)], "rot": -90},
    "menstrual": {"crop": None,                         "rot": 90},
}

cmap        = plt.get_cmap("viridis")
color_range = (0.0, 1.0)          # same scale as the mouse pseudotime; None = min-max of the human values
other_color = "#231f20"           # nuclei without a pseudotime value, drawn at the bottom
dot_size, dot_alpha = 8, 1.0      # same dot size as the 6-inch human single images
fig_size_in = 6                   # longer side of each image (the other side follows the tissue)
pad_frac    = 0.01                # margin around the tissue, as a fraction of its extent
background  = "black"
dpi_save    = 300

# Pseudotime discontinuity filter (same as mouse); set() = none
filter_groups = set()             # e.g. {"menstrual"}
filter_k      = 5
filter_q      = 0.90

# ---------------- checks ----------------
for col in [key_sample, x_key, y_key]:
    if col not in adata1.obs:
        raise ValueError(f"{col} not found in adata1.obs")
if key_value not in adata2.obs:
    raise ValueError(f"{key_value} not found in adata2.obs")
n_missing = (~adata2.obs_names.isin(adata1.obs_names)).sum()
if n_missing:
    raise ValueError(f"{n_missing} cells in adata2 are not in adata1")

# Timepoint and coordinates from adata1 (as in the other images), pseudotime from adata2
obs2 = adata1.obs.loc[adata2.obs_names, [key_sample, x_key, y_key]].copy()
obs2[key_value] = adata2.obs[key_value].astype(float).values

vals_all = obs2[key_value].to_numpy(float)
dmin, dmax = float(np.nanmin(vals_all)), float(np.nanmax(vals_all))
vmin, vmax = color_range if color_range is not None else (dmin, dmax)
print(f"{key_value}: data range {dmin:.3f}–{dmax:.3f}, color scale {vmin:.3f}–{vmax:.3f}, "
      f"{np.isfinite(vals_all).sum():,}/{vals_all.size:,} nuclei have a value")

def pt_discontinuity_filter(xy, pt, k=5, q=0.90):
    """True = keep: |pt - median pt of k nearest spatial neighbours| <= its q-quantile."""
    xy, pt = np.asarray(xy, float), np.asarray(pt, float)
    ok = np.isfinite(pt) & np.isfinite(xy).all(axis=1)
    xy_ok, pt_ok = xy[ok], pt[ok]
    _, j = cKDTree(xy_ok).query(xy_ok, k=k + 1)    # column 0 is the cell itself
    resid = np.abs(pt_ok - np.median(pt_ok[j[:, 1:]], axis=1))
    keep = np.zeros(len(pt), bool)
    keep[ok] = resid <= np.quantile(resid, q)
    return keep

def rotate_flip(x, y, deg, cx, cy):
    t = np.deg2rad(deg)
    xr = (x - cx) * np.cos(t) - (y - cy) * np.sin(t) + cx
    yr = (x - cx) * np.sin(t) + (y - cy) * np.cos(t) + cy
    return (-xr if flip_h else xr), yr

# ---------------- one image per panel ----------------
for timepoint, cfg in panels.items():
    # Frame from the whole section in adata1 (same crop, center and limits as the other human images)
    m1 = (adata1.obs[key_sample].astype(str) == timepoint).to_numpy()
    if not m1.any():
        print(f"Warning: no nuclei for {timepoint} in adata1. Skipping.")
        continue
    fx = adata1.obs.loc[m1, x_key].astype(float).to_numpy()
    fy = adata1.obs.loc[m1, y_key].astype(float).to_numpy()
    ok1 = np.isfinite(fx) & np.isfinite(fy)
    fx, fy = fx[ok1], fy[ok1]
    if cfg["crop"] is not None:
        (xmin, xmax), (ymin, ymax) = cfg["crop"]
        keep1 = (fx >= xmin) & (fx <= xmax) & (fy >= ymin) & (fy <= ymax)
        fx, fy = fx[keep1], fy[keep1]
        cx, cy = 0.5 * (xmin + xmax), 0.5 * (ymin + ymax)
    else:
        cx, cy = 0.5 * (fx.min() + fx.max()), 0.5 * (fy.min() + fy.max())
    fxr, fyr = rotate_flip(fx, fy, cfg["rot"], cx, cy)
    px = pad_frac * (fxr.max() - fxr.min())
    py = pad_frac * (fyr.max() - fyr.min())
    xlim = (fxr.min() - px, fxr.max() + px)
    ylim = (fyr.min() - py, fyr.max() + py)

    # adata2 nuclei of this timepoint
    m2 = obs2[key_sample].astype(str) == timepoint
    x   = obs2.loc[m2, x_key].to_numpy(float)
    y   = obs2.loc[m2, y_key].to_numpy(float)
    val = obs2.loc[m2, key_value].to_numpy(float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y, val = x[ok], y[ok], val[ok]
    if cfg["crop"] is not None:
        keep = (x >= xmin) & (x <= xmax) & (y >= ymin) & (y <= ymax)
        x, y, val = x[keep], y[keep], val[keep]
    if x.size == 0:
        print(f"Warning: no adata2 nuclei for {timepoint}. Skipping.")
        continue

    # Discontinuity filter on the section (before plotting); nuclei without pseudotime are kept
    if timepoint in filter_groups:
        has_pt = np.isfinite(val)
        if has_pt.sum() > filter_k:
            keep_pt = pt_discontinuity_filter(np.c_[x, y], val, k=filter_k, q=filter_q)
            drop = has_pt & ~keep_pt
            print(f"{timepoint}: discontinuity filter removed {drop.sum():,} / {has_pt.sum():,} pseudotime nuclei")
            x, y, val = x[~drop], y[~drop], val[~drop]

    xr, yr = rotate_flip(x, y, cfg["rot"], cx, cy)
    has_val = np.isfinite(val)
    print(f"{timepoint}: rotation={cfg['rot']}°, flipped={flip_h}, {xr.size} nuclei, "
          f"{has_val.sum()} with pseudotime, median {np.nanmedian(val):.3f}")

    # Figure shaped like the tissue, longer side = fig_size_in
    xr_range, yr_range = xlim[1] - xlim[0], ylim[1] - ylim[0]
    if xr_range >= yr_range:
        fig_w, fig_h = fig_size_in, fig_size_in * yr_range / xr_range
    else:
        fig_w, fig_h = fig_size_in * xr_range / yr_range, fig_size_in
    fig = plt.figure(figsize=(fig_w, fig_h))
    ax = fig.add_axes([0, 0, 1, 1])

    # Nuclei without pseudotime first (bottom layer)
    if (~has_val).any():
        ax.scatter(xr[~has_val], yr[~has_val], s=dot_size, color=other_color,
                   alpha=dot_alpha, linewidth=0, zorder=1)

    # Nuclei with pseudotime, sorted so later pseudotime is drawn on top
    if has_val.any():
        o = np.argsort(val[has_val])
        ax.scatter(xr[has_val][o], yr[has_val][o], c=val[has_val][o],
                   cmap=cmap, vmin=vmin, vmax=vmax,
                   s=dot_size, alpha=dot_alpha, linewidth=0, zorder=2)

    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal", adjustable="box")
    ax.set_facecolor(background)
    ax.axis("off")
    fig.patch.set_facecolor(background)

    outpath = os.path.join(output_dir, f"20261113-Spatial_Human-{timepoint}-transferred_pseudotime.png")
    fig.savefig(outpath, dpi=dpi_save, facecolor=background, pad_inches=0)
    plt.close(fig)
    print(f"Saved {outpath}")

# ---------------- colorbar as a separate image ----------------
fig, ax = plt.subplots(figsize=(0.3, 2.0))
cb = fig.colorbar(plt.cm.ScalarMappable(norm=plt.Normalize(vmin, vmax), cmap=cmap), cax=ax)
cb.outline.set_visible(False)
cb.set_ticks([])
fig.patch.set_alpha(0)
ax.patch.set_alpha(0)

outpath = os.path.join(output_dir, "20261113-Spatial_Human-transferred_pseudotime-colorbar.png")
fig.savefig(outpath, dpi=dpi_save, bbox_inches="tight", pad_inches=0, transparent=True)
plt.close(fig)
print(f"Saved {outpath}")

### Stro-7 module scores

In [ ]:
# Settings
score_col    = "Ulrich2024_sg_Stro7_top50"   # in adata2.obs
key_sample   = "timepoint"
x_key, y_key = "x_um_dbscan", "y_um_dbscan"
flip_h = True                     # mirror the images left-right (applied after rotation)

# Per timepoint: crop in raw coordinates (None = whole section), then rotate (counter-clockwise)
panels = {
    "secretory": {"crop": [(400, 5500), (750, 5250)], "rot": -90},
    "menstrual": {"crop": None,                         "rot": 90},
}

cmap            = plt.get_cmap("viridis")
pct             = (1, 99)         # shared color range = these percentiles of all scored nuclei in both panels
show_background = True            # draw nuclei without a score (not in adata2, or NaN) underneath
other_color     = "#231f20"
dot_size, dot_alpha = 8, 1.0      # same dot size as the 6-inch human single images
fig_size_in = 6                   # reference: longer side of a single image
grid_h_in   = 3                   # height of each panel in the combined image (width follows the tissue)
pad_frac    = 0.01                # margin around the tissue, as a fraction of its extent
background  = "black"
dpi_save    = 300

outpath = os.path.join(output_dir, f"20261113-Spatial_Human-{score_col}-spatial-grid-row.png")
cb_path = os.path.join(output_dir, f"20261113-Spatial_Human-{score_col}-colorbar.png")

# ---------------- checks ----------------
for col in [key_sample, x_key, y_key]:
    if col not in adata1.obs:
        raise ValueError(f"{col} not found in adata1.obs")
if score_col not in adata2.obs:
    raise ValueError(f"{score_col} not found in adata2.obs")
n_missing = (~adata2.obs_names.isin(adata1.obs_names)).sum()
if n_missing:
    raise ValueError(f"{n_missing} cells in adata2 are not in adata1")

# Score for every adata1 nucleus (NaN if not in adata2 or not scored)
score_all = pd.Series(np.nan, index=adata1.obs_names)
score_all.loc[adata2.obs_names] = adata2.obs[score_col].astype(float).values

# ---------------- crop, rotate and flip each section once ----------------
rotated = {}
for tp, cfg in panels.items():
    obs = adata1.obs
    mask = (obs[key_sample].astype(str) == tp).to_numpy()
    if not mask.any():
        print(f"Warning: no nuclei for {tp}. Skipping.")
        continue
    x = obs.loc[mask, x_key].astype(float).to_numpy()
    y = obs.loc[mask, y_key].astype(float).to_numpy()
    s = score_all.to_numpy()[mask]
    ok = np.isfinite(x) & np.isfinite(y)
    x, y, s = x[ok], y[ok], s[ok]

    if cfg["crop"] is not None:
        (xmin, xmax), (ymin, ymax) = cfg["crop"]
        keep = (x >= xmin) & (x <= xmax) & (y >= ymin) & (y <= ymax)
        x, y, s = x[keep], y[keep], s[keep]
        cx, cy = 0.5 * (xmin + xmax), 0.5 * (ymin + ymax)
    else:
        cx, cy = 0.5 * (x.min() + x.max()), 0.5 * (y.min() + y.max())

    t = np.deg2rad(cfg["rot"])
    xr = (x - cx) * np.cos(t) - (y - cy) * np.sin(t) + cx
    yr = (x - cx) * np.sin(t) + (y - cy) * np.cos(t) + cy
    if flip_h:
        xr = -xr

    # tight limits around the tissue plus a small margin
    px = pad_frac * (xr.max() - xr.min())
    py = pad_frac * (yr.max() - yr.min())
    xlim = (xr.min() - px, xr.max() + px)
    ylim = (yr.min() - py, yr.max() + py)
    rotated[tp] = (xr, yr, s, xlim, ylim)

order = [tp for tp in panels if tp in rotated]

# ---------------- shared color range ----------------
all_s = np.concatenate([rotated[tp][2] for tp in order])
vmin, vmax = np.nanpercentile(all_s, pct)

summary = pd.DataFrame({
    'nuclei':        [rotated[tp][2].size for tp in order],
    'scored':        [int(np.isfinite(rotated[tp][2]).sum()) for tp in order],
    'median score':  [np.nanmedian(rotated[tp][2]) for tp in order],
    'score min':     [np.nanmin(rotated[tp][2]) for tp in order],
    'score max':     [np.nanmax(rotated[tp][2]) for tp in order],
}, index=pd.Index(order, name=score_col))
print(summary.round(3).to_string())
print(f"\nShared color scale: {vmin:.3f} to {vmax:.3f} ({pct[0]}st–{pct[1]}th percentile)\n")

def draw_panel(ax, tp, dot_s):
    xr, yr, s, xlim, ylim = rotated[tp]
    has = np.isfinite(s)
    if show_background and (~has).any():
        ax.scatter(xr[~has], yr[~has], s=dot_s, color=other_color,
                   alpha=dot_alpha, linewidth=0, zorder=1)
    o = np.argsort(s[has])                   # highest scores drawn on top
    ax.scatter(xr[has][o], yr[has][o], c=s[has][o], cmap=cmap, vmin=vmin, vmax=vmax,
               s=dot_s, alpha=dot_alpha, linewidth=0, zorder=2)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal", adjustable="box")
    ax.set_facecolor(background)
    ax.axis("off")

# ---------------- combined image: equal heights, widths follow each tissue ----------------
widths = []
for tp in order:
    _, _, _, xlim, ylim = rotated[tp]
    widths.append(grid_h_in * (xlim[1] - xlim[0]) / (ylim[1] - ylim[0]))
W, H = sum(widths), grid_h_in
dot_size_grid = dot_size * (grid_h_in / fig_size_in) ** 2

fig = plt.figure(figsize=(W, H))
left = 0.0
for tp, w in zip(order, widths):
    ax = fig.add_axes([left / W, 0, w / W, 1])
    draw_panel(ax, tp, dot_size_grid)
    left += w

fig.patch.set_facecolor(background)
fig.savefig(outpath, dpi=dpi_save, facecolor=background, pad_inches=0)
plt.show()
print(f"Saved {outpath}")

# ---------------- colorbar as a separate image ----------------
fig, ax = plt.subplots(figsize=(0.3, 2.0))
cb = fig.colorbar(plt.cm.ScalarMappable(norm=plt.Normalize(vmin, vmax), cmap=cmap), cax=ax)
cb.outline.set_visible(False)
cb.set_ticks([])
fig.patch.set_alpha(0)
ax.patch.set_alpha(0)
fig.savefig(cb_path, dpi=dpi_save, bbox_inches="tight", pad_inches=0, transparent=True)
plt.close(fig)
print(f"Saved {cb_path}  (scale {vmin:.3f} to {vmax:.3f})")

## Functionalis/Basalis Markers

In [ ]:
plt.rcParams['svg.fonttype'] = 'none'   # labels stay editable text in the SVG

# ---------------------------- settings ----------------------------
# One row per gene, top to bottom: (mouse symbol, human symbol)
gene_pairs = [
    ('Ebf1', 'EBF1'), ('Lama2', 'LAMA2'), ('Adamts19', 'ADAMTS19'), ('Mir100hg', 'MIR100HG'),
    ('Slc8a1', 'SLC8A1'), ('Ror2', 'ROR2'), ('Igfbp5', 'IGFBP5'), ('Mamdc2', 'MAMDC2'),
    ('Gja1', 'GJA1'), ('Inhba', 'INHBA'), ('Tnfrsf21', 'TNFRSF21'), ('Rhob', 'RHOB'),
    ('Igfbp4', 'IGFBP4'), ('Cebpb', 'CEBPB'),
]

key_sample   = "timepoint"
x_key, y_key = "x_um_dbscan", "y_um_dbscan"
mouse_layer  = 'log1p_counts'   # adata3
human_layer  = 'log1p_counts'   # adata1

# Mouse (adata3): rotate around the whole section, then show this window of the rotated image
mouse_panels = {
    "1.5dpi_late": {"rot": 303, "view": [(150, 6250),  (-300, 5800)]},
    "3.5dpi":      {"rot": 135, "view": [(2600, 9100), (2200, 8700)]},
}
# Human (adata1): crop in raw coordinates (None = whole section), rotate, then flip
human_panels = {
    "secretory": {"crop": [(400, 5500), (750, 5250)], "rot": -90},
    "menstrual": {"crop": None,                         "rot": 90},
}
human_flip_h = True

cmap          = plt.get_cmap("viridis")
vmax_pct      = 99.5         # per gene and species: top of scale = this percentile of expressing cells in view
mouse_dot_ref = 25           # dot size of the 6-inch mouse single images
human_dot_ref = 8            # dot size of the 6-inch human single images
fig_size_in   = 6            # reference size for dot scaling
dot_alpha     = 1.0

panel_h_in     = 1.6         # height of every panel (inches); width follows the window/tissue
gap_in         = 0.03        # space between the two timepoints of one species
cbar_gap_in    = 0.05        # space between a species block and its colorbar
cbar_w_in      = 0.07        # colorbar thickness
cbar_lab_in    = 0.30        # room for the colorbar numbers
species_gap_in = 0.20        # extra space between the mouse and human blocks
row_gap_in     = 0.04        # space between gene rows
row_label_in   = 1.10        # room for gene names on the left
header_in      = 0.45        # room for species + timepoint titles
pad_frac       = 0.01        # human: margin around the tissue
fontsize       = 8
other_color    = "#B0B0B0"   # used if a gene is missing in one species
background     = "black"
dpi_save       = 300         # resolution of the rasterized panels inside the SVG
text_color     = 'white' if background == 'black' else 'black'

outpath = os.path.join(output_dir, "20261113-Spatial_MouseHuman-func-bas_markers-grid.svg")

# ---------------------------- checks ----------------------------
for a, name, layer in [(adata3, 'adata3', mouse_layer), (adata1, 'adata1', human_layer)]:
    for col in [key_sample, x_key, y_key]:
        if col not in a.obs:
            raise ValueError(f"{col} not found in {name}.obs")
    if layer not in a.layers:
        raise ValueError(f"Layer '{layer}' not in {name}.layers: {list(a.layers)}")
for m, h in gene_pairs:
    if m not in adata3.var_names:
        print(f"Warning: {m} not in adata3.var_names (mouse panels drawn grey)")
    if h not in adata1.var_names:
        print(f"Warning: {h} not in adata1.var_names (human panels drawn grey)")

# ---------------------------- expression ----------------------------
def to_1d(X):
    return (X.toarray() if sps.issparse(X) else np.asarray(X)).ravel().astype(float)

def as_csc(X):
    return sps.csc_matrix(X) if sps.issparse(X) else np.asarray(X)   # fast column (gene) access

_M = as_csc(adata3.layers[mouse_layer])
_H = as_csc(adata1.layers[human_layer])

def mouse_expr(gene):
    return to_1d(_M[:, adata3.var_names.get_loc(gene)])

def human_expr(gene):
    return to_1d(_H[:, adata1.var_names.get_loc(gene)])

# ---------------------------- rotate / crop each section once ----------------------------
def prep_mouse():
    out = {}
    labels = adata3.obs[key_sample].astype(str).to_numpy()
    xs = adata3.obs[x_key].astype(float).to_numpy()
    ys = adata3.obs[y_key].astype(float).to_numpy()
    for tp, cfg in mouse_panels.items():
        idx = np.where(labels == tp)[0]
        x, y = xs[idx], ys[idx]
        ok = np.isfinite(x) & np.isfinite(y)
        idx, x, y = idx[ok], x[ok], y[ok]
        if idx.size == 0:
            print(f"Warning: no mouse nuclei for {tp}. Skipping.")
            continue
        cx, cy = 0.5 * (x.min() + x.max()), 0.5 * (y.min() + y.max())   # whole-section centre
        t = np.deg2rad(cfg["rot"])
        xr = (x - cx) * np.cos(t) - (y - cy) * np.sin(t) + cx
        yr = (x - cx) * np.sin(t) + (y - cy) * np.cos(t) + cy
        (vx0, vx1), (vy0, vy1) = cfg["view"]
        inv = (xr >= vx0) & (xr <= vx1) & (yr >= vy0) & (yr <= vy1)
        out[tp] = dict(idx=idx[inv], x=xr[inv], y=yr[inv], xlim=(vx0, vx1), ylim=(vy0, vy1))
        print(f"mouse {tp}: {inv.sum()} nuclei in view")
    return out

def prep_human():
    out = {}
    labels = adata1.obs[key_sample].astype(str).to_numpy()
    xs = adata1.obs[x_key].astype(float).to_numpy()
    ys = adata1.obs[y_key].astype(float).to_numpy()
    for tp, cfg in human_panels.items():
        idx = np.where(labels == tp)[0]
        x, y = xs[idx], ys[idx]
        ok = np.isfinite(x) & np.isfinite(y)
        idx, x, y = idx[ok], x[ok], y[ok]
        if cfg["crop"] is not None:
            (xmin, xmax), (ymin, ymax) = cfg["crop"]
            keep = (x >= xmin) & (x <= xmax) & (y >= ymin) & (y <= ymax)
            idx, x, y = idx[keep], x[keep], y[keep]
            cx, cy = 0.5 * (xmin + xmax), 0.5 * (ymin + ymax)
        elif idx.size:
            cx, cy = 0.5 * (x.min() + x.max()), 0.5 * (y.min() + y.max())
        if idx.size == 0:
            print(f"Warning: no human nuclei for {tp}. Skipping.")
            continue
        t = np.deg2rad(cfg["rot"])
        xr = (x - cx) * np.cos(t) - (y - cy) * np.sin(t) + cx
        yr = (x - cx) * np.sin(t) + (y - cy) * np.cos(t) + cy
        if human_flip_h:
            xr = -xr
        px = pad_frac * (xr.max() - xr.min())
        py = pad_frac * (yr.max() - yr.min())
        out[tp] = dict(idx=idx, x=xr, y=yr,
                       xlim=(xr.min() - px, xr.max() + px), ylim=(yr.min() - py, yr.max() + py))
        print(f"human {tp}: {idx.size} nuclei")
    return out

dot_scale = (panel_h_in / fig_size_in) ** 2
blocks = [
    dict(name="Mouse", adata=adata3, panels=prep_mouse(), expr=mouse_expr, dot=mouse_dot_ref * dot_scale, gi=0),
    dict(name="Human", adata=adata1, panels=prep_human(), expr=human_expr, dot=human_dot_ref * dot_scale, gi=1),
]

# ---------------------------- layout (inches) ----------------------------
x_in = row_label_in
for b_i, b in enumerate(blocks):
    b["pos"] = []
    tps = list(b["panels"])
    for i, tp in enumerate(tps):
        p = b["panels"][tp]
        w = panel_h_in * (p["xlim"][1] - p["xlim"][0]) / (p["ylim"][1] - p["ylim"][0])
        b["pos"].append((tp, x_in, w))
        x_in += w + (gap_in if i < len(tps) - 1 else 0)
    b["cbar_x"] = x_in + cbar_gap_in
    x_in = b["cbar_x"] + cbar_w_in + cbar_lab_in
    if b_i < len(blocks) - 1:
        x_in += species_gap_in
W = x_in
n_rows = len(gene_pairs)
H = header_in + n_rows * panel_h_in + (n_rows - 1) * row_gap_in

fig = plt.figure(figsize=(W, H))
fig.patch.set_facecolor(background)

# Headers: species over each block, timepoint over each column
for b in blocks:
    x_start = b["pos"][0][1]
    x_end = b["pos"][-1][1] + b["pos"][-1][2]
    fig.text((x_start + x_end) / 2 / W, (H - 0.05) / H, b["name"],
             ha='center', va='top', fontsize=fontsize + 1, fontweight='bold', color=text_color)
    for tp, x0, w in b["pos"]:
        fig.text((x0 + w / 2) / W, (H - header_in + 0.05) / H, tp,
                 ha='center', va='bottom', fontsize=fontsize, color=text_color)

# ---------------------------- one row per gene ----------------------------
for r, pair in enumerate(gene_pairs):
    bottom_in = H - header_in - (r + 1) * panel_h_in - r * row_gap_in

    fig.text((row_label_in - 0.08) / W, (bottom_in + panel_h_in / 2) / H, f"{pair[0]} / {pair[1]}",
             ha='right', va='center', fontsize=fontsize, fontstyle='italic', color=text_color)

    for b in blocks:
        gene = pair[b["gi"]]
        has_gene = gene in b["adata"].var_names
        expr = b["expr"](gene) if has_gene else None

        # Color scale per gene and species, shared by its two timepoints
        if has_gene:
            pos = np.concatenate([expr[b["panels"][tp]["idx"]] for tp, _, _ in b["pos"]])
            pos = pos[pos > 0]
            vmax = float(np.percentile(pos, vmax_pct)) if pos.size else 1.0

        for tp, x0, w in b["pos"]:
            p = b["panels"][tp]
            ax = fig.add_axes([x0 / W, bottom_in / H, w / W, panel_h_in / H])
            if has_gene:
                val = expr[p["idx"]]
                o = np.argsort(val)                 # higher expression drawn on top
                ax.scatter(p["x"][o], p["y"][o], c=val[o], cmap=cmap, vmin=0, vmax=vmax,
                           s=b["dot"], alpha=dot_alpha, linewidth=0, rasterized=True)
            else:
                ax.scatter(p["x"], p["y"], color=other_color,
                           s=b["dot"], alpha=dot_alpha, linewidth=0, rasterized=True)
            ax.set_xlim(*p["xlim"])
            ax.set_ylim(*p["ylim"])
            ax.set_aspect("equal", adjustable="box")
            ax.set_facecolor(background)
            ax.axis("off")

        # Colorbar for this species and gene, with 0 and max
        if has_gene:
            cax = fig.add_axes([b["cbar_x"] / W, (bottom_in + 0.2 * panel_h_in) / H,
                                cbar_w_in / W, 0.6 * panel_h_in / H])
            cb = fig.colorbar(plt.cm.ScalarMappable(norm=plt.Normalize(0, vmax), cmap=cmap), cax=cax)
            cb.outline.set_visible(False)
            cb.set_ticks([0, vmax])
            cb.set_ticklabels(["0", f"{vmax:.1f}"])
            cax.tick_params(labelsize=fontsize - 1, colors=text_color, length=0, pad=2)

# ---------------------------- save ----------------------------
fig.savefig(outpath, format='svg', dpi=dpi_save, facecolor=background, pad_inches=0)
plt.close()
print(f"Saved {outpath}  ({W:.1f} x {H:.1f} in)")

# QC Graphs

## Mouse

### Cells Per sample the mouse dataset

In [ ]:
# ---------------------------- settings ----------------------------
group_key   = 'timepoint'
sample_key  = 'sample'
x_key, y_key = 'x_um_dbscan', 'y_um_dbscan'
group_order = ['uninduced', '1.5dpi_early', '1.5dpi_late', '2.5dpi', '3.5dpi', '4.5dpi_menstrual']

# ---------------------------- metadata ----------------------------
metadata = adata3.obs[[group_key, sample_key, x_key, y_key]].copy()
metadata[group_key] = metadata[group_key].astype(str)
metadata['mck'] = metadata[sample_key].astype(str).str.extract(r'mck_?(\d+)', expand=False).radd('mck_')
metadata['mapped'] = metadata[x_key].notna() & metadata[y_key].notna()   # has a spatial position

extra = sorted(set(metadata[group_key]) - set(group_order))
if extra:
    print(f"Note: timepoints not in group_order are listed at the end: {extra}")
order = [g for g in group_order if g in set(metadata[group_key])] + extra

def summarize(df, by):
    out = df.groupby(by, observed=True).agg(total=('mapped', 'size'), mapped=('mapped', 'sum'))
    out['% mapped'] = (out['mapped'] / out['total'] * 100).round(2)
    return out

# ---------------------------- per timepoint ----------------------------
per_tp = summarize(metadata, group_key).reindex(order)
per_tp.loc['Total'] = [len(metadata), int(metadata['mapped'].sum()),
                       round(metadata['mapped'].mean() * 100, 2)]
per_tp[['total', 'mapped']] = per_tp[['total', 'mapped']].astype(int)

print("Mouse dataset: mapped nuclei per timepoint")
display(per_tp)

### Counts/Genes per Cell

In [ ]:
# ---------------------------- settings ----------------------------
mouse_ad   = adata3
COUNTS     = 'raw_counts'
group_key  = 'timepoint'
outdir = Path(output_dir)
plt.rcParams['svg.fonttype'] = 'none'

group_order = ['uninduced', '1.5dpi_early', '1.5dpi_late', '2.5dpi', '3.5dpi', '4.5dpi_menstrual']

# Sample colors; each timepoint inherits the color of its sample (if it has exactly one)
sample_colors = {
    'mck_8':  '#990540',
    'mck_1':  '#f16f47',
    'mck_12': '#e7f59a',
    'mck_15': '#65c2a7',
    'mck_10': '#ffdf8d',
    'mck_11': '#624fa3',
}
fallback_colors = ['#990540', '#f16f47', '#e7f59a', '#65c2a7', '#ffdf8d', '#624fa3', '#8c8c8c', '#3b7dd8']

# Tick labels shown on the x-axis
display_names = {g: g.replace('_', ' ') for g in group_order}   # e.g. '1.5 dpi early'

SHOW_MEDIAN = True         # short black line at each timepoint's median
fig_size    = (8, 4)

# ---------------------------- per-nucleus metrics from raw counts ----------------------------
if COUNTS in mouse_ad.layers:
    X = mouse_ad.layers[COUNTS]
else:
    print(f"No '{COUNTS}' layer; using .X (check it holds raw counts)")
    X = mouse_ad.X
X = X.tocsr() if sps.issparse(X) else sps.csr_matrix(np.asarray(X))

umis  = np.asarray(X.sum(axis=1)).ravel()
genes = np.asarray((X > 0).sum(axis=1)).ravel()

mck = mouse_ad.obs['sample'].astype(str).str.extract(r'mck_?(\d+)', expand=False).radd('mck_').values
df_all = pd.DataFrame({'group': mouse_ad.obs[group_key].astype(str).values,
                       'mck': mck, 'umis': umis, 'genes': genes},
                      index=mouse_ad.obs_names)

missing = [g for g in group_order if g not in set(df_all['group'])]
if missing:
    raise ValueError(f"Missing in adata.obs['{group_key}']: {missing}. Present: {sorted(set(df_all['group']))}")
extra = sorted(set(df_all['group']) - set(group_order))
if extra:
    print(f"Note: timepoints not in group_order are excluded: {extra}")
df_all = df_all[df_all['group'].isin(group_order)]

if (df_all[['umis', 'genes']] <= 0).any().any():
    n0 = int((df_all[['umis', 'genes']] <= 0).any(axis=1).sum())
    print(f"Note: dropping {n0} nuclei with 0 counts (can't be shown on a log axis)")
    df_all = df_all[(df_all['umis'] > 0) & (df_all['genes'] > 0)]

# Timepoint colors: reuse the sample color when a timepoint is a single sample
tp_samples = df_all.groupby('group')['mck'].apply(lambda s: sorted(s.dropna().unique()))
color_dict = {}
for i, g in enumerate(group_order):
    smp = tp_samples.get(g, [])
    if len(smp) == 1 and smp[0] in sample_colors:
        color_dict[g] = sample_colors[smp[0]]
    else:
        color_dict[g] = fallback_colors[i % len(fallback_colors)]
        print(f"Note: {g} has samples {smp}; using fallback color {color_dict[g]}")

summary = df_all.groupby('group').agg(samples=('mck', lambda s: ', '.join(sorted(s.dropna().unique()))),
                                      nuclei=('umis', 'size'),
                                      median_umis=('umis', 'median'),
                                      median_genes=('genes', 'median')).reindex(group_order)
summary[['nuclei', 'median_umis', 'median_genes']] = summary[['nuclei', 'median_umis', 'median_genes']].round(0).astype(int)
print(summary.to_string(), '\n')

# ---------------------------- plotting helper ----------------------------
def nice_log_ticks(vmin, vmax):
    """Readable ticks (e.g. 500, 1,000, 2,000, 5,000) inside [vmin, vmax]; finer steps if the range is narrow."""
    inside = []
    for mults in [(1, 2, 5), (1, 2, 3, 5, 7), tuple(range(1, 10))]:
        ticks = [m * 10 ** e for e in range(0, 8) for m in mults]
        inside = [t for t in ticks if vmin <= t <= vmax]
        if len(inside) >= 3:
            break
    return inside

def violin_log(df, col, ylabel, fname):
    data = [np.log10(df.loc[df['group'] == g, col].values) for g in group_order]
    pos = np.arange(len(group_order))

    fig = plt.figure(figsize=fig_size)
    ax = fig.add_subplot(111)
    parts = ax.violinplot(data, positions=pos, widths=0.8,
                          showmeans=False, showmedians=False, showextrema=False)
    for body, g in zip(parts['bodies'], group_order):
        body.set_facecolor(color_dict[g])
        body.set_edgecolor('black')
        body.set_linewidth(0.8)
        body.set_alpha(1)

    if SHOW_MEDIAN:
        med = [np.median(d) for d in data]
        ax.hlines(med, pos - 0.18, pos + 0.18, colors='black', linewidth=1.2, zorder=3)

    # y-axis: positions in log10, labels in real counts
    vmin = 10 ** min(d.min() for d in data)
    vmax = 10 ** max(d.max() for d in data)
    ticks = nice_log_ticks(vmin, vmax)
    ax.set_yticks(np.log10(ticks))
    ax.set_yticklabels([f"{t:,}" for t in ticks])
    ax.set_ylim(np.log10(vmin) - 0.05, np.log10(vmax) + 0.05)

    ax.set_xticks(pos)
    ax.set_xticklabels([display_names.get(g, g) for g in group_order], rotation=45, ha='right', color='black')
    ax.set_xlabel('')
    ax.set_ylabel(ylabel)
    ax.tick_params(axis='y', colors='black')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    fig.patch.set_facecolor('white')

    fig.tight_layout()
    fig.set_size_inches(*fig_size)
    base = outdir / fname
    fig.savefig(base.with_suffix('.svg'), format='svg', facecolor='white')
    plt.close(fig)
    print(f"Saved {base.with_suffix('.png')} (and .svg)")

# ---------------------------- plots ----------------------------
violin_log(df_all, 'umis',  'UMIs per nucleus (log scale)',  "20261113-QC_Mouse_All_UMIsPerNucleus_byTimepoint_logscale")
violin_log(df_all, 'genes', 'Genes per nucleus (log scale)', "20261113-QC_Mouse_All_GenesPerNucleus_byTimepoint_logscale")

## Human

### Cells Per sample

In [ ]:
# ---------------------------- settings ----------------------------
group_key   = 'timepoint'
sample_key  = 'sample'
x_key, y_key = 'x_um_dbscan', 'y_um_dbscan'
group_order = ['secretory', 'menstrual']

# ---------------------------- metadata ----------------------------
metadata = adata1.obs[[group_key, sample_key, x_key, y_key]].copy()
metadata[group_key] = metadata[group_key].astype(str)
metadata['mck'] = metadata[sample_key].astype(str).str.extract(r'mck_?(\d+)', expand=False).radd('mck_')
metadata['mapped'] = metadata[x_key].notna() & metadata[y_key].notna()   # has a spatial position

extra = sorted(set(metadata[group_key]) - set(group_order))
if extra:
    print(f"Note: timepoints not in group_order are listed at the end: {extra}")
order = [g for g in group_order if g in set(metadata[group_key])] + extra

def summarize(df, by):
    out = df.groupby(by, observed=True).agg(total=('mapped', 'size'), mapped=('mapped', 'sum'))
    out['% mapped'] = (out['mapped'] / out['total'] * 100).round(2)
    return out

# ---------------------------- per timepoint ----------------------------
per_tp = summarize(metadata, group_key).reindex(order)
per_tp.loc['Total'] = [len(metadata), int(metadata['mapped'].sum()),
                       round(metadata['mapped'].mean() * 100, 2)]
per_tp[['total', 'mapped']] = per_tp[['total', 'mapped']].astype(int)

print("Human dataset: mapped nuclei per timepoint")
display(per_tp)

In [ ]:
adata3.obs[['total_counts', 'n_genes_by_counts']].median()

### Counts/Genes per Cell

In [ ]:
# ---------------------------- settings ----------------------------
import scipy.sparse as sps   # distinct name so plotting loops can't overwrite it

ad_plot    = adata1
COUNTS     = 'raw_counts'
group_key  = 'timepoint'
outdir = Path(output_dir)
plt.rcParams['svg.fonttype'] = 'none'

group_order = ['secretory', 'menstrual']

# Color per timepoint
color_dict = {
    'secretory':  '#eab073',
    'menstrual':  '#1e6a37',
}

# Tick labels shown on the x-axis
display_names = {g: g.replace('_', ' ') for g in group_order}

SHOW_MEDIAN = True         # short black line at each timepoint's median
fig_size    = (3, 4)
file_tag    = "20261113-QC_Human_All"

# ---------------------------- per-nucleus metrics from raw counts ----------------------------
if COUNTS in ad_plot.layers:
    X = ad_plot.layers[COUNTS]
else:
    print(f"No '{COUNTS}' layer; using .X (check it holds raw counts)")
    X = ad_plot.X
X = X.tocsr() if sps.issparse(X) else sps.csr_matrix(np.asarray(X))

umis  = np.asarray(X.sum(axis=1)).ravel()
genes = np.asarray((X > 0).sum(axis=1)).ravel()

df_all = pd.DataFrame({'group':  ad_plot.obs[group_key].astype(str).values,
                       'sample': ad_plot.obs[sample_key].astype(str).values,
                       'umis': umis, 'genes': genes},
                      index=ad_plot.obs_names)

missing = [g for g in group_order if g not in set(df_all['group'])]
if missing:
    raise ValueError(f"Missing in adata.obs['{group_key}']: {missing}. Present: {sorted(set(df_all['group']))}")
no_color = [g for g in group_order if g not in color_dict]
if no_color:
    raise ValueError(f"No color for: {no_color}")
extra = sorted(set(df_all['group']) - set(group_order))
if extra:
    print(f"Note: timepoints not in group_order are excluded: {extra}")
df_all = df_all[df_all['group'].isin(group_order)]

if (df_all[['umis', 'genes']] <= 0).any().any():
    n0 = int((df_all[['umis', 'genes']] <= 0).any(axis=1).sum())
    print(f"Note: dropping {n0} nuclei with 0 counts (can't be shown on a log axis)")
    df_all = df_all[(df_all['umis'] > 0) & (df_all['genes'] > 0)]

summary = df_all.groupby('group').agg(samples=('sample', lambda s: ', '.join(sorted(s.unique()))),
                                      nuclei=('umis', 'size'),
                                      median_umis=('umis', 'median'),
                                      median_genes=('genes', 'median')).reindex(group_order)
summary[['nuclei', 'median_umis', 'median_genes']] = summary[['nuclei', 'median_umis', 'median_genes']].round(0).astype(int)
print(summary.to_string(), '\n')

# ---------------------------- plotting helper ----------------------------
def nice_log_ticks(vmin, vmax):
    """Readable ticks (e.g. 500, 1,000, 2,000, 5,000) inside [vmin, vmax]; finer steps if the range is narrow."""
    inside = []
    for mults in [(1, 2, 5), (1, 2, 3, 5, 7), tuple(range(1, 10))]:
        ticks = [m * 10 ** e for e in range(0, 8) for m in mults]
        inside = [t for t in ticks if vmin <= t <= vmax]
        if len(inside) >= 3:
            break
    return inside

def violin_log(df, col, ylabel, fname):
    data = [np.log10(df.loc[df['group'] == g, col].values) for g in group_order]
    pos = np.arange(len(group_order))

    fig = plt.figure(figsize=fig_size)
    ax = fig.add_subplot(111)
    parts = ax.violinplot(data, positions=pos, widths=0.8,
                          showmeans=False, showmedians=False, showextrema=False)
    for body, g in zip(parts['bodies'], group_order):
        body.set_facecolor(color_dict[g])
        body.set_edgecolor('black')
        body.set_linewidth(0.8)
        body.set_alpha(1)

    if SHOW_MEDIAN:
        med = [np.median(d) for d in data]
        ax.hlines(med, pos - 0.18, pos + 0.18, colors='black', linewidth=1.2, zorder=3)

    # y-axis: positions in log10, labels in real counts
    vmin = 10 ** min(d.min() for d in data)
    vmax = 10 ** max(d.max() for d in data)
    ticks = nice_log_ticks(vmin, vmax)
    ax.set_yticks(np.log10(ticks))
    ax.set_yticklabels([f"{t:,}" for t in ticks])
    ax.set_ylim(np.log10(vmin) - 0.05, np.log10(vmax) + 0.05)

    ax.set_xticks(pos)
    ax.set_xticklabels([display_names.get(g, g) for g in group_order], rotation=45, ha='right', color='black')
    ax.set_xlabel('')
    ax.set_ylabel(ylabel)
    ax.tick_params(axis='y', colors='black')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    fig.patch.set_facecolor('white')

    fig.tight_layout()
    fig.set_size_inches(*fig_size)
    base = outdir / fname
    fig.savefig(base.with_suffix('.svg'), format='svg', facecolor='white')
    plt.close(fig)
    print(f"Saved {base.with_suffix('.png')} (and .svg)")

# ---------------------------- plots ----------------------------
violin_log(df_all, 'umis',  'UMIs per nucleus (log scale)',  f"{file_tag}_UMIsPerNucleus_byTimepoint_logscale")
violin_log(df_all, 'genes', 'Genes per nucleus (log scale)', f"{file_tag}_GenesPerNucleus_byTimepoint_logscale")

# Mouse pregnancy Decidual Zone Scores

### Module Score Heatmap

In [ ]:
obj = adata4

# Li et al. 2022 Biol Reprod 107:529, Table 1 (top 20 markers per Visium domain, pregnancy D7.5)
li_domains = {
    "PDZ":   ["Prl3c1","Htra1","Fabp4","Prl8a2","Tnnc1","Gm2694","Dbi","Gm2a","Psap","Cryab",
              "Erv3","Ctla2b","Crct1","Adamts5","Acsbg1","Krtdap","Smpd1","Gng11","Hopx","Ntn5"],
    "TDZ":   ["Psca","Sprr2h","Gpnmb","Erv3","Krtdap","Prl8a2","Fabp4","Olr1","Psap","Akr1b3",
              "Prdx6","Rgcc","Acsbg1","Hsd11b1","Dbi","Hopx","Cyb5a","Lipg","Ehd3","Dnaja4"],
    "SDZ":   ["Mt3","Mt4","Gatm","Mgp","Coch","Dio3","Sfrp5","Pdzk1ip1","Dmkn","Mt2",
              "Atoh8","Anxa8","Mt1","Nampt","Prrx2","Tdo2","Drd4","A2m","Prkg2","Ass1"],
    "VSZ":   ["Stmn2","Hmgn5","Fmo1","Wnt6","Rcn1","Hmgcs2","Tdo2","Sned1","Ntrk1","Ear2",
              "Tac2","Susd2","Anxa5","Fbn1","Srgn","Ptn","Cxcl14","Efemp1","Tmem176a","Syt13"],
    "MD":    ["Gzmf","Spp1","Gzmd","Gzme","Gzmg","Gzmc","Gzma","Prf1","Gzmb","Fcer1g",
              "Ifitm1","AW112010","Tyrobp","Il2rb","Cd52","Tmsb15a","Chn1","Rac2","Ptn","Trbc1"],
    "udStr": ["Sfrp4","Rbp4","Igfbp5","Igfbp3","Penk","Spink8","Prxl2a","C3","Mt4","Ctla2a",
              "Mfap4","Bgn","Itm2a","Col1a2","Igf1","Rnase4","Col1a1","Hsd11b2","Serpine2","Pcolce"],
}

# ---------------- settings ----------------
timepoint   = '3.5dpi'
tp_key      = 'timepoint'
use_domains = ['PDZ', 'TDZ', 'SDZ', 'VSZ', 'MD', 'udStr']   # heatmap column order
group_map = {'eFib1': 'eFib', 'eFib2': 'eFib', 'eFib2i': 'eFib', 'eFib3': 'eFib', 'eFibP': 'eFib',
             'dec1': 'dec1', 'dec2': 'dec2', 'dec3': 'dec3'}
group_order = ['eFib', 'dec1', 'dec2', 'dec3']
drop_shared = False    # True removes genes that appear in more than one module
min_pct     = 0.01     # gene must be detected in >= 1% of nuclei of at least one group
min_genes   = 4        # skip a module with fewer genes left
out_prefix  = 'Li2022_sg_3dpi_'   # score columns stored on obj
vmin, vmax  = -1.8, 1.8
outpath = os.path.join(output_dir, f"20261113-Heatmap_Mouse_Li2022_ModuleScore_{timepoint}.svg")

# ---------------- nuclei of this timepoint, log-normalised values in X ----------------
in_tp = obj.obs[tp_key].astype(str) == timepoint
if not in_tp.any():
    raise ValueError(f"'{timepoint}' not in obj.obs['{tp_key}']: {sorted(obj.obs[tp_key].astype(str).unique())}")
grp = obj.obs['cell_subtype'].astype(str).map(group_map)
cells = (in_tp & grp.notna()).to_numpy()

tmp = sc.AnnData(
    X=obj.layers['log1p_counts'][cells],
    obs=pd.DataFrame({'group': pd.Categorical(grp[cells], categories=group_order)},
                     index=obj.obs_names[cells]),
    var=pd.DataFrame(index=obj.var_names))

# ---------------- filter genes ----------------
all_genes = sorted({g for d in use_domains for g in li_domains[d]})
present = [g for g in all_genes if g in tmp.var_names]
Xg = tmp[:, present].X
Xg = Xg.toarray() if hasattr(Xg, 'toarray') else np.asarray(Xg)
det = pd.DataFrame(Xg > 0, columns=present).groupby(tmp.obs['group'].to_numpy()).mean()
expressed = set(det.columns[(det >= min_pct).any(axis=0)])

n_modules = Counter(g for d in use_domains for g in set(li_domains[d]))
shared = {g for g, n in n_modules.items() if n > 1} if drop_shared else set()
modules = {d: [g for g in dict.fromkeys(li_domains[d]) if g in expressed and g not in shared]
           for d in use_domains}
scored = [d for d in use_domains if len(modules[d]) >= min_genes]

# ---------------- module score ----------------
for d in scored:
    sc.tl.score_genes(tmp, modules[d], score_name=f'sg_{d}', ctrl_size=50,
                      random_state=0, use_raw=False)
    obj.obs[f'{out_prefix}{d}'] = np.nan
    obj.obs.loc[tmp.obs_names, f'{out_prefix}{d}'] = tmp.obs[f'sg_{d}']

sg = tmp.obs[[f'sg_{d}' for d in scored]]
sg.columns = scored
mean_sg = sg.groupby(tmp.obs['group'], observed=True).mean().loc[group_order]
mat = (mean_sg - mean_sg.mean()) / mean_sg.std(ddof=0)        # z across groups, per module

# ---------------- heatmap ----------------
plt.rcParams['svg.fonttype'] = 'none'     # editable text in the svg
fig, ax = plt.subplots(figsize=(0.9 * len(scored) + 2, 3.5))
im = ax.imshow(mat.values, cmap='viridis', vmin=vmin, vmax=vmax, aspect='auto')
ax.set_xticks(range(len(scored)), scored)
ax.set_yticks(range(len(group_order)), group_order)
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        v = mat.values[i, j]
        ax.text(j, i, f'{v:.1f}', ha='center', va='center', fontsize=9,
                color='black' if (v - vmin) / (vmax - vmin) > 0.6 else 'white')
ax.set_title(f'Module score, {timepoint} (z across groups)')
fig.colorbar(im, ax=ax, shrink=0.8, label='z')
plt.tight_layout()
fig.savefig(outpath, format='svg', bbox_inches='tight')
plt.close()
print(mat.round(1).to_string())     # scaled scores, as printed in the heatmap boxes

### Spatial maping of pregnancy zone scores

In [ ]:
# Settings
key_sample   = "timepoint"
x_key, y_key = "x_um_dbscan", "y_um_dbscan"

timepoint = "3.5dpi"
rot, view = 135, [(2600, 9100), (2200, 8700)]           # same rotation and window as before

domains      = ['PDZ', 'TDZ', 'SDZ', 'VSZ', 'MD', 'udStr']   # left to right
score_prefix = 'Li2022_sg_3dpi_'     # must match out_prefix in the module-score cell
cmap         = 'viridis'
scale        = 'shared'           # 'per_panel' or 'shared'
pct          = (1, 99)               # color range clipped to these percentiles

show_unscored = True                 # draw stromal nuclei without a score (e.g. decAp) underneath
other_color   = "#231f20"
dot_size      = 25                   # point size of the 6-inch single images
fig_size_in   = 6
grid_panel_in = 3                    # size of each panel (inches)
background    = "black"
dpi_save      = 300
outpath = os.path.join(output_dir, f"20261113-Spatial_Mouse-Li2022-{score_prefix.strip('_')}-{timepoint}-grid.png")

# ---------------- checks ----------------
for col in [key_sample, x_key, y_key]:
    if col not in adata3.obs:
        raise ValueError(f"{col} not found in adata3.obs")
n_missing = (~adata4.obs_names.isin(adata3.obs_names)).sum()
if n_missing:
    raise ValueError(f"{n_missing} cells in adata4 are not in adata3")

missing = [d for d in domains if f"{score_prefix}{d}" not in adata4.obs]
if missing:
    print(f"Warning: no score column for {missing}, skipped")
domains = [d for d in domains if d not in missing]
if not domains:
    raise ValueError(f"No '{score_prefix}*' columns in adata4.obs. "
                     f"Present: {[c for c in adata4.obs.columns if c.startswith('Li2022')]}")

# ---------------- rotation center: the whole section in adata3 (same as the other images) ----------------
m3 = (adata3.obs[key_sample].astype(str) == timepoint).to_numpy()
if not m3.any():
    raise ValueError(f"No nuclei for '{timepoint}' in adata3. "
                     f"Present: {sorted(adata3.obs[key_sample].astype(str).unique())}")
fx = adata3.obs.loc[m3, x_key].astype(float).to_numpy()
fy = adata3.obs.loc[m3, y_key].astype(float).to_numpy()
ok = np.isfinite(fx) & np.isfinite(fy)
cx, cy = 0.5 * (fx[ok].min() + fx[ok].max()), 0.5 * (fy[ok].min() + fy[ok].max())

# ---------------- adata4 nuclei of this timepoint (timepoint + coordinates from adata3) ----------------
obs = adata3.obs.loc[adata4.obs_names, [key_sample, x_key, y_key]]
obs = obs[obs[key_sample].astype(str) == timepoint]
x = obs[x_key].astype(float).to_numpy()
y = obs[y_key].astype(float).to_numpy()
ok = np.isfinite(x) & np.isfinite(y)
obs, x, y = obs[ok], x[ok], y[ok]

t = np.deg2rad(rot)
xr = (x - cx) * np.cos(t) - (y - cy) * np.sin(t) + cx
yr = (x - cx) * np.sin(t) + (y - cy) * np.cos(t) + cy

scores = adata4.obs.loc[obs.index, [f"{score_prefix}{d}" for d in domains]].astype(float)
scores.columns = domains
(vx0, vx1), (vy0, vy1) = view
in_view = (xr >= vx0) & (xr <= vx1) & (yr >= vy0) & (yr <= vy1)

# ---------------- color ranges (nuclei inside the window) ----------------
sv = scores[in_view]
if scale == 'shared':
    lo, hi = np.nanpercentile(sv.values, pct)
    ranges = {d: (lo, hi) for d in domains}
else:
    ranges = {d: tuple(np.nanpercentile(sv[d], pct)) for d in domains}

display(pd.DataFrame({
    'color min': [ranges[d][0] for d in domains],
    'color max': [ranges[d][1] for d in domains],
    'score min': [sv[d].min() for d in domains],
    'score max': [sv[d].max() for d in domains],
    'n scored nuclei': [sv[d].notna().sum() for d in domains],
}, index=pd.Index(domains, name=f'{timepoint} ({score_prefix}*)'))
    .style.format('{:.3f}').format('{:,}', subset=['n scored nuclei']))

# ---------------- grid: panels touching, no labels ----------------
n = len(domains)
fig = plt.figure(figsize=(grid_panel_in * n, grid_panel_in))
fig.patch.set_facecolor(background)
dot_size_grid = dot_size * (grid_panel_in / fig_size_in) ** 2

for i, d in enumerate(domains):
    ax = fig.add_axes([i / n, 0, 1 / n, 1])
    s = scores[d].to_numpy()
    has = np.isfinite(s)

    if show_unscored and (~has).any():
        ax.scatter(xr[~has], yr[~has], s=dot_size_grid, color=other_color, linewidth=0, zorder=1)

    o = np.argsort(s[has])                      # highest scores drawn on top
    vmin, vmax = ranges[d]
    ax.scatter(xr[has][o], yr[has][o], c=s[has][o], cmap=cmap, vmin=vmin, vmax=vmax,
               s=dot_size_grid, linewidth=0, zorder=2)

    ax.set_xlim(vx0, vx1)
    ax.set_ylim(vy0, vy1)
    ax.set_aspect("equal", adjustable="box")
    ax.set_facecolor(background)
    ax.axis("off")

fig.savefig(outpath, dpi=dpi_save, facecolor=background, pad_inches=0)
plt.close()
print(f"saved {outpath}")

# Human Stro-7 Module Score

In [ ]:
# ---------------- settings ----------------
obj         = adata2
key_group   = 'cell_subtype'
tp_key      = 'timepoint'
timepoint   = None              # None = both timepoints; or 'secretory' / 'menstrual'
pattern     = r'^Ulrich2024_sg_.*_top50$'   # score columns in obj.obs to use (one heatmap column each)
group_order = ['myoFib', 'eFib_a', 'eFib_b', 'eFib_c', 'dec_a', 'dec_b', 'dec_c', 'dec_d']
annot       = 'z'               # numbers in the boxes: 'z' (same as the colors) or 'mean' (raw mean score)
vmin, vmax  = -2, 2         # with 8 groups, z can reach about ±2.65

tag     = timepoint if timepoint else 'all'
outpath = os.path.join(output_dir, f"20261113-Heatmap_Human_Ulrich2024_ModuleScore_{tag}.svg")

# ---------------- score columns and nuclei ----------------
score_cols = [c for c in obj.obs.columns if re.match(pattern, c)]
if not score_cols:
    raise ValueError(f"No obs columns match {pattern}. "
                     f"Ulrich columns present: {[c for c in obj.obs.columns if 'Ulrich' in c]}")
print(f"Using {len(score_cols)} score column(s): {score_cols}")

sel = obj.obs[key_group].astype(str).isin(group_order)
if timepoint is not None:
    sel &= obj.obs[tp_key].astype(str) == timepoint
    if not sel.any():
        raise ValueError(f"No {group_order} nuclei for '{timepoint}' in obj.obs['{tp_key}']")

# Column names shown on the heatmap: drop the shared prefix/suffix
short = {c: re.sub(r'^Ulrich2024_sg_|_top50$', '', c) for c in score_cols}

sg = obj.obs.loc[sel, score_cols].astype(float).rename(columns=short)
grp = pd.Categorical(obj.obs.loc[sel, key_group].astype(str), categories=group_order)
mean_sg = sg.groupby(grp, observed=False).mean().reindex(group_order)
n_per_group = pd.Series(grp).value_counts().reindex(group_order)

absent = n_per_group[n_per_group.fillna(0) == 0].index.tolist()
if absent:
    print(f"Note: no nuclei for {absent} ({tag}); rows left empty")

mat = (mean_sg - mean_sg.mean()) / mean_sg.std(ddof=0)        # z across groups, per module

print(f"\nNuclei per group ({tag}):")
print(n_per_group.fillna(0).astype(int).to_string())
print("\nMean module score:")
print(mean_sg.round(3).to_string())
print("\nz across groups (heatmap colors):")
print(mat.round(2).to_string())

# ---------------- heatmap ----------------
cols = list(mat.columns)
fig, ax = plt.subplots(figsize=(0.9 * len(cols) + 2, 0.4 * len(group_order) + 1.5))
im = ax.imshow(mat.values, cmap='viridis', vmin=vmin, vmax=vmax, aspect='auto')
ax.set_xticks(range(len(cols)), cols)
ax.set_yticks(range(len(group_order)), group_order)
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        z = mat.values[i, j]
        if not np.isfinite(z):
            continue
        shown = z if annot == 'z' else mean_sg.values[i, j]
        ax.text(j, i, f'{shown:.1f}' if annot == 'z' else f'{shown:.2f}',
                ha='center', va='center', fontsize=9,
                color='black' if (z - vmin) / (vmax - vmin) > 0.6 else 'white')
title_numbers = 'z' if annot == 'z' else 'numbers: mean score'
ax.set_title(f'Module score, human {tag} (color: z across groups; {title_numbers})')
fig.colorbar(im, ax=ax, shrink=0.8, label='z')
plt.tight_layout()
fig.savefig(outpath, format='svg', bbox_inches='tight')
plt.show()
print(f"Saved {outpath}")